# Tests de import trips

In [1]:
import pandas as pd 

print(pd. __version__)

2.3.1


In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 1. Copia de dataframe

In [5]:
df = pd.DataFrame({
    "trip_id": [1, 2, 3],
    "origen": ["A", "B", "C"],
    "valor": [10, 20, 30],
})

df_copy = df.copy(deep=True)

df_copy.loc[1, "origen"] = "X"      # cambia solo la fila 1 en la copia
df_copy["valor"] = df_copy["valor"] * 100

print("ORIGINAL")
display(df)

print("COPIA")
display(df_copy)

ORIGINAL


,trip_id,origen,valor
0,1,A,10
1,2,B,20
2,3,C,30


COPIA


,trip_id,origen,valor
0,1,A,1000
1,2,X,2000
2,3,C,3000


### 2. Chequeo de dataset vacio

In [9]:
df_empty = pd.DataFrame()
df_empty_cols = pd.DataFrame(columns=["trip_id", "origin", "destination"])

print("DATAFRAME EMPTY ROWS AND COLS")
display(df_empty)

print("DATAFRAME EMPTY COLUMNS")
display(df_empty_cols)

print("df_empty.empty:", df_empty.empty)
print("len(df_empty):", len(df_empty))
print("df_empty.shape:", df_empty.shape)

print("\ndf_empty_cols.empty:", df_empty_cols.empty)
print("len(df_empty_cols):", len(df_empty_cols))
print("df_empty_cols.shape:", df_empty_cols.shape)

DATAFRAME EMPTY ROWS AND COLS


""


DATAFRAME EMPTY COLUMNS


,trip_id,origin,destination


df_empty.empty: True
len(df_empty): 0
df_empty.shape: (0, 0)

df_empty_cols.empty: True
len(df_empty_cols): 0
df_empty_cols.shape: (0, 3)


### 3. Creación de ImportOptions

In [15]:
from pylondrina.importing import ImportOptions
from dataclasses import asdict

# 1) Default
opts_default = ImportOptions()

# 2) Con valores
opts_custom = ImportOptions(
    keep_extra_fields=False,
    selected_fields=("trip_id", "origin_zone", "destination_zone", "origin_time_utc", "destination_time_utc"),
    strict=True,
    strict_domains=True,
    single_stage=True,
    source_timezone="America/Santiago",
)

print("DEFAULT asdict:")
print(asdict(opts_default))

print("\nCUSTOM asdict:")
print(asdict(opts_custom))

print("\nAcceso directo:")
print("single_stage:", opts_custom.single_stage)
print("source_timezone:", opts_custom.source_timezone)
print("selected_fields:", opts_custom.selected_fields)

DEFAULT asdict:
{'keep_extra_fields': True, 'selected_fields': None, 'strict': False, 'strict_domains': False, 'single_stage': False, 'source_timezone': None}

CUSTOM asdict:
{'keep_extra_fields': False, 'selected_fields': ('trip_id', 'origin_zone', 'destination_zone', 'origin_time_utc', 'destination_time_utc'), 'strict': True, 'strict_domains': True, 'single_stage': True, 'source_timezone': 'America/Santiago'}

Acceso directo:
single_stage: True
source_timezone: America/Santiago
selected_fields: ('trip_id', 'origin_zone', 'destination_zone', 'origin_time_utc', 'destination_time_utc')


### 4. Creación y exploración de TripSchema

In [18]:
from pylondrina.schema import DomainSpec, FieldSpec, TripSchema

# 1) Definir un dominio categórico (ej: modo)
mode_domain = DomainSpec(
    values=["walk", "bike", "car", "bus"],
    extendable=True,
    aliases={"Walking": "walk", "BICYCLE": "bike"}
)

# 2) Definir FieldSpec (dos campos requeridos + uno categórico opcional)
f_trip_id = FieldSpec(
    name="trip_id",
    dtype="string",
    required=True
)

f_origin_zone = FieldSpec(
    name="origin_zone",
    dtype="string",
    required=True
)

f_mode = FieldSpec(
    name="mode",
    dtype="categorical",
    required=False,
    domain=mode_domain
)

# 3) Construir el dict fields (catalogo)
fields = {
    f_trip_id.name: f_trip_id,
    f_origin_zone.name: f_origin_zone,
    f_mode.name: f_mode,
}

# 4) Definir required (lista de nombres de campos requeridos)
required = ["trip_id", "origin_zone"]

# 5) Crear TripSchema
schema = TripSchema(
    version="0.1.0",
    fields=fields,
    required=required,
    semantic_rules=None
)

# Explorar fields y required
print("schema.version:", schema.version)
print("schema.fields keys:", list(schema.fields.keys()))
print("schema.required:", schema.required)

print("required ⊆ fields:", set(schema.required).issubset(set(schema.fields.keys())))

# Caso de error: required con un campo no definido en fields
schema_bad = TripSchema(
    version="0.1.0",
    fields=fields,
    required=["trip_id", "origin_zone", "missing_field"]  # no está en fields
)
print("required ⊆ fields (bad schema):", set(schema_bad.required).issubset(set(schema_bad.fields.keys())))

schema.version: 0.1.0
schema.fields keys: ['trip_id', 'origin_zone', 'mode']
schema.required: ['trip_id', 'origin_zone']
required ⊆ fields: True
required ⊆ fields (bad schema): False


### 5. Verificacion de TripSchema

In [2]:
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec, verify_trip_schema_fields, build_schema_effective_from_findings

# Domain OK
dom_ok = DomainSpec(values=["walk", "bus", "car"], extendable=True, aliases={"Walking": "walk"})

# Domain malo: values no-string
dom_bad = DomainSpec(values=["ok", 123], extendable=True, aliases=None)

schema_test = TripSchema(
    version="0.1.0",
    fields={
        # --- dtypes válidos ---
        "s": FieldSpec(name="s", dtype="string", constraints={"nullable": True, "length": {"min": 1, "max": 10}}),
        "i": FieldSpec(name="i", dtype="int", constraints={"range": {"min": 0, "max": 10}, "unique": False}),
        "f": FieldSpec(name="f", dtype="float", constraints={"range": {"min": 0.0, "max": 1.0}}),
        "b": FieldSpec(name="b", dtype="bool", constraints={"nullable": False}),
        "dt": FieldSpec(name="dt", dtype="datetime", constraints={"datetime": {"timezone": "UTC", "allow_naive": False, "min": None, "max": None}}),
        "cat_ok": FieldSpec(name="cat_ok", dtype="categorical", domain=dom_ok, constraints={"nullable": True}),

        # --- dtype inválido ---
        "bad_dtype": FieldSpec(name="bad_dtype", dtype="geopoint", constraints={"nullable": True}),

        # --- categorical sin domain (caso a degradar) ---
        "cat_no_domain": FieldSpec(name="cat_no_domain", dtype="categorical", domain=None, constraints={"nullable": True}),

        # --- categorical con domain malo (values no-string) ---
        "cat_bad_domain": FieldSpec(name="cat_bad_domain", dtype="categorical", domain=dom_bad),

        # --- constraints: key desconocida ---
        "unknown_constraint": FieldSpec(name="unknown_constraint", dtype="string", constraints={"foo": 1, "nullable": True}),

        # --- constraints: pattern válido e inválido ---
        "pat_ok": FieldSpec(name="pat_ok", dtype="string", constraints={"pattern": r"^[A-Z]{3}\d{2}$"}),
        "pat_bad": FieldSpec(name="pat_bad", dtype="string", constraints={"pattern": "["}),  # regex inválida

        # --- constraints que no corresponden al tipo ---
        "range_on_string": FieldSpec(name="range_on_string", dtype="string", constraints={"range": {"min": 0, "max": 10}}),
        "length_on_int": FieldSpec(name="length_on_int", dtype="int", constraints={"length": {"min": 1, "max": 5}}),

        # --- h3 constraints (aunque dtype sea string, h3 es semántico) ---
        "h3_cell": FieldSpec(name="h3_cell", dtype="string", constraints={"h3": {"require_valid": True, "resolution": 8, "allow_mixed_resolution": False}}),
    },
    required=[],
    semantic_rules=None,
)

findings = verify_trip_schema_fields(schema_test)

for f in findings:
    print(f)

schema_effective = build_schema_effective_from_findings(schema_test, findings)

def get_effective_dtype(schema, schema_effective, field_name: str) -> str:
    return schema_effective.dtype_effective.get(field_name, schema.fields[field_name].dtype)

print("\ndtype base purpose:", schema_test.fields["cat_no_domain"].dtype)
print("dtype efectivo purpose:", get_effective_dtype(schema_test, schema_effective, "cat_no_domain"))

# test A1: dtype invalido -> override
print("\ndtype efectivo bad_dtype:", schema_effective.dtype_effective["bad_dtype"] )
print("razones:", schema_effective.overrides["bad_dtype"]["reasons"])

# A2: categorical sin domain -> override (degradación)
print("\ndtype efectivo cat_no_domain:", schema_effective.dtype_effective["cat_no_domain"] )
print("razones:", schema_effective.overrides["cat_no_domain"]["reasons"])

# B: Probar que los findings “error” se distinguen claramente (pattern inválido)
errs = [f for f in findings if f["level"] == "error"]
print("\nFindings de nivel ERROR:")
for err in errs:
    print(err)

# C: Probar que constraints incompatibles NO degradan dtype (solo warning)
warnings = [f for f in findings if f["level"] == "warning"]
print("\nFindings de nivel WARNING:")
for w in warnings:
    print(w)

# D: Probar que SchemaEffective no toca el schema base (inmutabilidad práctica)
print("\ndtype base (inmutable):", schema_test.fields["cat_no_domain"].dtype )
print("dtype efectivo (degradado):", schema_effective.dtype_effective["cat_no_domain"] )

{'field': 'bad_dtype', 'level': 'warning', 'kind': 'dtype_invalid', 'detail': "dtype='geopoint' no está en ['bool', 'categorical', 'datetime', 'float', 'int', 'string']"}
{'field': 'cat_no_domain', 'level': 'warning', 'kind': 'categorical_no_domain', 'detail': "dtype='categorical' pero domain=None"}
{'field': 'cat_bad_domain', 'level': 'warning', 'kind': 'domain_values_not_string', 'detail': 'DomainSpec.values contiene no-string: [123]'}
{'field': 'unknown_constraint', 'level': 'warning', 'kind': 'unknown_constraints', 'detail': "constraints contiene llaves desconocidas: ['foo']"}
{'field': 'pat_bad', 'level': 'error', 'kind': 'pattern_invalid', 'detail': 'regex no compila: unterminated character set at position 0'}
{'field': 'range_on_string', 'level': 'warning', 'kind': 'constraint_incompatible_with_dtype', 'detail': "constraints ['range'] no aplican a dtype='string'"}
{'field': 'length_on_int', 'level': 'warning', 'kind': 'constraint_incompatible_with_dtype', 'detail': "constraints 

#### Prueba que se detecta categorical_empty_domain

In [3]:
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec
from pylondrina.schema import verify_trip_schema_fields  

# Caso B: domain existe pero values vacío -> debería emitir categorical_empty_domain
dom_empty = DomainSpec(values=[], extendable=True, aliases=None)

# Categórico "normal"
dom_mode = DomainSpec(values=["walk", "bus"], extendable=True, aliases=None)

schema_test = TripSchema(
    version="0.1.0",
    fields={
        "mode_bootstrap": FieldSpec(name="mode_bootstrap", dtype="categorical", domain=dom_empty),
        "mode_ok": FieldSpec(name="mode_ok", dtype="categorical", domain=dom_mode),
        "cat_no_domain": FieldSpec(name="cat_no_domain", dtype="categorical", domain=None),
    },
    required=[],
)

findings = verify_trip_schema_fields(schema_test)
for f in findings:
    print(f)

# asserts “caseros”
assert any(f["field"] == "mode_bootstrap" and f["kind"] == "categorical_empty_domain" for f in findings)
assert any(f["field"] == "cat_no_domain" and f["kind"] == "categorical_no_domain" for f in findings)
assert not any(f["field"] == "mode_ok" and f["kind"] == "categorical_empty_domain" for f in findings)

{'field': 'mode_bootstrap', 'level': 'info', 'kind': 'categorical_empty_domain', 'detail': "dtype='categorical' y DomainSpec.values vacío (bootstrapping en S6)"}
{'field': 'cat_no_domain', 'level': 'warning', 'kind': 'categorical_no_domain', 'detail': "dtype='categorical' pero domain=None"}


### 6. Verificar si en field_correspondence existe un campo canonico que no esta en en el schema

In [17]:
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec

schema = TripSchema(
    version="0.1.0",
    fields={
        "user_id": FieldSpec(name="user_id", dtype="string", required=True),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=False),
    },
    required=["user_id"],
)

field_correspondence = {
    "user_id": "id_persona",
    "origin_latitude": "lat_o",
    "BAD_CANONICAL": "algo",   # <-- este NO existe en schema.fields
}

invalid = [canon for canon in field_correspondence.keys() if canon not in schema.fields]

print("Invalid canonicals:", invalid)

Invalid canonicals: ['BAD_CANONICAL']


### 7. Verificar si en field_correspondence existe un campo de la fuente que no este en el dataframe 

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "id_persona": [1, 2, 3],
    "lat_o": [-33.45, -33.46, -33.47],
    # nota: NO se incluye "lon_o"
})

field_correspondence = {
    "user_id": "id_persona",   # OK (existe)
    "origin_latitude": "lat_o",# OK (existe)
    "origin_longitude": "lon_o" # NO existe en df.columns -> caso de prueba
}

missing_sources = [
    source_col
    for canonical, source_col in field_correspondence.items()
    if source_col not in df.columns
]
print("Missing source columns:", missing_sources)

missing_pairs = [
    (canonical, source_col)
    for canonical, source_col in field_correspondence.items()
    if source_col not in df.columns
]
print("Missing pairs:", missing_pairs)

Missing source columns: ['lon_o']
Missing pairs: [('origin_longitude', 'lon_o')]


### 8. Verificar que en field_correspondence no se repita valores en el conjunto de clave:valor

In [21]:
from collections import defaultdict

def find_value_collisions(field_correspondence: dict[str, str]) -> dict[str, list[str]]:
    """
    Retorna un dict {source_col: [canonical1, canonical2, ...]} solo para source_col con colisión (len>1).
    """
    inverse = defaultdict(list)  # source_col -> canonicals
    for canonical, source_col in field_correspondence.items():
        inverse[source_col].append(canonical)

    collisions = {src: c_list for src, c_list in inverse.items() if len(c_list) > 1}
    return collisions

field_correspondence = {
    "user_id": "id_persona",
    "origin_latitude": "lat_o",
    "destination_latitude": "lat_o",   # repetido
    "origin_longitude": "lon_o",
}

collisions = find_value_collisions(field_correspondence)
print(collisions)

{'lat_o': ['origin_latitude', 'destination_latitude']}


### 9. Verificar (antes de hacer la correspondencia) que un campo canonico objeto de la correspondencia no este ya en las columnas del dataframe 

In [23]:
import pandas as pd

df = pd.DataFrame({
    "origin_latitude": [-33.45, -33.46],  # ya existe canónica
    "lat_o": [-33.40, -33.41],            # columna fuente adicional
    "id_persona": [1, 2],
})

field_correspondence = {
    "origin_latitude": "lat_o",        # conflicto: canonical ya está en df.columns y source != canonical
    "user_id": "user_id",              # identidad (si existiera), debe ignorarse en esta validación
}

conflicts = [
    (canonical, source)
    for canonical, source in field_correspondence.items()
    if source != canonical and canonical in df.columns
]

# Recorrer pares y detectar el conflicto (ignorando identidad)
print("V4 conflicts:", conflicts)
assert conflicts == [("origin_latitude", "lat_o")]

# solo los canonicals conflictivos:
conflicting_canonicals = [c for c, _ in conflicts]
print("Conflicting canonicals:", conflicting_canonicals)

V4 conflicts: [('origin_latitude', 'lat_o')]
Conflicting canonicals: ['origin_latitude']


### 10. Aplicación de correspondencia de campos

In [28]:
import pandas as pd

df = pd.DataFrame({
    "id_persona": [1, 2],
    "lat_o": [-33.45, -33.46],
    "lon_o": [-70.65, -70.66],
})
print("ANTES:", list(df.columns))

field_correspondence = {
    "user_id": "id_persona",
    "origin_latitude": "lat_o",
    "origin_longitude": "lon_o",
}

rename_map = {source: canonical for canonical, source in field_correspondence.items()}
print("Rename map:", rename_map)

df = df.rename(columns=rename_map, copy=False)

print("DESPUÉS:", list(df.columns))

ANTES: ['id_persona', 'lat_o', 'lon_o']
Rename map: {'id_persona': 'user_id', 'lat_o': 'origin_latitude', 'lon_o': 'origin_longitude'}
DESPUÉS: ['user_id', 'origin_latitude', 'origin_longitude']


### 11. Revisar de que los campos canonicos esten en el dataframe y que no hay columnas duplicadas

In [33]:
import pandas as pd

df_post = pd.DataFrame({
    "user_id": [1, 2],
    "origin_latitude": [-33.45, -33.46],
    "origin_longitude": [-70.65, -70.66],
})

field_correspondence_applied = {
    "user_id": "id_persona",
    "origin_latitude": "lat_o",
    "origin_longitude": "lon_o",
}

# Verificar campos canonicos

missing_canonicals = [
    canonical
    for canonical in field_correspondence_applied.keys()
    if canonical not in df_post.columns
]

print("Missing canonicals:", missing_canonicals)

# Verificar que no hay columnas duplicadas 
has_duplicates = df_post.columns.duplicated().any()
print("Has duplicated columns?", has_duplicates)

duplicated_cols = df_post.columns[df_post.columns.duplicated()].tolist()
print("Duplicated columns:", duplicated_cols)

# Caso negativo
df_bad = df_post.copy()
df_bad.columns = ["user_id", "origin_latitude", "origin_latitude"]  # fuerza duplicado

print("\nCASO BAD - columnas:", list(df_bad.columns))
has_duplicates = df_bad.columns.duplicated().any()
print("Has duplicated columns?", has_duplicates)

duplicated_cols = df_bad.columns[df_bad.columns.duplicated()].tolist()
print("Duplicated columns:", duplicated_cols)

Missing canonicals: []
Has duplicated columns? False
Duplicated columns: []

CASO BAD - columnas: ['user_id', 'origin_latitude', 'origin_latitude']
Has duplicated columns? True
Duplicated columns: ['origin_latitude']


In [34]:
field_correspondence = {
    "user_id": "id_persona",          # se aplica
    "origin_latitude": "lat_o",       # se aplica
    "origin_longitude": "origin_longitude",  # identidad -> se ignora
}
field_correspondence_applied = {
    canonical: source
    for canonical, source in field_correspondence.items()
    if canonical != source
}

field_correspondence_applied

{'user_id': 'id_persona', 'origin_latitude': 'lat_o'}

### 12. Primer intento de chequeo de required

In [ ]:
import pandas as pd

required = {"trip_id", "origin_zone", "destination_zone"}

# Ajusta esta lista a tu diseño real (S5):
DERIVABLES_ALWAYS = {
    "movement_id",        # si lo defines como derivable
    # h3 indexes (si aplica)
    "origin_h3",
    "destination_h3",
}

DERIVABLES_IF_SINGLE_STAGE = {
    "trip_id",
    "movement_seq",
}

# DF OK: contiene todos los required
df_ok = pd.DataFrame({
    "trip_id": ["t1", "t2"],
    "origin_zone": ["A", "B"],
    "destination_zone": ["C", "D"],
})

# DF BAD: le falta destination_zone
df_bad = pd.DataFrame({
    "trip_id": ["t1", "t2"],
    "origin_zone": ["A", "B"],
})

def first_required_check(required: set[str], df: pd.DataFrame, *, single_stage: bool = False):
    missing_required = set(required) - set(df.columns)

    derivables = set(DERIVABLES_ALWAYS)
    if single_stage:
        derivables |= set(DERIVABLES_IF_SINGLE_STAGE)

    missing_derivable = {f for f in missing_required if f in derivables}
    missing_non_derivable = missing_required - missing_derivable

    return missing_required, missing_derivable, missing_non_derivable

miss, miss_der, miss_non = first_required_check(required, df_ok, single_stage=False)
print("Missing required con DF_OK (missing required):", miss)
print("Missing derivable con DF_OK (missing derivable):", miss_der)
print("Missing non-derivable con DF_OK (missing non-derivable):", miss_non)


Missing required con DF_OK (missing required): set()
Missing derivable con DF_OK (missing derivable): set()
Missing non-derivable con DF_OK (missing non-derivable): set()


#### Probar con DF BAD (falta un required no-derivable → error)

In [42]:
miss, miss_der, miss_non = first_required_check(required, df_bad, single_stage=False)
print("missing_required:", miss)
print("missing_derivable:", miss_der)
print("missing_non_derivable:", miss_non)

missing_required: {'destination_zone'}
missing_derivable: set()
missing_non_derivable: {'destination_zone'}


#### Variante para probar “derivable” condicionado a single_stage=True

Ejemplo: supongamos que required incluye movement_seq, pero permites derivarlo si single_stage=True.

In [ ]:
required2 = {"trip_id", "origin_zone", "destination_zone", "movement_seq"}

df_missing_seq = pd.DataFrame({
    "trip_id": ["t1", "t2"],
    "origin_zone": ["A", "B"],
    "destination_zone": ["C", "D"],
})

# single_stage=False -> movement_seq faltante sería no-derivable (error)
miss, miss_der, miss_non = first_required_check(required2, df_missing_seq, single_stage=False)
print("Con single_stage=False:")
print("missing_required:", miss)
print("missing_derivable:", miss_der)
print("missing_non_derivable:", miss_non)

# single_stage=True -> movement_seq faltante pasa a derivable (no aborta en S5 por esto)
miss, miss_der, miss_non = first_required_check(required2, df_missing_seq, single_stage=True)
print("\nCon single_stage=True:")
print("missing_required:", miss)
print("missing_derivable:", miss_der)
print("missing_non_derivable:", miss_non)

Con single_stage=False:
missing_required: {'movement_seq'}
missing_derivable: set()
missing_non_derivable: {'movement_seq'}

Con single_stage=True:
missing_required: {'movement_seq'}
missing_derivable: {'movement_seq'}
missing_non_derivable: set()


### 13. Deteccion de campos temporales y tier/nivel de prioridad

In [45]:
import pandas as pd

# Caso A: DF Tier 1
df_t1 = pd.DataFrame({
    "origin_time_utc": ["2026-03-01T08:00:00Z", "2026-03-01T09:00:00Z"],
    "destination_time_utc": ["2026-03-01T08:30:00Z", "2026-03-01T09:40:00Z"],
})

# Caso B: DF Tier 2
df_t2 = pd.DataFrame({
    "origin_time_local_hhmm": ["08:00", "09:00"],
    "destination_time_local_hhmm": ["08:30", "09:40"],
})

# Caso C: DF Tier 3 (sin info temporal OD)
df_t3 = pd.DataFrame({
    "trip_id": ["t1", "t2"],
    "origin_zone": ["A", "B"],
})
# Nota: en la implementacion hay que usarr schema en vez de df (mismo concepto: presencia de nombres de campo). En S5 real se usa work.columns.

TIER1_FIELDS = {"origin_time_utc", "destination_time_utc"}
TIER2_FIELDS = {"origin_time_local_hhmm", "destination_time_local_hhmm"}

# Función simple de deteccion (por columnas)
def detect_temporal_tier_from_columns(columns) -> str:
    cols = set(columns)

    if TIER1_FIELDS.issubset(cols):
        return "tier_1"
    if TIER2_FIELDS.issubset(cols):
        return "tier_2"
    return "tier_3"

# Probar con los tres dataframes y guardar variables
tier_t1 = detect_temporal_tier_from_columns(df_t1.columns)
tier_t2 = detect_temporal_tier_from_columns(df_t2.columns)
tier_t3 = detect_temporal_tier_from_columns(df_t3.columns)

print(tier_t1, tier_t2, tier_t3)

assert tier_t1 == "tier_1"
assert tier_t2 == "tier_2"
assert tier_t3 == "tier_3"

tier_1 tier_2 tier_3


#### Variante: detectar usando TripSchema

In [ ]:
from pylondrina.schema import TripSchema, FieldSpec

schema_t1 = TripSchema(fields={
    "origin_time_utc": FieldSpec(name="origin_time_utc", dtype="datetime"),
    "destination_time_utc": FieldSpec(name="destination_time_utc", dtype="datetime"),
})

schema_t2 = TripSchema(fields={
    "origin_time_local_hhmm": FieldSpec(name="origin_time_local_hhmm", dtype="string"),
    "destination_time_local_hhmm": FieldSpec(name="destination_time_local_hhmm", dtype="string"),
})

schema_t3 = TripSchema(fields={
    "trip_id": FieldSpec(name="trip_id", dtype="string"),
})

tier_schema_t1 = detect_temporal_tier_from_columns(schema_t1.fields.keys())
tier_schema_t2 = detect_temporal_tier_from_columns(schema_t2.fields.keys())
tier_schema_t3 = detect_temporal_tier_from_columns(schema_t3.fields.keys())

assert tier_schema_t1 == "tier_1"
assert tier_schema_t2 == "tier_2"
assert tier_schema_t3 == "tier_3"

# En implementacion se veria algo como:
# temporal_tier_detected = detect_temporal_tier_from_columns(work.columns)

#### 14. Normalizar o coercion de una columna que es un campo categorico

In [ ]:
import pandas as pd
import numpy as np

work = pd.DataFrame({
    # columna tipo object (strings mezclados con None/np.nan)
    "col_str_object": ["walk", None, "bus", np.nan, "car", ""],

    # columna categórica real de pandas + nulos
    "col_categorical": pd.Categorical(["A", None, "B", "A", np.nan, "C"]),
})

print("DataFrame de prueba:")
display(work)

s1 = work["col_str_object"].astype("string")
s2 = work["col_categorical"].astype("string")

s1 = s1.str.strip()
s1 = s1.replace("", pd.NA)

print("dtype original col_str_object:", work["col_str_object"].dtype)
print("dtype tras astype('string'):", s1.dtype)

print("dtype original col_categorical:", work["col_categorical"].dtype)
print("dtype tras astype('string'):", s2.dtype)

work["col_str_object"] = s1
work["col_categorical"] = s2
# Para hhcaerlo en un loop
# work[f] = work[f].astype("string").str.strip().replace("", pd.NA)

print("\nValores despues de la coerción a string (nulos convertidos a <NA>):")
display(work)

DataFrame de prueba:


,col_str_object,col_categorical
0,walk,A
1,None,NaN
2,bus,B
3,NaN,A
4,car,NaN
5,,C


dtype original col_str_object: object
dtype tras astype('string'): string
dtype original col_categorical: category
dtype tras astype('string'): string

Valores despues de la coerción a string (nulos convertidos a <NA>):


,col_str_object,col_categorical
0,walk,A
1,<NA>,<NA>
2,bus,B
3,<NA>,A
4,car,<NA>
5,<NA>,C


#### 15. La aplicacion de value_correpondence para un campo categorico en particular

In [30]:
import pandas as pd
import numpy as np

work = pd.DataFrame({
    "purpose": ["estudio", "Trabajo", "home", "ocio", None, np.nan, "" , "estudio"],
})

print("dataframe original:")
display(work)

value_correspondence = {
    "purpose": {
        "estudio": "study",
        "Trabajo": "work",
        "home": "home",   # identidad (puede existir; en applied puedes decidir incluirla o no)
    }
}

# Paso A: Normalizar a StringDtype + limpiar vacío
f = "purpose"
s = work[f].astype("string")

# limpieza mínima recomendada para categorías:
s = s.str.strip()
s = s.replace("", pd.NA)
print("\nValores normalizados (antes de replace):")
display(s)

# Paso B: Calcular “pares usados” antes de aplicar replace
vc_map = value_correspondence.get(f, {})

# valores fuente observados (pre-mapeo), ya normalizados
observed_src = set(s.dropna().unique())

# pares efectivamente usados (solo keys que aparecen)
used_pairs = {src: vc_map[src] for src in observed_src if src in vc_map}
print("\nPares de mapeo efectivamente usados (observados en datos):")
print(used_pairs)

# Paso C: Aplicar mapeo (reemplazo)
s_mapped = s.replace(vc_map)

print("\nValores después de aplicar replace con value_correspondence:")
display(s_mapped)

# Paso D — Construir value_correspondence_applied y el contador
value_correspondence_applied = {}
n_domain_mappings_applied = 0

if used_pairs:
    # si quieres excluir identidades (src==canon), filtra aquí:
    used_pairs_no_identity = {k: v for k, v in used_pairs.items() if k != v}

    value_correspondence_applied[f] = used_pairs_no_identity
    n_domain_mappings_applied += len(used_pairs_no_identity)

print("\nvalue_correspondence_applied (solo pares usados, excluyendo identidades):")
print(value_correspondence_applied)
print(f"\nNúmero de mapeos aplicados: {n_domain_mappings_applied}")

# Paso E: Escribir de vuelta al DataFrame 
work[f] = s_mapped.astype("string")
print("\nDataFrame final después de aplicar value_correspondence:")
display(work)

dataframe original:


,purpose
0,estudio
1,Trabajo
2,home
3,ocio
4,None
5,NaN
6,
7,estudio



Valores normalizados (antes de replace):


0    estudio
1    Trabajo
2       home
3       ocio
4       <NA>
5       <NA>
6       <NA>
7    estudio
Name: purpose, dtype: string


Pares de mapeo efectivamente usados (observados en datos):
{'estudio': 'study', 'Trabajo': 'work', 'home': 'home'}

Valores después de aplicar replace con value_correspondence:


0    study
1     work
2     home
3     ocio
4     <NA>
5     <NA>
6     <NA>
7    study
Name: purpose, dtype: string


value_correspondence_applied (solo pares usados, excluyendo identidades):
{'purpose': {'estudio': 'study', 'Trabajo': 'work'}}

Número de mapeos aplicados: 2

DataFrame final después de aplicar value_correspondence:


,purpose
0,study
1,work
2,home
3,ocio
4,<NA>
5,<NA>
6,<NA>
7,study


Consideracion: cómo capturar lo mínimo de domains_effective en el mismo estilo (sin implementar toda la política out-of-domain todavía):

In [37]:
# Ejemplo: base vacío + extendable True -> bootstrapping permitido
base_values = set(["study", "work", "home"])               # DomainSpec.values
extendable = True

observed_values = set(s_mapped.dropna().unique())  # canónicos post-mapeo
extended_values = observed_values - base_values    # si base vacío => extended = observed

domains_effective_snapshot = {
    "base_values": sorted(base_values),
    "observed_values": sorted(observed_values),
    "extended_values": sorted(extended_values) if extendable else [],
    "unknown_values": [],          # en esta prueba no los manejamos aún
    "aliases_applied": {},         # aún no lo aplicamos (sería otro paso)
    "extendable": extendable,
}

print("\nDomains effective snapshot:")
print(f"Base values: {domains_effective_snapshot['base_values']}")
print(f"Observed values: {domains_effective_snapshot['observed_values']}") 
print(f"Extended values (bootstrappable): {domains_effective_snapshot['extended_values']}")
print(f"Unknown values: {domains_effective_snapshot['unknown_values']}")
print(f"Aliases applied: {domains_effective_snapshot['aliases_applied']}")
print(f"Extendable: {domains_effective_snapshot['extendable']}")


Domains effective snapshot:
Base values: ['home', 'study', 'work']
Observed values: ['home', 'ocio', 'study', 'work']
Extended values (bootstrappable): ['ocio']
Unknown values: []
Aliases applied: {}
Extendable: True


##### Con un dataframe con más valores y campos

In [43]:
import pandas as pd
import numpy as np

work = pd.DataFrame({
    "purpose": [
        "estudio", "Trabajo", "home", "ocio", None, "", "Estudio", "compras", "trabajo", np.nan, "salud", "home"
    ],
    "mode": [
        "Bus", "metro", "walk", "Bike", "car", "", None, "Scooter", "bus", "WALK", np.nan, "uber"
    ],
    "gender": [
        "M", "F", "male", "female", "m", "", None, "Other", "F", np.nan, "Unknown", "f"
    ],
})
print("DataFrame original:")
display(work)

value_correspondence = {
    "purpose": {
        "estudio": "study",
        "Estudio": "study",
        "Trabajo": "work",
        "trabajo": "work",
        "compras": "shopping",
        "home": "home",   # identidad (si decides filtrarla en applied)
    },
    "mode": {
        "Bus": "bus",
        "bus": "bus",
        "metro": "metro",
        "Bike": "bike",
        "walk": "walk",
        "WALK": "walk",
        "uber": "car",     # ejemplo: lo llevas a un canónico
    },
    "gender": {
        "male": "M",
        "m": "M",
        "M": "M",          # identidad
        "female": "F",
        "f": "F",
        "F": "F",          # identidad
        "Unknown": "unknown",
    }
}

def apply_value_correspondence_for_field(work: pd.DataFrame, f: str, vc_map: dict[str, str], *, drop_identity: bool = True):
    # Normalizar a StringDtype y limpiar vacíos
    s = work[f].astype("string")
    s = s.str.strip()
    s = s.replace("", pd.NA)

    # Pares usados (existencia, no ocurrencia)
    observed_src = set(s.dropna().unique())
    used_pairs = {src: vc_map[src] for src in observed_src if src in vc_map}

    if drop_identity:
        used_pairs = {k: v for k, v in used_pairs.items() if k != v}

    # Aplicar mapping
    s_mapped = s.replace(vc_map)

    return s_mapped, used_pairs

value_correspondence_applied = {}
n_domain_mappings_applied = 0

work_mapped = work.copy(deep=True)

for f, vc_map in value_correspondence.items():
    if f not in work_mapped.columns:
        continue

    s_mapped, used_pairs = apply_value_correspondence_for_field(work_mapped, f, vc_map, drop_identity=True)
    work_mapped[f] = s_mapped  # escribir de vuelta

    if used_pairs:
        value_correspondence_applied[f] = used_pairs
        n_domain_mappings_applied += len(used_pairs)

print("DataFrame después de aplicar value_correspondence:")
display(work_mapped)
print("\nvalue_correspondence_applied (solo pares usados, excluyendo identidades):")
print(value_correspondence_applied)
print(f"\nMappings aplicados: {n_domain_mappings_applied}")

# Inspeccion: VEr que pares se registraron (por campo)
for f, pairs in value_correspondence_applied.items():
    print(f"\nField: {f}")
    for k, v in pairs.items():
        print(f"  {k!r} -> {v!r}")

DataFrame original:


,purpose,mode,gender
0,estudio,Bus,M
1,Trabajo,metro,F
2,home,walk,male
3,ocio,Bike,female
4,None,car,m
5,,,
6,Estudio,None,None
7,compras,Scooter,Other
8,trabajo,bus,F
9,NaN,WALK,NaN


DataFrame después de aplicar value_correspondence:


,purpose,mode,gender
0,study,bus,M
1,work,metro,F
2,home,walk,M
3,ocio,bike,F
4,<NA>,car,M
5,<NA>,<NA>,<NA>
6,study,<NA>,<NA>
7,shopping,Scooter,Other
8,work,bus,F
9,<NA>,walk,<NA>



value_correspondence_applied (solo pares usados, excluyendo identidades):
{'purpose': {'compras': 'shopping', 'estudio': 'study', 'Estudio': 'study', 'trabajo': 'work', 'Trabajo': 'work'}, 'mode': {'Bike': 'bike', 'uber': 'car', 'WALK': 'walk', 'Bus': 'bus'}, 'gender': {'male': 'M', 'female': 'F', 'Unknown': 'unknown', 'f': 'F', 'm': 'M'}}

Mappings aplicados: 14

Field: purpose
  'compras' -> 'shopping'
  'estudio' -> 'study'
  'Estudio' -> 'study'
  'trabajo' -> 'work'
  'Trabajo' -> 'work'

Field: mode
  'Bike' -> 'bike'
  'uber' -> 'car'
  'WALK' -> 'walk'
  'Bus' -> 'bus'

Field: gender
  'male' -> 'M'
  'female' -> 'F'
  'Unknown' -> 'unknown'
  'f' -> 'F'
  'm' -> 'M'


#### Setup común para las pruebas 16–18

In [4]:
import pandas as pd
import numpy as np

DEFAULT_UNKNOWN = "unknown"

def normalize_cat_series(s: pd.Series) -> pd.Series:
    """StringDtype + strip + convertir '' (y whitespace-only) a NA."""
    s = s.astype("string")
    s = s.str.strip()
    s = s.replace("", pd.NA)
    return s

def apply_value_correspondence(s: pd.Series, vc_map: dict[str, str] | None):
    """
    Aplica mapping src->canon y devuelve:
    - s_mapped
    - used_pairs (solo pares efectivamente usados, no ocurrencias)
    """
    if not vc_map:
        return s, {}

    observed_src = set(s.dropna().unique())
    used_pairs = {src: vc_map[src] for src in observed_src if src in vc_map}

    s_mapped = s.replace(vc_map)
    return s_mapped, used_pairs

def get_unknown_token(domain) -> str:
    """Fallback: DomainSpec no tiene unknown_value en tu código actual."""
    unk = getattr(domain, "unknown_value", None)
    if not unk:
        unk = DEFAULT_UNKNOWN
    return unk

### 16. Prueba: Mapeo a "unknown" cuando NO se permite extender (DomainSpec.extendable=False)

Qué probar:
- Un campo categórico con DomainSpec.extendable=False.
- Luego de aplicar value_correspondence, cualquier valor fuera del dominio base debe mapearse a unknown.
- Registrar estadísticas mínimas:
    - out_of_domain_values (valores únicos mapeados a unknown)
    - n_rows_mapped_to_unknown (cuántas filas quedaron en unknown)
    - unknown_value usado (fallback si no existe en DomainSpec)

In [153]:
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec
from pylondrina.importing import ImportOptions

# Dominio NO extendible (base “cerrada”)
mode_domain_closed = DomainSpec(
    values=["bus", "metro", "walk", "bike", "car"],
    extendable=False,
    aliases=None,
)

schema16 = TripSchema(
    fields={
        "mode": FieldSpec(name="mode", dtype="categorical", domain=mode_domain_closed),
    }
)

work16 = pd.DataFrame({
    "mode": [
        "Bus", "metro", "walk", "Bike", "car", "Scooter", "uber", "", None, np.nan,
        "WALK", "bus", "skate", "taxi", "metro"
    ]
})

vc16 = {
    "mode": {
        "Bus": "bus",
        "Bike": "bike",
        "WALK": "walk",
        "uber": "car",      # ejemplo: lo normalizas a car
    }
}

# Caso explícito: el usuario sí quiere conservar este campo
options16 = ImportOptions(
    strict_domains=False,   # no importa aquí; el domain es no-extendible
    selected_fields=["mode"],
)

# ------------------------------------------------------------------
# Paso nuevo: calcular target_schema_fields ANTES de procesar categóricos
# ------------------------------------------------------------------
schema_fields16 = set(schema16.fields.keys())
required_fields16 = set(schema16.required)

if options16.selected_fields:
    selected_fields16 = set(options16.selected_fields)
    invalid_selected16 = sorted(selected_fields16 - schema_fields16)
    if invalid_selected16:
        raise ValueError(
            f"selected_fields contiene campos fuera del schema: {invalid_selected16}"
        )
    target_schema_fields16 = required_fields16 | selected_fields16
else:
    target_schema_fields16 = set(schema_fields16)

print("schema_fields16:", sorted(schema_fields16))
print("required_fields16:", sorted(required_fields16))
print("selected_fields16:", sorted(options16.selected_fields) if options16.selected_fields else options16.selected_fields)
print("target_schema_fields16:", sorted(target_schema_fields16))



f = "mode"

if f not in target_schema_fields16:
    print(f"Se omite procesamiento de {f!r} porque no está en target_schema_fields16")
else:
    domain = schema16.fields[f].domain
    s = normalize_cat_series(work16[f])

    print("Dataframe con valores normalizados (string + strip + ''->NA):")
    display(s)

    # Paso 1: aplicar value_correspondence
    s_mapped, used_pairs = apply_value_correspondence(s, vc16.get(f))

    print("\nSerie después de aplicar value_correspondence:")
    display(s_mapped)

    base = set(domain.values)
    observed = set(s_mapped.dropna().unique())
    out_of_domain = observed - base

    unknown = get_unknown_token(domain)

    # Política caso 16: NO extendible -> mapear out_of_domain a unknown
    s_final = s_mapped.where(~s_mapped.isin(out_of_domain), other=unknown)

    # Estadísticas
    stats16 = {
        "unknown_value": unknown,
        "base_values": sorted(base),
        "observed_values": sorted(observed),
        "out_of_domain_values": sorted(out_of_domain),
        "n_unique_out_of_domain": len(out_of_domain),
        "n_rows_mapped_to_unknown": int((s_final == unknown).sum()),
        "used_pairs": used_pairs,  # pares usados (no ocurrencias)
    }

    work16_out = work16.copy(deep=True)
    work16_out[f] = s_final

    print("DataFrame final después de aplicar value_correspondence y mapear out-of-domain a unknown:")
    display(work16_out)

    print("\nEstadísticas del caso 16:")
    for key, value in stats16.items():
        print(f"  {key}: {value}")

schema_fields16: ['mode']
required_fields16: []
selected_fields16: ['mode']
target_schema_fields16: ['mode']
Dataframe con valores normalizados (string + strip + ''->NA):


0         Bus
1       metro
2        walk
3        Bike
4         car
5     Scooter
6        uber
7        <NA>
8        <NA>
9        <NA>
10       WALK
11        bus
12      skate
13       taxi
14      metro
Name: mode, dtype: string


Serie después de aplicar value_correspondence:


0         bus
1       metro
2        walk
3        bike
4         car
5     Scooter
6         car
7        <NA>
8        <NA>
9        <NA>
10       walk
11        bus
12      skate
13       taxi
14      metro
Name: mode, dtype: string

DataFrame final después de aplicar value_correspondence y mapear out-of-domain a unknown:


,mode
0,bus
1,metro
2,walk
3,bike
4,car
5,unknown
6,car
7,<NA>
8,<NA>
9,<NA>



Estadísticas del caso 16:
  unknown_value: unknown
  base_values: ['bike', 'bus', 'car', 'metro', 'walk']
  observed_values: ['Scooter', 'bike', 'bus', 'car', 'metro', 'skate', 'taxi', 'walk']
  out_of_domain_values: ['Scooter', 'skate', 'taxi']
  n_unique_out_of_domain: 3
  n_rows_mapped_to_unknown: 3
  used_pairs: {'Bike': 'bike', 'WALK': 'walk', 'uber': 'car', 'Bus': 'bus'}


### 17. Prueba: Error por políticas (DomainSpec.extendable=True pero options.strict_domains=True y aparece out-of-domain)

Qué probar
- Si DomainSpec.extendable=True pero options.strict_domains=True, entonces no se puede extender.
- Si aparece cualquier out-of-domain post-mapeo, debes detectar el caso y abortar (emitir issue fatal en implementación real).

In [154]:
# Dominio extendible (pero política global strict_domains=True lo prohíbe ante out-of-domain)
mode_domain_extendable = DomainSpec(
    values=["bus", "metro", "walk", "bike"],
    extendable=True,
    aliases=None,
)

schema17 = TripSchema(
    fields={
        "mode": FieldSpec(name="mode", dtype="categorical", domain=mode_domain_extendable),
    },
    required=[],
)

work17 = pd.DataFrame({
    "mode": ["bus", "metro", "walk", "scooter", "bike", "uber", "Bus", None, "", "WALK", "metro"]
})

vc17 = {
    "mode": {
        "Bus": "bus",
        "WALK": "walk",
    }
}

# Caso explícito: el usuario sí quiere conservar este campo
options17 = ImportOptions(
    strict_domains=True,   # <- política de “no extender”
    selected_fields=["mode"],
)

# ------------------------------------------------------------------
# Paso nuevo: calcular target_schema_fields ANTES de procesar categóricos
# ------------------------------------------------------------------
schema_fields17 = set(schema17.fields.keys())
required_fields17 = set(schema17.required)

if options17.selected_fields:
    selected_fields17 = set(options17.selected_fields)
    invalid_selected17 = sorted(selected_fields17 - schema_fields17)
    if invalid_selected17:
        raise ValueError(
            f"selected_fields contiene campos fuera del schema: {invalid_selected17}"
        )
    target_schema_fields17 = required_fields17 | selected_fields17
else:
    target_schema_fields17 = set(schema_fields17)

print("schema_fields17:", sorted(schema_fields17))
print("required_fields17:", sorted(required_fields17))
print("selected_fields17:", sorted(options17.selected_fields) if options17.selected_fields else options17.selected_fields)
print("target_schema_fields17:", sorted(target_schema_fields17))

f = "mode"

if f not in target_schema_fields17:
    print(f"Se omite procesamiento de {f!r} porque no está en target_schema_fields17")
else:
    domain = schema17.fields[f].domain
    s = normalize_cat_series(work17[f])
    s_mapped, used_pairs = apply_value_correspondence(s, vc17.get(f))

    base = set(domain.values)
    observed = set(s_mapped.dropna().unique())
    out_of_domain = observed - base

    print("base:", sorted(base))
    print("observed:", sorted(observed))
    print("out_of_domain:", sorted(out_of_domain))
    print("used_pairs:", used_pairs)

    # Caso 17: extendable=True pero strict_domains=True -> abort si hay out_of_domain
    try:
        if domain.extendable and options17.strict_domains and out_of_domain:
            raise ValueError(f"STRICT_DOMAINS_ABORT: out_of_domain={sorted(out_of_domain)}")
        print("NO abort (esto sería inesperado en esta prueba)")
    except ValueError as e:
        print("OK: abort detectado ->", e)

schema_fields17: ['mode']
required_fields17: []
selected_fields17: ['mode']
target_schema_fields17: ['mode']
base: ['bike', 'bus', 'metro', 'walk']
observed: ['bike', 'bus', 'metro', 'scooter', 'uber', 'walk']
out_of_domain: ['scooter', 'uber']
used_pairs: {'WALK': 'walk', 'Bus': 'bus'}
OK: abort detectado -> STRICT_DOMAINS_ABORT: out_of_domain=['scooter', 'uber']


### 18: Prueba: Extensión permitida + bootstrapping (sin mutar schema base, solo SchemaEffective)

Qué probar:
- DomainSpec.extendable=True y options.strict_domains=False → extensión real.
- Si DomainSpec.values=[] (vacío) y extendable=True → bootstrapping (added = observed).
- Se actualiza SchemaEffective (dominios efectivos), NO el schema base.

In [155]:
import pandas as pd
import numpy as np

from pylondrina.schema import TripSchema, FieldSpec, DomainSpec, TripSchemaEffective
from pylondrina.importing import ImportOptions

# Domains (extendibles)
purpose_domain_boot = DomainSpec(values=[], extendable=True, aliases=None)  # bootstrapping
mode_domain_partial = DomainSpec(values=["bus", "metro", "walk"], extendable=True, aliases=None)

schema18 = TripSchema(
    fields={
        "purpose": FieldSpec(name="purpose", dtype="categorical", domain=purpose_domain_boot),
        "mode": FieldSpec(name="mode", dtype="categorical", domain=mode_domain_partial),
    },
    required=[],
)

work18 = pd.DataFrame({
    "purpose": ["estudio", "Trabajo", "home", "ocio", "compras", None, "", "salud", "estudio", "Trabajo", np.nan, "home"],
    "mode": ["bus", "metro", "walk", "Bike", "Scooter", "", None, "Bus", "uber", "WALK", np.nan, "metro"],
})

vc18 = {
    "purpose": {"estudio": "study", "Trabajo": "work", "compras": "shopping", "home": "home"},
    "mode": {"Bike": "bike", "Bus": "bus", "WALK": "walk", "uber": "car"},
}

print("DataFrame original:")
display(work18)

# Caso de prueba: el usuario solo quiere conservar 'purpose'
options18 = ImportOptions(
    strict_domains=False,
    selected_fields=["purpose"],
)

eff18 = TripSchemaEffective()
work18_out = work18.copy(deep=True)

# ------------------------------------------------------------------
# Paso nuevo: calcular target_schema_fields ANTES de construir domains_effective
# ------------------------------------------------------------------
schema_fields18 = set(schema18.fields.keys())
required_fields18 = set(schema18.required)

if options18.selected_fields:
    selected_fields18 = set(options18.selected_fields)
    invalid_selected18 = sorted(selected_fields18 - schema_fields18)
    if invalid_selected18:
        raise ValueError(
            f"selected_fields contiene campos fuera del schema: {invalid_selected18}"
        )
    target_schema_fields18 = required_fields18 | selected_fields18
else:
    target_schema_fields18 = set(schema_fields18)

print("schema_fields18:", sorted(schema_fields18))
print("required_fields18:", sorted(required_fields18))
print("selected_fields18:", sorted(options18.selected_fields) if options18.selected_fields else options18.selected_fields)
print("target_schema_fields18:", sorted(target_schema_fields18))

# ------------------------------------------------------------------
# Procesar SOLO los campos categóricos del schema que:
# - estén en target_schema_fields18
# - existan en el dataframe
# ------------------------------------------------------------------
categorical_fields_to_process18 = [
    f for f, fs in schema18.fields.items()
    if fs.dtype == "categorical"
    and f in target_schema_fields18
    and f in work18_out.columns
]

print("categorical_fields_to_process18:", categorical_fields_to_process18)

for f in categorical_fields_to_process18:
    fs = schema18.fields[f]
    domain = fs.domain

    # normalizar + mapping
    s = normalize_cat_series(work18_out[f])
    s_mapped, used_pairs = apply_value_correspondence(s, vc18.get(f))

    base = set(domain.values or [])
    observed = set(s_mapped.dropna().unique())
    out_of_domain = observed - base

    # Política caso 18: extendable=True & strict_domains=False -> extensión real
    added_values = set()
    if out_of_domain and domain.extendable and not options18.strict_domains:
        added_values = set(out_of_domain)

    unknown = get_unknown_token(domain)

    # Guardar snapshot SOLO para campos realmente objetivo
    eff18.domains_effective[f] = {
        "base_values": sorted(base),
        "observed_values": sorted(observed),
        "extended_values": sorted(added_values),
        "unknown_values": [],  # en extensión no mapeamos a unknown
        "extendable": bool(domain.extendable),
        "unknown_value": unknown,
        "n_unique_observed": len(observed),
        "n_added": len(added_values),
        "value_correspondence_applied": {k: v for k, v in used_pairs.items() if k != v},
    }

    if added_values:
        eff18.overrides.setdefault(f, {})
        eff18.overrides[f].setdefault("reasons", []).append("domain_extended")
        eff18.overrides[f]["added_values"] = sorted(added_values)

    # escribir de vuelta solo para campos procesados
    work18_out[f] = s_mapped.astype("string")

print("Domains effective construidos en SchemaEffective:")
display(eff18.domains_effective)

print("Overrides aplicados:")
display(eff18.overrides)

print("DataFrame resultante:")
display(work18_out)

DataFrame original:


,purpose,mode
0,estudio,bus
1,Trabajo,metro
2,home,walk
3,ocio,Bike
4,compras,Scooter
5,None,
6,,None
7,salud,Bus
8,estudio,uber
9,Trabajo,WALK


schema_fields18: ['mode', 'purpose']
required_fields18: []
selected_fields18: ['purpose']
target_schema_fields18: ['purpose']
categorical_fields_to_process18: ['purpose']
Domains effective construidos en SchemaEffective:


{'purpose': {'base_values': [],
  'observed_values': ['home', 'ocio', 'salud', 'shopping', 'study', 'work'],
  'extended_values': ['home', 'ocio', 'salud', 'shopping', 'study', 'work'],
  'unknown_values': [],
  'extendable': True,
  'unknown_value': 'unknown',
  'n_unique_observed': 6,
  'n_added': 6,
  'value_correspondence_applied': {'estudio': 'study',
   'compras': 'shopping',
   'Trabajo': 'work'}}}

Overrides aplicados:


{'purpose': {'reasons': ['domain_extended'],
  'added_values': ['home', 'ocio', 'salud', 'shopping', 'study', 'work']}}

DataFrame resultante:


,purpose,mode
0,study,bus
1,work,metro
2,home,walk
3,ocio,Bike
4,shopping,Scooter
5,<NA>,
6,<NA>,None
7,salud,Bus
8,study,uber
9,work,WALK


### 19. Pruebar: comportamiento de set() y operaciones (para dominios canónicos/observados)

In [ ]:
# Dominio base "canon" del schema (puede venir con repetidos por error humano)
base_list = ["bus", "metro", "walk", "bike", "walk", "metro"]

# Valores observados en el DF (post-mapeo): incluye valores fuera del dominio y repetidos
observed_list = ["bus", "walk", "scooter", "car", "car", "bus", ""]

# Valores agregados por extensión (ejemplo), puede venir repetido
added_list = ["scooter", "car", "car"]

unknown = "unknown"

base = set(base_list)
observed = set(observed_list)
added = set(added_list)
# hacer sets elimina duplicados automáticamente
print("base:", base)
print("observed:", observed)
print("added:", added)
# Para volver a convertir en lista: list(base)

print("\nbool(base):", bool(base))
print("bool(set()):", bool(set()))
# Esto se usa para decidir cosas como extended = bool(added_values).

out_of_domain = observed - base
print("\nobserved - base:", out_of_domain)
print("base - observed:", base - observed)
# Sirve para obtener los valores que no estan en el dominio base 

effective = base | added | {unknown}
print("\neffective:", effective)
print("effective sorted:", sorted(effective))
# Sirve para hacer union de los sets y para construir el dominio efectivo final (base + extendidos + unknown)
# sorted ademas entrega una lista ordenada, lo cual es útil para presentación y para tests (evita problemas de orden en asserts)

print("\nobserved ∩ base:", observed & base)
# Sirve para interseccion de sets, por ejemplo para contar cuántos valores observados ya estaban en el dominio base (puede ser útil para estadísticas)

print("\nobserved Δ base:", observed ^ base)
# Sirve para obtener la diferencia simétrica (valores que están en observed o en base pero no en ambos), puede ser útil para análisis exploratorio de la relación entre lo observado y el dominio base. En este caso muestra los valores que son exclusivos de cada conjunto.

base: {'bus', 'walk', 'bike', 'metro'}
observed: {'', 'walk', 'scooter', 'bus', 'car'}
added: {'car', 'scooter'}

bool(base): True
bool(set()): False

observed - base: {'', 'car', 'scooter'}
base - observed: {'bike', 'metro'}

effective: {'bus', 'car', 'unknown', 'walk', 'bike', 'metro', 'scooter'}
effective sorted: ['bike', 'bus', 'car', 'metro', 'scooter', 'unknown', 'walk']


### 20. La Coercion para cada tipo de datos que pueden venir en un dataframe

In [35]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    # ---- STRING ---- (12)
    "s_from_str": ["hola", "", "  mundo  ", None, np.nan, "x", "  ", "y", "z", "", None, "fin"],
    "s_already_string": pd.Series(["a", pd.NA, "", " b ", None, "c", "  ", "d", pd.NA, "", "e", "f"], dtype="string"),

    # ---- INT ---- (12)
    "i_from_str": ["1", "02", "", None, "3.0", "x", "1,000", np.nan, " -7 ", "5", "0", " 8 "],
    "i_already_int": pd.Series([1, pd.NA, None, 0, -5, pd.NA, 10, 2, pd.NA, 7, None, 3], dtype="Int64"),

    # ---- FLOAT ---- (12)
    "f_from_str": ["1.5", "2", "", None, "NaN", "3,14", "x", np.nan, " -0.25 ", "0", "5.5", " 7.0 "],
    "f_already_float": pd.Series([1.25, np.nan, None, 0.0, -3.5, np.nan, 2.0, None, np.nan, 10.5, -0.1, np.nan], dtype="float64"),

    # ---- BOOL ---- (12)
    "b_from_str": ["true", "False", "1", "0", "yes", "no", "", None, "T", "F", "maybe", np.nan],
    "b_already_bool": pd.Series([True, False, pd.NA, None, True, pd.NA, False, True, None, pd.NA, True, False], dtype="boolean"),

    # ---- DATETIME ---- (12)  (aunque dijimos dejarlo para S8, te lo dejo consistente)
    "dt_from_str": ["2026-03-01 08:00", "2026-03-01T09:15:00Z", "", None, "not_a_date", np.nan,
                    "2026-03-02 10:00", "2026-03-03", "2026/03/04 12:00", "03-05-2026", " ", "2026-03-06T00:00:00"],
    "dt_already_datetime": pd.to_datetime(
        pd.Series(["2026-03-01", None, "2026-03-02", pd.NaT, "2026-03-03", None,
                   "2026-03-04", pd.NaT, "2026-03-05", None, "2026-03-06", pd.NaT]),
        errors="coerce"
    ),
})

print("DataFrame original con tipos mixtos y valores sucios:")
display(df)

# Coerción de string: astype + strip + reemplazo de '' por NA
s1 = df["s_from_str"].astype("string").str.strip().replace("", pd.NA)
s2 = df["s_already_string"].astype("string").str.strip().replace("", pd.NA)

print("Después de coerción a string, strip y limpieza de vacíos:")
print(s1.dtype, s2.dtype)
display(pd.DataFrame({"s_from_str": s1, "s_already_string": s2}))

# Coerción a numérico Int: to_numeric + astype(Int64) para enteros (maneja no-convertibles como NA)
i1_num = pd.to_numeric(df["i_from_str"], errors="coerce")
i1 = i1_num.astype("Int64")

i2 = df["i_already_int"].astype("Int64")  # idempotente

print("\nDespués de coerción a Int64 (numérica + limpieza de no-convertibles):")
print(i1.dtype, i2.dtype)
display(pd.DataFrame({"i_from_str": i1, "i_already_int": i2}))


f1 = pd.to_numeric(df["f_from_str"], errors="coerce")
f2 = pd.to_numeric(df["f_already_float"], errors="coerce")  # robusto

print("\nDespués de coerción a float64 (numérica + limpieza de no-convertibles):")
print(f1.dtype, f2.dtype)
display(pd.DataFrame({"f_from_str": f1, "f_already_float": f2}))

# Coerción a booleano: normalización a string + mapping manual (no hay función built-in que maneje tantos casos)
TRUE_SET = {"true", "t", "1", "yes", "y"}
FALSE_SET = {"false", "f", "0", "no", "n"}

def coerce_bool_series(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip().str.lower().replace("", pd.NA)
    out = pd.Series([pd.NA] * len(s), dtype="boolean")
    out[s.isin(TRUE_SET)] = True
    out[s.isin(FALSE_SET)] = False
    # lo no reconocido queda <NA>
    return out

b1 = coerce_bool_series(df["b_from_str"])
b2 = df["b_already_bool"].astype("boolean")  # idempotente

print("\nDespués de coerción a boolean (normalización + mapping manual):")
print(b1.dtype, b2.dtype)
display(pd.DataFrame({"b_from_str": b1, "b_already_bool": b2}))

# Coerción a datetime: to_datetime con errors='coerce' para manejar no-fechas, Aunque esto es de S8, lo dejo aquí para consistencia y porque es un caso común de coerción robusta.
dt1 = pd.to_datetime(df["dt_from_str"], errors="coerce")   # invalid -> NaT
dt2 = pd.to_datetime(df["dt_already_datetime"], errors="coerce")  # idempotente

print("\nDespués de coerción a datetime (manejo de no-fechas):")
print(dt1.dtype, dt2.dtype)
display(pd.DataFrame({"dt_from_str": dt1, "dt_already_datetime": dt2}))

# Finalmente, se puede escribir de vuelta al DataFrame o crear uno nuevo con las columnas coercidas para inspección.
df_coerced = df.copy(deep=True)

df_coerced["s_from_str"] = s1
df_coerced["s_already_string"] = s2
df_coerced["i_from_str"] = i1
df_coerced["i_already_int"] = i2
df_coerced["f_from_str"] = f1
df_coerced["f_already_float"] = f2
df_coerced["b_from_str"] = b1
df_coerced["b_already_bool"] = b2
df_coerced["dt_from_str"] = dt1
df_coerced["dt_already_datetime"] = dt2

print("\nDataFrame final con columnas coercidas:")
display(df_coerced)
print("\nTipos de datos finales:")
display(df_coerced.dtypes)

DataFrame original con tipos mixtos y valores sucios:


,s_from_str,s_already_string,i_from_str,i_already_int,f_from_str,f_already_float,b_from_str,b_already_bool,dt_from_str,dt_already_datetime
0,hola,a,1,1,1.5,1.25,true,True,2026-03-01 08:00,2026-03-01
1,,<NA>,02,<NA>,2,NaN,False,False,2026-03-01T09:15:00Z,NaT
2,mundo,,,<NA>,,NaN,1,<NA>,,2026-03-02
3,None,b,None,0,None,0.00,0,<NA>,None,NaT
4,NaN,<NA>,3.0,-5,NaN,-3.50,yes,True,not_a_date,2026-03-03
5,x,c,x,<NA>,"3,14",NaN,no,<NA>,NaN,NaT
6,,,"1,000",10,x,2.00,,False,2026-03-02 10:00,2026-03-04
7,y,d,NaN,2,NaN,NaN,None,True,2026-03-03,NaT
8,z,<NA>,-7,<NA>,-0.25,NaN,T,<NA>,2026/03/04 12:00,2026-03-05
9,,,5,7,0,10.50,F,<NA>,03-05-2026,NaT


Después de coerción a string, strip y limpieza de vacíos:
string string


,s_from_str,s_already_string
0,hola,a
1,<NA>,<NA>
2,mundo,<NA>
3,<NA>,b
4,<NA>,<NA>
5,x,c
6,<NA>,<NA>
7,y,d
8,z,<NA>
9,<NA>,<NA>



Después de coerción a Int64 (numérica + limpieza de no-convertibles):
Int64 Int64


...

,dt_from_str,dt_already_datetime
0,2026-03-01 08:00:00,2026-03-01
1,NaT,NaT
2,NaT,2026-03-02
3,NaT,NaT
4,NaT,2026-03-03
5,NaT,NaT
6,2026-03-02 10:00:00,2026-03-04
7,NaT,NaT
8,NaT,2026-03-05
9,NaT,NaT



DataFrame final con columnas coercidas:


,s_from_str,s_already_string,i_from_str,i_already_int,f_from_str,f_already_float,b_from_str,b_already_bool,dt_from_str,dt_already_datetime
0,hola,a,1,1,1.50,1.25,True,True,2026-03-01 08:00:00,2026-03-01
1,<NA>,<NA>,2,<NA>,2.00,NaN,False,False,NaT,NaT
2,mundo,<NA>,<NA>,<NA>,NaN,NaN,True,<NA>,NaT,2026-03-02
3,<NA>,b,<NA>,0,NaN,0.00,False,<NA>,NaT,NaT
4,<NA>,<NA>,3,-5,NaN,-3.50,True,True,NaT,2026-03-03
5,x,c,<NA>,<NA>,NaN,NaN,False,<NA>,NaT,NaT
6,<NA>,<NA>,<NA>,10,NaN,2.00,<NA>,False,2026-03-02 10:00:00,2026-03-04
7,y,d,<NA>,2,NaN,NaN,<NA>,True,NaT,NaT
8,z,<NA>,-7,<NA>,-0.25,NaN,True,<NA>,NaT,2026-03-05
9,<NA>,<NA>,5,7,0.00,10.50,False,<NA>,NaT,NaT



Tipos de datos finales:


s_from_str             string[python]
s_already_string       string[python]
i_from_str                      Int64
i_already_int                   Int64
f_from_str                    float64
f_already_float               float64
b_from_str                    boolean
b_already_bool                boolean
dt_from_str            datetime64[ns]
dt_already_datetime    datetime64[ns]
dtype: object

#### Considerando revision de tipos de las columnas

In [42]:
import pandas as pd
import numpy as np
from pandas.api import types as ptypes

df = pd.DataFrame({
    # ---- STRING ---- (12)
    "s_from_str": ["hola", "", "  mundo  ", None, np.nan, "x", "  ", "y", "z", "", None, "fin"],
    "s_already_string": pd.Series(["a", pd.NA, "", " b ", None, "c", "  ", "d", pd.NA, "", "e", "f"], dtype="string"),

    # ---- INT ---- (12)
    "i_from_str": ["1", "02", "", None, "3.0", "x", "1,000", np.nan, " -7 ", "5", "0", " 8 "],
    "i_already_int": pd.Series([1, pd.NA, None, 0, -5, pd.NA, 10, 2, pd.NA, 7, None, 3], dtype="Int64"),

    # ---- FLOAT ---- (12)
    "f_from_str": ["1.5", "2", "", None, "NaN", "3,14", "x", np.nan, " -0.25 ", "0", "5.5", " 7.0 "],
    "f_already_float": pd.Series([1.25, np.nan, None, 0.0, -3.5, np.nan, 2.0, None, np.nan, 10.5, -0.1, np.nan], dtype="float64"),

    # ---- BOOL ---- (12)
    "b_from_str": ["true", "False", "1", "0", "yes", "no", "", None, "T", "F", "maybe", np.nan],
    "b_already_bool": pd.Series([True, False, pd.NA, None, True, pd.NA, False, True, None, pd.NA, True, False], dtype="boolean"),

    # ---- DATETIME ---- (12)
    "dt_from_str": ["2026-03-01 08:00", "2026-03-01T09:15:00Z", "", None, "not_a_date", np.nan,
                    "2026-03-02 10:00", "2026-03-03", "2026/03/04 12:00", "03-05-2026", " ", "2026-03-06T00:00:00"],
    "dt_already_datetime": pd.to_datetime(
        pd.Series(["2026-03-01", None, "2026-03-02", pd.NaT, "2026-03-03", None,
                   "2026-03-04", pd.NaT, "2026-03-05", None, "2026-03-06", pd.NaT]),
        errors="coerce"
    ),
})

print("DataFrame original con tipos mixtos y valores sucios:")
display(df)


TRUE_SET = {"true", "t", "1", "yes", "y"}
FALSE_SET = {"false", "f", "0", "no", "n"}

def is_already_correct_dtype(s: pd.Series, expected: str) -> bool:
    dt = s.dtype

    if expected == "string":
        return str(dt) == "string"

    if expected == "int":
        # estándar Golondrina: Int64 nullable
        return str(dt) == "Int64"

    if expected == "float":
        return ptypes.is_float_dtype(dt)

    if expected == "bool":
        # estándar Golondrina: boolean nullable
        return str(dt) == "boolean"

    if expected == "datetime":
        # en S7 basta con que sea datetime; normalización/tz en S8
        return ptypes.is_datetime64_any_dtype(dt)

    raise ValueError(f"Unknown expected dtype: {expected}")


def coerce_bool_series(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip().str.lower().replace("", pd.NA)
    out = pd.Series([pd.NA] * len(s), dtype="boolean")
    out[s.isin(TRUE_SET)] = True
    out[s.isin(FALSE_SET)] = False
    return out


def coerce_series_to_dtype(
    s: pd.Series,
    expected: str,
    *,
    parse_datetime: bool = False,      # recomendado: False en S7, True cuando empieces S8
    empty_string_as_na: bool = True,   # recomendado: True para categóricos/strings
):
    """
    Devuelve: (s_out, stats)
    stats incluye dtype_before/after y cuántos NA quedaron.
    """
    before_dtype = str(s.dtype)
    na_before = int(pd.isna(s).sum())

    if expected == "string":
        out = s.astype("string")
        out = out.str.strip()
        if empty_string_as_na:
            out = out.replace("", pd.NA)

    elif expected == "int":
        # Nota: esto convierte "3.0" -> 3.0 -> Int64 (queda 3)
        out = pd.to_numeric(s, errors="coerce")
        out = out.astype("Int64")

    elif expected == "float":
        out = pd.to_numeric(s, errors="coerce")

    elif expected == "bool":
        out = coerce_bool_series(s)

    elif expected == "datetime":
        # Conservador en S7:
        if ptypes.is_datetime64_any_dtype(s.dtype):
            out = s
        else:
            if parse_datetime:
                # S8 lo hará con más reglas; aquí es “parse básico”
                tmp = s.astype("string").str.strip().replace("", pd.NA)
                out = pd.to_datetime(tmp, errors="coerce")
            else:
                # no tocar en S7 si no es datetime
                out = s

    else:
        raise ValueError(f"Unknown expected dtype: {expected}")

    after_dtype = str(out.dtype)
    na_after = int(pd.isna(out).sum())

    stats = {
        "expected": expected,
        "dtype_before": before_dtype,
        "dtype_after": after_dtype,
        "na_before": na_before,
        "na_after": na_after,
        "na_delta": na_after - na_before,
        "changed": (before_dtype != after_dtype),
        "already_correct": is_already_correct_dtype(s, expected),
    }
    return out, stats

expected_types = {
    "s_from_str": "string",
    "s_already_string": "string",
    "i_from_str": "int",
    "i_already_int": "int",
    "f_from_str": "float",
    "f_already_float": "float",
    "b_from_str": "bool",
    "b_already_bool": "bool",
    # datetime: en S7 conservador (no parse). Para ver parse, activa parse_datetime=True abajo.
    "dt_from_str": "datetime",
    "dt_already_datetime": "datetime",
}

df_coerced = df.copy(deep=True)
stats_rows = []

for col, expected in expected_types.items():
    s_out, st = coerce_series_to_dtype(df_coerced[col], expected, parse_datetime=False)
    df_coerced[col] = s_out
    st["column"] = col
    stats_rows.append(st)

stats = pd.DataFrame(stats_rows).loc[:, ["column","expected","dtype_before","dtype_after","already_correct","na_before","na_after","na_delta","changed"]]

print("\nEstadisticas de coerción por columna:")
display(stats)

print("\nTipos finales")
display(df_coerced.dtypes)

df_dt_parsed = df.copy(deep=True)
s_out, st = coerce_series_to_dtype(df_dt_parsed["dt_from_str"], "datetime", parse_datetime=True)
df_dt_parsed["dt_from_str"] = s_out

st, df_dt_parsed[["dt_from_str","dt_already_datetime"]].head(12)

DataFrame original con tipos mixtos y valores sucios:


,s_from_str,s_already_string,i_from_str,i_already_int,f_from_str,f_already_float,b_from_str,b_already_bool,dt_from_str,dt_already_datetime
0,hola,a,1,1,1.5,1.25,true,True,2026-03-01 08:00,2026-03-01
1,,<NA>,02,<NA>,2,NaN,False,False,2026-03-01T09:15:00Z,NaT
2,mundo,,,<NA>,,NaN,1,<NA>,,2026-03-02
3,None,b,None,0,None,0.00,0,<NA>,None,NaT
4,NaN,<NA>,3.0,-5,NaN,-3.50,yes,True,not_a_date,2026-03-03
5,x,c,x,<NA>,"3,14",NaN,no,<NA>,NaN,NaT
6,,,"1,000",10,x,2.00,,False,2026-03-02 10:00,2026-03-04
7,y,d,NaN,2,NaN,NaN,None,True,2026-03-03,NaT
8,z,<NA>,-7,<NA>,-0.25,NaN,T,<NA>,2026/03/04 12:00,2026-03-05
9,,,5,7,0,10.50,F,<NA>,03-05-2026,NaT



Estadisticas de coerción por columna:


,column,expected,dtype_before,dtype_after,already_correct,na_before,na_after,na_delta,changed
0,s_from_str,string,object,string,False,3,6,3,True
1,s_already_string,string,string,string,True,3,6,3,False
2,i_from_str,int,object,Int64,False,2,5,3,True
3,i_already_int,int,Int64,Int64,True,5,5,0,False
4,f_from_str,float,object,float64,False,2,6,4,True
5,f_already_float,float,float64,float64,True,6,6,0,False
6,b_from_str,bool,object,boolean,False,2,4,2,True
7,b_already_bool,bool,boolean,boolean,True,5,5,0,False
8,dt_from_str,datetime,object,object,False,2,2,0,False
9,dt_already_datetime,datetime,datetime64[ns],datetime64[ns],True,6,6,0,False



Tipos finales


s_from_str             string[python]
s_already_string       string[python]
i_from_str                      Int64
i_already_int                   Int64
f_from_str                    float64
f_already_float               float64
b_from_str                    boolean
b_already_bool                boolean
dt_from_str                    object
dt_already_datetime    datetime64[ns]
dtype: object

({'expected': 'datetime',
  'dtype_before': 'object',
  'dtype_after': 'datetime64[ns]',
  'na_before': 2,
  'na_after': 10,
  'na_delta': 8,
  'changed': True,
  'already_correct': False},
            dt_from_str dt_already_datetime
 0  2026-03-01 08:00:00          2026-03-01
 1                  NaT                 NaT
 2                  NaT          2026-03-02
 3                  NaT                 NaT
 4                  NaT          2026-03-03
 5                  NaT                 NaT
 6  2026-03-02 10:00:00          2026-03-04
 7                  NaT                 NaT
 8                  NaT          2026-03-05
 9                  NaT                 NaT
 10                 NaT          2026-03-06
 11                 NaT                 NaT)

### 21. Los distintos tipos de parseos a campos datetime

#### Tier 1: schema con campos datetime

In [69]:
import pandas as pd
import numpy as np

from pylondrina.schema import TripSchema, FieldSpec, DomainSpec  # DomainSpec no se usa aquí, pero dejo el import estándar
from pylondrina.importing import ImportOptions

opt_iana   = ImportOptions(source_timezone="America/Santiago")  # IANA
opt_offset = ImportOptions(source_timezone="-03:00")            # offset fijo
opt_utc    = ImportOptions(source_timezone="UTC")               # UTC
opt_z      = ImportOptions(source_timezone="Z")                 # Z
opt_bad    = ImportOptions(source_timezone="America/Sancthiagou")  # inválido

# --- Schema Tier 1 (varios datetime fields) ---
schema_t1 = TripSchema(
    fields={
        "dt_utc_explicit": FieldSpec(name="dt_utc_explicit", dtype="datetime"),
        "dt_tzaware_offset": FieldSpec(name="dt_tzaware_offset", dtype="datetime"),
        "dt_naive": FieldSpec(name="dt_naive", dtype="datetime"),
        "dt_str_iso": FieldSpec(name="dt_str_iso", dtype="datetime"),
        "dt_numeric": FieldSpec(name="dt_numeric", dtype="datetime"),
        "dt_foursquare": FieldSpec(name="dt_foursquare", dtype="datetime"),
        "dt_adatrap": FieldSpec(name="dt_adatrap", dtype="datetime"),
    }
)

N = 10  # todas las columnas mismo largo

work_t1 = pd.DataFrame({
    # (1) tz-aware UTC explícito (ya listo)
    "dt_utc_explicit": pd.to_datetime(
        ["2026-03-01T08:00:00Z", "2026-03-01T09:00:00Z", None, "2026-03-02T10:00:00Z",
         "2026-03-03T11:00:00Z", "2026-03-03T12:00:00Z", "2026-03-04T08:30:00Z",
         "2026-03-04T09:30:00Z", "2026-03-05T10:30:00Z", "2026-03-06T00:00:00Z"],
        utc=True, errors="coerce"
    ),

    # (2) tz-aware con offset fijo (ej -03:00), debe convertirse a UTC
    "dt_tzaware_offset": pd.to_datetime(
        ["2026-03-01T08:00:00-03:00", "2026-03-01T09:00:00-03:00", None, "2026-03-02T10:00:00-03:00",
         "2026-03-03T11:00:00-03:00", "2026-03-03T12:00:00-03:00", "2026-03-04T08:30:00-03:00",
         "2026-03-04T09:30:00-03:00", "2026-03-05T10:30:00-03:00", "2026-03-06T00:00:00-03:00"],
        errors="coerce"
    ),

    # (3) datetime naive (sin tz)
    "dt_naive": pd.to_datetime(
        ["2026-03-01 08:00:00", "2026-03-01 09:00:00", None, "2026-03-02 10:00:00",
         "2026-03-03 11:00:00", "2026-03-03 12:00:00", "2026-03-04 08:30:00",
         "2026-03-04 09:30:00", "2026-03-05 10:30:00", "2026-03-06 00:00:00"],
        errors="coerce"
    ),

    # (4) string ISO (algunos inválidos) -> parseable a datetime
    "dt_str_iso": [
        "2026-03-01 08:00:00", "2026-03-01 09:15:00", None, "",
        "2026-03-02 10:00:00", "2026-03-03 11:00:00", np.nan,
        "2026-03-04 12:00:00", "2026-03-05 00:00:00", "2026-03-06 01:02:03"
    ],

    # (5) numérica (NO parsear epoch en v1)
    "dt_numeric": [1710000000, 1710003600, None, np.nan, 0, 123.45, 9999999999, -1, 42, None],

    # (6) Foursquare real (tz +0000)
    "dt_foursquare": [
        "Tue Apr 03 18:00:06 +0000 2012",
        "Tue Apr 03 18:00:07 +0000 2012",
        None,
        "Wed Jan 29 16:44:24 +0000 2014",
        "Wed Jan 29 16:44:25 +0000 2014",
        "Thu Feb 13 09:10:11 +0000 2014",
        np.nan,
        "Fri Mar 14 12:30:00 +0000 2014",
        "Sat Apr 12 00:00:00 +0000 2014",
        None,
    ],

    # (7) ADATRAP real (naive)
    "dt_adatrap": [
        "2019-05-19 16:10:32",
        "2019-05-19 11:33:19",
        None,
        "2019-05-19 17:55:20",
        "2019-05-20 08:01:00",
        "",
        "2019-05-21 12:00:00",
        np.nan,
        "2019-05-22 23:59:59",
        "not_a_date",  
    ],
})

print("DataFrame original con varios formatos de datetime:")
display(work_t1)
print("\nTipos de datos originales:")
display(work_t1.dtypes)

def datetime_fields_present(schema: TripSchema, work: pd.DataFrame) -> list[str]:
    return [
        fname
        for fname, fs in schema.fields.items()
        if fs.dtype == "datetime" and fname in work.columns
    ]

dt_fields = datetime_fields_present(schema_t1, work_t1)

print("\nCampos de tipo datetime presentes en el DataFrame según el schema:")
print(dt_fields)

DataFrame original con varios formatos de datetime:


,dt_utc_explicit,dt_tzaware_offset,dt_naive,dt_str_iso,dt_numeric,dt_foursquare,dt_adatrap
0,2026-03-01 08:00:00+00:00,2026-03-01 08:00:00-03:00,2026-03-01 08:00:00,2026-03-01 08:00:00,1.710000e+09,Tue Apr 03 18:00:06 +0000 2012,2019-05-19 16:10:32
1,2026-03-01 09:00:00+00:00,2026-03-01 09:00:00-03:00,2026-03-01 09:00:00,2026-03-01 09:15:00,1.710004e+09,Tue Apr 03 18:00:07 +0000 2012,2019-05-19 11:33:19
2,NaT,NaT,NaT,None,NaN,None,None
3,2026-03-02 10:00:00+00:00,2026-03-02 10:00:00-03:00,2026-03-02 10:00:00,,NaN,Wed Jan 29 16:44:24 +0000 2014,2019-05-19 17:55:20
4,2026-03-03 11:00:00+00:00,2026-03-03 11:00:00-03:00,2026-03-03 11:00:00,2026-03-02 10:00:00,0.000000e+00,Wed Jan 29 16:44:25 +0000 2014,2019-05-20 08:01:00
5,2026-03-03 12:00:00+00:00,2026-03-03 12:00:00-03:00,2026-03-03 12:00:00,2026-03-03 11:00:00,1.234500e+02,Thu Feb 13 09:10:11 +0000 2014,
6,2026-03-04 08:30:00+00:00,2026-03-04 08:30:00-03:00,2026-03-04 08:30:00,NaN,1.000000e+10,NaN,2019-05-21 12:00:00
7,2026-03-04 09:30:00+00:00,2026-03-04 09:30:00-03:00,2026-03-04 09:30:00,2026-03-04 12:00:00,-1.000000e+00,Fri Mar 14 12:30:00 +0000 2014,NaN
8,2026-03-05 10:30:00+00:00,2026-03-05 10:30:00-03:00,2026-03-05 10:30:00,2026-03-05 00:00:00,4.200000e+01,Sat Apr 12 00:00:00 +0000 2014,2019-05-22 23:59:59
9,2026-03-06 00:00:00+00:00,2026-03-06 00:00:00-03:00,2026-03-06 00:00:00,2026-03-06 01:02:03,NaN,None,not_a_date



Tipos de datos originales:


dt_utc_explicit            datetime64[ns, UTC]
dt_tzaware_offset    datetime64[ns, UTC-03:00]
dt_naive                        datetime64[ns]
dt_str_iso                              object
dt_numeric                             float64
dt_foursquare                           object
dt_adatrap                              object
dtype: object


Campos de tipo datetime presentes en el DataFrame según el schema:
['dt_utc_explicit', 'dt_tzaware_offset', 'dt_naive', 'dt_str_iso', 'dt_numeric', 'dt_foursquare', 'dt_adatrap']


In [75]:
import re
from pandas.api import types as ptypes
from zoneinfo import ZoneInfo

OFFSET_RE = re.compile(r"^[+-](?:0\d|1\d|2[0-3]):[0-5]\d$")

def normalize_source_timezone(tz: str | None):
    if tz is None:
        return None, "none"
    tz = str(tz).strip()
    if tz == "":
        return None, "empty"
    if tz.upper() in {"UTC", "Z"}:
        return "UTC", "utc"
    if OFFSET_RE.match(tz):
        return tz, "offset"
    # IANA
    try:
        ZoneInfo(tz)  # valida
        return tz, "iana"
    except Exception:
        return None, "invalid"

def normalize_datetime_column(s: pd.Series, *, source_timezone: str | None):
    tz_norm, tz_kind = normalize_source_timezone(source_timezone)

    # Caso D: numérica -> no parsear epoch en v1
    if ptypes.is_numeric_dtype(s.dtype):
        out = pd.Series([pd.NaT] * len(s), dtype="datetime64[ns]")
        return out, {"status": "not_parsed_numeric", "tz_kind": tz_kind, "n_nat": int(out.isna().sum())}

    # Caso A/C: si ya es datetime
    if ptypes.is_datetime64_any_dtype(s.dtype):
        # tz-aware
        if isinstance(s.dtype, pd.DatetimeTZDtype):
            tzname = str(getattr(s.dtype, "tz", ""))
            if tzname in {"UTC", "utc"}:
                return s, {"status": "utc", "tz_kind": tz_kind, "n_nat": int(s.isna().sum())}
            return s.dt.tz_convert("UTC"), {"status": "tzaware_to_utc", "tz_kind": tz_kind, "n_nat": int(s.isna().sum())}

        # naive datetime
        if tz_norm is None:
            return s, {"status": "naive_unconverted", "tz_kind": tz_kind, "n_nat": int(s.isna().sum())}
        localized = s.dt.tz_localize(tz_norm).dt.tz_convert("UTC")
        return localized, {"status": "naive_localized_to_utc", "tz_kind": tz_kind, "n_nat": int(localized.isna().sum())}

    # Caso B: string/objeto -> intentar parse
    tmp = s.astype("string").str.strip().replace("", pd.NA)
    #parsed = pd.to_datetime(tmp, errors="coerce")  # sin utc=True para no forzar naive
    parsed = pd.to_datetime(tmp, format="mixed", errors="coerce")

    # FIX: chequear parsed.dtype, no s.dtype
    if isinstance(parsed.dtype, pd.DatetimeTZDtype):
        return parsed.dt.tz_convert("UTC"), {
            "status": "string_tzaware_to_utc",
            "tz_kind": tz_kind,
            "n_nat": int(parsed.isna().sum()),
        }

    # naive parseado
    if ptypes.is_datetime64_any_dtype(parsed.dtype):
        if tz_norm is None:
            return parsed, {"status": "string_naive_unconverted", "tz_kind": tz_kind, "n_nat": int(parsed.isna().sum())}
        localized = parsed.dt.tz_localize(tz_norm).dt.tz_convert("UTC")
        return localized, {"status": "string_naive_localized_to_utc", "tz_kind": tz_kind, "n_nat": int(localized.isna().sum())}

    # Si quedó object raro, lo tratamos como fallo de parse
    out = pd.Series([pd.NaT] * len(s), dtype="datetime64[ns]")
    return out, {"status": "parse_failed", "tz_kind": tz_kind, "n_nat": int(out.isna().sum())}

def run_s8_lab(work: pd.DataFrame, source_timezone: str | None):
    out = work.copy(deep=True)
    status_by_col = {}
    for col in datetime_fields_present(schema_t1, out):
        out[col], status_by_col[col] = normalize_datetime_column(out[col], source_timezone=source_timezone)
    return out, status_by_col

out_iana, status_iana = run_s8_lab(work_t1, source_timezone="America/Santiago")
out_off,  status_off  = run_s8_lab(work_t1, source_timezone="-03:00")
out_none, status_none = run_s8_lab(work_t1, source_timezone=None)
out_bad,  status_bad  = run_s8_lab(work_t1, source_timezone="America/Sancthiagou")

# Mostrar resultados y status para timezone IANA en el caso de America/Santiago
print("Resultados con timezone IANA:")
display(out_iana)

print("Status de normalización con timezone IANA:")
for col, status in status_iana.items():
    print(f"{col}: {status}")


# Mostrar resultados y status para timezone offset fijo en el caso de -03:00
print("\nResultados con timezone offset fijo:")
display(out_off)

print("Status de normalización con timezone offset fijo:")
for col, status in status_off.items():
    print(f"{col}: {status}")

# Mostrar resultados y status para timezone None
print("\nResultados con timezone None:")
display(out_none)

print("Status de normalización con timezone None:")
for col, status in status_none.items():
    print(f"{col}: {status}")

# Mostrar resultados y status para timezone inválido
print("\nResultados con timezone inválido:")
display(out_bad)

print("Status de normalización con timezone inválido:")
for col, status in status_bad.items():
    print(f"{col}: {status}")


Resultados con timezone IANA:


,dt_utc_explicit,dt_tzaware_offset,dt_naive,dt_str_iso,dt_numeric,dt_foursquare,dt_adatrap
0,2026-03-01 08:00:00+00:00,2026-03-01 11:00:00+00:00,2026-03-01 11:00:00+00:00,2026-03-01 11:00:00+00:00,NaT,2012-04-03 18:00:06+00:00,2019-05-19 20:10:32+00:00
1,2026-03-01 09:00:00+00:00,2026-03-01 12:00:00+00:00,2026-03-01 12:00:00+00:00,2026-03-01 12:15:00+00:00,NaT,2012-04-03 18:00:07+00:00,2019-05-19 15:33:19+00:00
2,NaT,NaT,NaT,NaT,NaT,NaT,NaT
3,2026-03-02 10:00:00+00:00,2026-03-02 13:00:00+00:00,2026-03-02 13:00:00+00:00,NaT,NaT,2014-01-29 16:44:24+00:00,2019-05-19 21:55:20+00:00
4,2026-03-03 11:00:00+00:00,2026-03-03 14:00:00+00:00,2026-03-03 14:00:00+00:00,2026-03-02 13:00:00+00:00,NaT,2014-01-29 16:44:25+00:00,2019-05-20 12:01:00+00:00
5,2026-03-03 12:00:00+00:00,2026-03-03 15:00:00+00:00,2026-03-03 15:00:00+00:00,2026-03-03 14:00:00+00:00,NaT,2014-02-13 09:10:11+00:00,NaT
6,2026-03-04 08:30:00+00:00,2026-03-04 11:30:00+00:00,2026-03-04 11:30:00+00:00,NaT,NaT,NaT,2019-05-21 16:00:00+00:00
7,2026-03-04 09:30:00+00:00,2026-03-04 12:30:00+00:00,2026-03-04 12:30:00+00:00,2026-03-04 15:00:00+00:00,NaT,2014-03-14 12:30:00+00:00,NaT
8,2026-03-05 10:30:00+00:00,2026-03-05 13:30:00+00:00,2026-03-05 13:30:00+00:00,2026-03-05 03:00:00+00:00,NaT,2014-04-12 00:00:00+00:00,2019-05-23 03:59:59+00:00
9,2026-03-06 00:00:00+00:00,2026-03-06 03:00:00+00:00,2026-03-06 03:00:00+00:00,2026-03-06 04:02:03+00:00,NaT,NaT,NaT


Status de normalización con timezone IANA:
dt_utc_explicit: {'status': 'utc', 'tz_kind': 'iana', 'n_nat': 1}
dt_tzaware_offset: {'status': 'tzaware_to_utc', 'tz_kind': 'iana', 'n_nat': 1}
dt_naive: {'status': 'naive_localized_to_utc', 'tz_kind': 'iana', 'n_nat': 1}
dt_str_iso: {'status': 'string_naive_localized_to_utc', 'tz_kind': 'iana', 'n_nat': 3}
dt_numeric: {'status': 'not_parsed_numeric', 'tz_kind': 'iana', 'n_nat': 10}
dt_foursquare: {'status': 'string_tzaware_to_utc', 'tz_kind': 'iana', 'n_nat': 3}
dt_adatrap: {'status': 'string_naive_localized_to_utc', 'tz_kind': 'iana', 'n_nat': 4}

Resultados con timezone offset fijo:


,dt_utc_explicit,dt_tzaware_offset,dt_naive,dt_str_iso,dt_numeric,dt_foursquare,dt_adatrap
0,2026-03-01 08:00:00+00:00,2026-03-01 11:00:00+00:00,2026-03-01 11:00:00+00:00,2026-03-01 11:00:00+00:00,NaT,2012-04-03 18:00:06+00:00,2019-05-19 19:10:32+00:00
1,2026-03-01 09:00:00+00:00,2026-03-01 12:00:00+00:00,2026-03-01 12:00:00+00:00,2026-03-01 12:15:00+00:00,NaT,2012-04-03 18:00:07+00:00,2019-05-19 14:33:19+00:00
2,NaT,NaT,NaT,NaT,NaT,NaT,NaT
3,2026-03-02 10:00:00+00:00,2026-03-02 13:00:00+00:00,2026-03-02 13:00:00+00:00,NaT,NaT,2014-01-29 16:44:24+00:00,2019-05-19 20:55:20+00:00
4,2026-03-03 11:00:00+00:00,2026-03-03 14:00:00+00:00,2026-03-03 14:00:00+00:00,2026-03-02 13:00:00+00:00,NaT,2014-01-29 16:44:25+00:00,2019-05-20 11:01:00+00:00
5,2026-03-03 12:00:00+00:00,2026-03-03 15:00:00+00:00,2026-03-03 15:00:00+00:00,2026-03-03 14:00:00+00:00,NaT,2014-02-13 09:10:11+00:00,NaT
6,2026-03-04 08:30:00+00:00,2026-03-04 11:30:00+00:00,2026-03-04 11:30:00+00:00,NaT,NaT,NaT,2019-05-21 15:00:00+00:00
7,2026-03-04 09:30:00+00:00,2026-03-04 12:30:00+00:00,2026-03-04 12:30:00+00:00,2026-03-04 15:00:00+00:00,NaT,2014-03-14 12:30:00+00:00,NaT
8,2026-03-05 10:30:00+00:00,2026-03-05 13:30:00+00:00,2026-03-05 13:30:00+00:00,2026-03-05 03:00:00+00:00,NaT,2014-04-12 00:00:00+00:00,2019-05-23 02:59:59+00:00
9,2026-03-06 00:00:00+00:00,2026-03-06 03:00:00+00:00,2026-03-06 03:00:00+00:00,2026-03-06 04:02:03+00:00,NaT,NaT,NaT


Status de normalización con timezone offset fijo:
dt_utc_explicit: {'status': 'utc', 'tz_kind': 'offset', 'n_nat': 1}
dt_tzaware_offset: {'status': 'tzaware_to_utc', 'tz_kind': 'offset', 'n_nat': 1}
dt_naive: {'status': 'naive_localized_to_utc', 'tz_kind': 'offset', 'n_nat': 1}
dt_str_iso: {'status': 'string_naive_localized_to_utc', 'tz_kind': 'offset', 'n_nat': 3}
dt_numeric: {'status': 'not_parsed_numeric', 'tz_kind': 'offset', 'n_nat': 10}
dt_foursquare: {'status': 'string_tzaware_to_utc', 'tz_kind': 'offset', 'n_nat': 3}
dt_adatrap: {'status': 'string_naive_localized_to_utc', 'tz_kind': 'offset', 'n_nat': 4}

Resultados con timezone None:


,dt_utc_explicit,dt_tzaware_offset,dt_naive,dt_str_iso,dt_numeric,dt_foursquare,dt_adatrap
0,2026-03-01 08:00:00+00:00,2026-03-01 11:00:00+00:00,2026-03-01 08:00:00,2026-03-01 08:00:00,NaT,2012-04-03 18:00:06+00:00,2019-05-19 16:10:32
1,2026-03-01 09:00:00+00:00,2026-03-01 12:00:00+00:00,2026-03-01 09:00:00,2026-03-01 09:15:00,NaT,2012-04-03 18:00:07+00:00,2019-05-19 11:33:19
2,NaT,NaT,NaT,NaT,NaT,NaT,NaT
3,2026-03-02 10:00:00+00:00,2026-03-02 13:00:00+00:00,2026-03-02 10:00:00,NaT,NaT,2014-01-29 16:44:24+00:00,2019-05-19 17:55:20
4,2026-03-03 11:00:00+00:00,2026-03-03 14:00:00+00:00,2026-03-03 11:00:00,2026-03-02 10:00:00,NaT,2014-01-29 16:44:25+00:00,2019-05-20 08:01:00
5,2026-03-03 12:00:00+00:00,2026-03-03 15:00:00+00:00,2026-03-03 12:00:00,2026-03-03 11:00:00,NaT,2014-02-13 09:10:11+00:00,NaT
6,2026-03-04 08:30:00+00:00,2026-03-04 11:30:00+00:00,2026-03-04 08:30:00,NaT,NaT,NaT,2019-05-21 12:00:00
7,2026-03-04 09:30:00+00:00,2026-03-04 12:30:00+00:00,2026-03-04 09:30:00,2026-03-04 12:00:00,NaT,2014-03-14 12:30:00+00:00,NaT
8,2026-03-05 10:30:00+00:00,2026-03-05 13:30:00+00:00,2026-03-05 10:30:00,2026-03-05 00:00:00,NaT,2014-04-12 00:00:00+00:00,2019-05-22 23:59:59
9,2026-03-06 00:00:00+00:00,2026-03-06 03:00:00+00:00,2026-03-06 00:00:00,2026-03-06 01:02:03,NaT,NaT,NaT


Status de normalización con timezone None:
dt_utc_explicit: {'status': 'utc', 'tz_kind': 'none', 'n_nat': 1}
dt_tzaware_offset: {'status': 'tzaware_to_utc', 'tz_kind': 'none', 'n_nat': 1}
dt_naive: {'status': 'naive_unconverted', 'tz_kind': 'none', 'n_nat': 1}
dt_str_iso: {'status': 'string_naive_unconverted', 'tz_kind': 'none', 'n_nat': 3}
dt_numeric: {'status': 'not_parsed_numeric', 'tz_kind': 'none', 'n_nat': 10}
dt_foursquare: {'status': 'string_tzaware_to_utc', 'tz_kind': 'none', 'n_nat': 3}
dt_adatrap: {'status': 'string_naive_unconverted', 'tz_kind': 'none', 'n_nat': 4}

Resultados con timezone inválido:


,dt_utc_explicit,dt_tzaware_offset,dt_naive,dt_str_iso,dt_numeric,dt_foursquare,dt_adatrap
0,2026-03-01 08:00:00+00:00,2026-03-01 11:00:00+00:00,2026-03-01 08:00:00,2026-03-01 08:00:00,NaT,2012-04-03 18:00:06+00:00,2019-05-19 16:10:32
1,2026-03-01 09:00:00+00:00,2026-03-01 12:00:00+00:00,2026-03-01 09:00:00,2026-03-01 09:15:00,NaT,2012-04-03 18:00:07+00:00,2019-05-19 11:33:19
2,NaT,NaT,NaT,NaT,NaT,NaT,NaT
3,2026-03-02 10:00:00+00:00,2026-03-02 13:00:00+00:00,2026-03-02 10:00:00,NaT,NaT,2014-01-29 16:44:24+00:00,2019-05-19 17:55:20
4,2026-03-03 11:00:00+00:00,2026-03-03 14:00:00+00:00,2026-03-03 11:00:00,2026-03-02 10:00:00,NaT,2014-01-29 16:44:25+00:00,2019-05-20 08:01:00
5,2026-03-03 12:00:00+00:00,2026-03-03 15:00:00+00:00,2026-03-03 12:00:00,2026-03-03 11:00:00,NaT,2014-02-13 09:10:11+00:00,NaT
6,2026-03-04 08:30:00+00:00,2026-03-04 11:30:00+00:00,2026-03-04 08:30:00,NaT,NaT,NaT,2019-05-21 12:00:00
7,2026-03-04 09:30:00+00:00,2026-03-04 12:30:00+00:00,2026-03-04 09:30:00,2026-03-04 12:00:00,NaT,2014-03-14 12:30:00+00:00,NaT
8,2026-03-05 10:30:00+00:00,2026-03-05 13:30:00+00:00,2026-03-05 10:30:00,2026-03-05 00:00:00,NaT,2014-04-12 00:00:00+00:00,2019-05-22 23:59:59
9,2026-03-06 00:00:00+00:00,2026-03-06 03:00:00+00:00,2026-03-06 00:00:00,2026-03-06 01:02:03,NaT,NaT,NaT


Status de normalización con timezone inválido:
dt_utc_explicit: {'status': 'utc', 'tz_kind': 'invalid', 'n_nat': 1}
dt_tzaware_offset: {'status': 'tzaware_to_utc', 'tz_kind': 'invalid', 'n_nat': 1}
dt_naive: {'status': 'naive_unconverted', 'tz_kind': 'invalid', 'n_nat': 1}
dt_str_iso: {'status': 'string_naive_unconverted', 'tz_kind': 'invalid', 'n_nat': 3}
dt_numeric: {'status': 'not_parsed_numeric', 'tz_kind': 'invalid', 'n_nat': 10}
dt_foursquare: {'status': 'string_tzaware_to_utc', 'tz_kind': 'invalid', 'n_nat': 3}
dt_adatrap: {'status': 'string_naive_unconverted', 'tz_kind': 'invalid', 'n_nat': 4}


#### Tier 2: columnas HH:MM

In [79]:
from pylondrina.schema import TripSchema, FieldSpec

schema_t2 = TripSchema(fields={
    "origin_time_local_hhmm": FieldSpec(name="origin_time_local_hhmm", dtype="string"),
    "destination_time_local_hhmm": FieldSpec(name="destination_time_local_hhmm", dtype="string"),
})

work_t2 = pd.DataFrame({
    "origin_time_local_hhmm": ["08:23", "12:30", "24:00", "ab:cd", "", None, "7:05", "09:60"],
    "destination_time_local_hhmm": ["09:10", "13:00", "23:59", "99:99", "  ", np.nan, "07:05", "00:00"],
})

print("DataFrame original con campos de hora local en formato HH:MM (con valores sucios):")
display(work_t2)

import re
HHMM_RE = re.compile(r"^(?P<h>\d{2}):(?P<m>\d{2})$")

def normalize_hhmm_series(s: pd.Series) -> tuple[pd.Series, dict]:
    s = s.astype("string").str.strip().replace("", pd.NA)
    ok = []
    out = []
    for v in s.tolist():
        if pd.isna(v):
            out.append(pd.NA); ok.append(True); continue
        m = HHMM_RE.match(v)
        if not m:
            out.append(pd.NA); ok.append(False); continue
        h = int(m.group("h")); mm = int(m.group("m"))
        if not (0 <= h <= 23 and 0 <= mm <= 59):
            out.append(pd.NA); ok.append(False); continue
        out.append(v); ok.append(True)
    out_s = pd.Series(out, dtype="string")
    stats = {"n_total": len(s), "n_invalid": ok.count(False), "n_na": int(out_s.isna().sum())}
    return out_s, stats

work_t2_norm = work_t2.copy(deep=True)
work_t2_norm["origin_time_local_hhmm"], st_o = normalize_hhmm_series(work_t2_norm["origin_time_local_hhmm"])
work_t2_norm["destination_time_local_hhmm"], st_d = normalize_hhmm_series(work_t2_norm["destination_time_local_hhmm"])

# print de stats de limpieza para ambas columnas
print("Stats de limpieza para origin_time_local_hhmm:")
print(st_o)
print("Stats de limpieza para destination_time_local_hhmm:")
print(st_d)

print("\nDataFrame después de normalizar campos HH:MM (valores no válidos convertidos a NA):")
display(work_t2_norm)

DataFrame original con campos de hora local en formato HH:MM (con valores sucios):


,origin_time_local_hhmm,destination_time_local_hhmm
0,08:23,09:10
1,12:30,13:00
2,24:00,23:59
3,ab:cd,99:99
4,,
5,None,NaN
6,7:05,07:05
7,09:60,00:00


Stats de limpieza para origin_time_local_hhmm:
{'n_total': 8, 'n_invalid': 4, 'n_na': 6}
Stats de limpieza para destination_time_local_hhmm:
{'n_total': 8, 'n_invalid': 1, 'n_na': 3}

DataFrame después de normalizar campos HH:MM (valores no válidos convertidos a NA):


,origin_time_local_hhmm,destination_time_local_hhmm
0,08:23,09:10
1,12:30,13:00
2,<NA>,23:59
3,<NA>,<NA>
4,<NA>,<NA>
5,<NA>,<NA>
6,<NA>,07:05
7,<NA>,00:00


### 22. Parseo de coordenadas OD (DD / DM / DMS -> DD)

In [86]:
import pandas as pd
import numpy as np
from pandas.api.types import is_numeric_dtype


df_coords_lab = pd.DataFrame({
    # 1) Numérico int
    "origin_latitude_int": [-33, -34, None, -35],
    "origin_longitude_int": [-70, -71, -72, None],

    # 2) Numérico float
    "origin_latitude_float": [-33.45, -33.39, np.nan, -33.50],
    "origin_longitude_float": [-70.57, -70.51, -70.60, np.nan],

    # 3) String DD directo
    "origin_latitude_str_dd": ["-33.446160", "-33.392693", None, " -33.500100 "],
    "origin_longitude_str_dd": ["-70.572755", "-70.517930", "", "-70.600200"],

    # 4) String DM
    # formato ejemplo: 33°26.7696'S
    "origin_latitude_str_dm": ["33°26.7696'S", "33°23.5616'S", None, ""],
    "origin_longitude_str_dm": ["70°34.3653'W", "70°31.0758'W", None, ""],

    # 5) String DMS
    # formato ejemplo: 33°26'46.176\"S
    "origin_latitude_str_dms": ['33°26\'46.176"S', '33°23\'33.696"S', None, ""],
    "origin_longitude_str_dms": ['70°34\'21.918"W', '70°31\'04.548"W', None, ""],

    # 6) Casos inspirados en fuentes reales
    # EOD / ADATRAP como casos que NO deberían parsearse como lat/lon geográfica
    "eod_x": ["335208,7188", "338536,4375", "354267,3438", None],
    "eod_y": ["6288387", "6291928", None, "6289000"],
    "adatrap_x": ["335889", "335085", None, "336000"],
    "adatrap_y": ["6292782", "6302493", "6295000", None],

    # 7) Foursquare DD realista
    "foursquare_lat": ["-33.446160", "-33.392693", None, "-33.500000"],
    "foursquare_lon": ["-70.572755", "-70.517930", "-70.600000", None],
})
print("DataFrame original con coordenadas en varios formatos (y casos reales de EOD/ADATRAP):")
display(df_coords_lab)

cols = [
    "origin_latitude_int",
    "origin_latitude_float",
    "origin_latitude_str_dd",
    "origin_latitude_str_dm",
    "origin_latitude_str_dms",
    "eod_x",
    "adatrap_x",
    "foursquare_lat",
]

print("Inspección de columnas de coordenadas (tipos y valores):")

for col in cols:
    print(f"\n--- {col} ---")
    print("dtype:", df_coords_lab[col].dtype)
#    print(df_coords_lab[col])



DataFrame original con coordenadas en varios formatos (y casos reales de EOD/ADATRAP):


,origin_latitude_int,origin_longitude_int,origin_latitude_float,origin_longitude_float,origin_latitude_str_dd,origin_longitude_str_dd,origin_latitude_str_dm,origin_longitude_str_dm,origin_latitude_str_dms,origin_longitude_str_dms,eod_x,eod_y,adatrap_x,adatrap_y,foursquare_lat,foursquare_lon
0,-33.0,-70.0,-33.45,-70.57,-33.446160,-70.572755,33°26.7696'S,70°34.3653'W,"33°26'46.176""S","70°34'21.918""W","335208,7188",6288387,335889,6292782,-33.446160,-70.572755
1,-34.0,-71.0,-33.39,-70.51,-33.392693,-70.517930,33°23.5616'S,70°31.0758'W,"33°23'33.696""S","70°31'04.548""W","338536,4375",6291928,335085,6302493,-33.392693,-70.517930
2,NaN,-72.0,NaN,-70.60,None,,None,None,None,None,"354267,3438",None,None,6295000,None,-70.600000
3,-35.0,NaN,-33.50,NaN,-33.500100,-70.600200,,,,,None,6289000,336000,None,-33.500000,None


Inspección de columnas de coordenadas (tipos y valores):

--- origin_latitude_int ---
dtype: float64

--- origin_latitude_float ---
dtype: float64

--- origin_latitude_str_dd ---
dtype: object

--- origin_latitude_str_dm ---
dtype: object

--- origin_latitude_str_dms ---
dtype: object

--- eod_x ---
dtype: object

--- adatrap_x ---
dtype: object

--- foursquare_lat ---
dtype: object


In [87]:
print("\n¿Es cada columna de tipo numérico?")
for col in cols:
    print(col, "-> is_numeric_dtype =", is_numeric_dtype(df_coords_lab[col]))


¿Es cada columna de tipo numérico?
origin_latitude_int -> is_numeric_dtype = True
origin_latitude_float -> is_numeric_dtype = True
origin_latitude_str_dd -> is_numeric_dtype = False
origin_latitude_str_dm -> is_numeric_dtype = False
origin_latitude_str_dms -> is_numeric_dtype = False
eod_x -> is_numeric_dtype = False
adatrap_x -> is_numeric_dtype = False
foursquare_lat -> is_numeric_dtype = False


In [89]:
import re

DM_PATTERN = re.compile(
    r"""^\s*
    (?P<deg>\d{1,3})
    [°\s]+
    (?P<minutes>\d{1,2}(?:\.\d+)?)
    \s*'
    ?\s*
    (?P<hem>[NSEW])
    \s*$""",
    re.VERBOSE | re.IGNORECASE
)

DMS_PATTERN = re.compile(
    r"""^\s*
    (?P<deg>\d{1,3})
    [°\s]+
    (?P<minutes>\d{1,2})
    '\s*
    (?P<seconds>\d{1,2}(?:\.\d+)?)
    "\s*
    (?P<hem>[NSEW])
    \s*$""",
    re.VERBOSE | re.IGNORECASE
)

def dm_to_dd(deg, minutes, hem):
    dd = float(deg) + float(minutes) / 60.0
    if hem.upper() in {"S", "W"}:
        dd = -dd
    return dd

def dms_to_dd(deg, minutes, seconds, hem):
    dd = float(deg) + float(minutes) / 60.0 + float(seconds) / 3600.0
    if hem.upper() in {"S", "W"}:
        dd = -dd
    return dd

def parse_coord_value(v):
    if pd.isna(v):
        return np.nan, "null"

    # si ya viene numérico
    if isinstance(v, (int, float, np.integer, np.floating)):
        return float(v), "numeric"

    s = str(v).strip()
    if s == "":
        return np.nan, "empty"

    # intento DD directo
    try:
        return float(s), "dd_direct"
    except ValueError:
        pass

    # intento DD con coma decimal
    try:
        return float(s.replace(",", ".")), "dd_comma_decimal"
    except ValueError:
        pass

    # intento DM
    m = DM_PATTERN.match(s)
    if m:
        return dm_to_dd(m.group("deg"), m.group("minutes"), m.group("hem")), "dm"

    # intento DMS
    m = DMS_PATTERN.match(s)
    if m:
        return dms_to_dd(
            m.group("deg"),
            m.group("minutes"),
            m.group("seconds"),
            m.group("hem")
        ), "dms"

    return np.nan, "unparsed"

def probe_parse_column(df, col):
    out = df[col].apply(parse_coord_value)
    result = pd.DataFrame({
        "original": df[col],
        "parsed": out.apply(lambda x: x[0]),
        "status": out.apply(lambda x: x[1]),
    })
    return result

display(probe_parse_column(df_coords_lab, "origin_latitude_str_dd"))
display(probe_parse_column(df_coords_lab, "origin_latitude_str_dm"))
display(probe_parse_column(df_coords_lab, "origin_latitude_str_dms"))
display(probe_parse_column(df_coords_lab, "eod_x"))
display(probe_parse_column(df_coords_lab, "foursquare_lat"))

,original,parsed,status
0,-33.446160,-33.446160,dd_direct
1,-33.392693,-33.392693,dd_direct
2,None,NaN,null
3,-33.500100,-33.500100,dd_direct


,original,parsed,status
0,33°26.7696'S,-33.446160,dm
1,33°23.5616'S,-33.392693,dm
2,None,NaN,null
3,,NaN,empty


,original,parsed,status
0,"33°26'46.176""S",-33.446160,dms
1,"33°23'33.696""S",-33.392693,dms
2,None,NaN,null
3,,NaN,empty


,original,parsed,status
0,"335208,7188",335208.7188,dd_comma_decimal
1,"338536,4375",338536.4375,dd_comma_decimal
2,"354267,3438",354267.3438,dd_comma_decimal
3,None,NaN,null


,original,parsed,status
0,-33.446160,-33.446160,dd_direct
1,-33.392693,-33.392693,dd_direct
2,None,NaN,null
3,-33.500000,-33.500000,dd_direct


In [91]:
df_s9_dd = pd.DataFrame({
    "origin_latitude": ["-33.446160", "-33.392693", None],
    "origin_longitude": ["-70.572755", "-70.517930", ""],
    "destination_latitude": [-33.45, -33.40, np.nan],
    "destination_longitude": [-70.58, -70.52, np.nan],
})

df_s9_dm = pd.DataFrame({
    "origin_latitude": ["33°26.7696'S", "33°23.5616'S", None],
    "origin_longitude": ["70°34.3653'W", "70°31.0758'W", None],
    "destination_latitude": ["33°27.0000'S", "", None],
    "destination_longitude": ["70°35.0000'W", None, ""],
})

df_s9_dms = pd.DataFrame({
    "origin_latitude": ['33°26\'46.176"S', '33°23\'33.696"S', None],
    "origin_longitude": ['70°34\'21.918"W', '70°31\'04.548"W', None],
    "destination_latitude": ['33°27\'00.000"S', "", None],
    "destination_longitude": ['70°35\'00.000"W', None, ""],
})

def parse_coord_series(series, kind):
    parsed = series.apply(parse_coord_value)
    values = parsed.apply(lambda x: x[0]).astype(float)
    status = parsed.apply(lambda x: x[1]).astype("string")

    return values, status

In [93]:
for col, kind in [
    ("origin_latitude", "lat"),
    ("origin_longitude", "lon"),
    ("destination_latitude", "lat"),
    ("destination_longitude", "lon"),
]:
    vals, status = parse_coord_series(df_s9_dd[col], kind)
    print(f"\n--- {col} ---")
    print(pd.DataFrame({
        "original": df_s9_dd[col],
        "parsed": vals,
        "status": status,
    }))


--- origin_latitude ---
     original     parsed     status
0  -33.446160 -33.446160  dd_direct
1  -33.392693 -33.392693  dd_direct
2        None        NaN       null

--- origin_longitude ---
     original     parsed     status
0  -70.572755 -70.572755  dd_direct
1  -70.517930 -70.517930  dd_direct
2                    NaN      empty

--- destination_latitude ---
   original  parsed   status
0    -33.45  -33.45  numeric
1    -33.40  -33.40  numeric
2       NaN     NaN     null

--- destination_longitude ---
   original  parsed   status
0    -70.58  -70.58  numeric
1    -70.52  -70.52  numeric
2       NaN     NaN     null


In [95]:
for col, kind in [
    ("origin_latitude", "lat"),
    ("origin_longitude", "lon"),
    ("destination_latitude", "lat"),
    ("destination_longitude", "lon"),
]:
    vals, status = parse_coord_series(df_s9_dm[col], kind)
    print(f"\n--- {col} ---")
    print(pd.DataFrame({
        "original": df_s9_dm[col],
        "parsed": vals,
        "status": status,
    }))


--- origin_latitude ---
       original     parsed status
0  33°26.7696'S -33.446160     dm
1  33°23.5616'S -33.392693     dm
2          None        NaN   null

--- origin_longitude ---
       original     parsed status
0  70°34.3653'W -70.572755     dm
1  70°31.0758'W -70.517930     dm
2          None        NaN   null

--- destination_latitude ---
       original  parsed status
0  33°27.0000'S  -33.45     dm
1                   NaN  empty
2          None     NaN   null

--- destination_longitude ---
       original     parsed status
0  70°35.0000'W -70.583333     dm
1          None        NaN   null
2                      NaN  empty


In [96]:
for col, kind in [
    ("origin_latitude", "lat"),
    ("origin_longitude", "lon"),
    ("destination_latitude", "lat"),
    ("destination_longitude", "lon"),
]:
    vals, status = parse_coord_series(df_s9_dms[col], kind)
    print(f"\n--- {col} ---")
    print(pd.DataFrame({
        "original": df_s9_dms[col],
        "parsed": vals,
        "status": status,
    }))


--- origin_latitude ---
         original     parsed status
0  33°26'46.176"S -33.446160    dms
1  33°23'33.696"S -33.392693    dms
2            None        NaN   null

--- origin_longitude ---
         original     parsed status
0  70°34'21.918"W -70.572755    dms
1  70°31'04.548"W -70.517930    dms
2            None        NaN   null

--- destination_latitude ---
         original  parsed status
0  33°27'00.000"S  -33.45    dms
1                     NaN  empty
2            None     NaN   null

--- destination_longitude ---
         original     parsed status
0  70°35'00.000"W -70.583333    dms
1            None        NaN   null
2                        NaN  empty


### Preprocess - Prueba 1 helper para par coordenada x/y

In [97]:
import pandas as pd
import numpy as np
from pyproj import Transformer

def _normalize_projected_coord_series(
    s: pd.Series,
    *,
    decimal_comma: bool = True,
    zero_as_missing: bool = False,
) -> tuple[pd.Series, pd.Series]:
    """
    Normaliza una serie con coordenadas proyectadas a float.
    
    Returns
    -------
    values : pd.Series float
        Serie numérica parseada.
    status : pd.Series string
        Estado por fila: ok_numeric, ok_string, empty, zero_as_missing, non_numeric, null
    """
    status = pd.Series(index=s.index, dtype="string")
    
    # Caso numérico
    if pd.api.types.is_numeric_dtype(s):
        values = pd.to_numeric(s, errors="coerce").astype(float)
        status[:] = "ok_numeric"
        status[values.isna()] = "null"
        if zero_as_missing:
            zero_mask = values == 0
            values = values.mask(zero_mask)
            status[zero_mask] = "zero_as_missing"
        return values, status

    # Caso string / object
    s2 = s.astype("string").str.strip()
    status[:] = "ok_string"
    
    null_mask = s2.isna()
    empty_mask = s2.eq("")
    
    if decimal_comma:
        s2 = s2.str.replace(",", ".", regex=False)
    
    values = pd.to_numeric(s2, errors="coerce").astype(float)
    
    status[null_mask] = "null"
    status[empty_mask] = "empty"
    status[values.isna() & ~null_mask & ~empty_mask] = "non_numeric"
    
    if zero_as_missing:
        zero_mask = values == 0
        values = values.mask(zero_mask)
        status[zero_mask] = "zero_as_missing"
    
    return values, status


def project_xy_to_latlon(
    df: pd.DataFrame,
    *,
    x_col: str,
    y_col: str,
    source_crs: str,
    lon_col: str,
    lat_col: str,
    decimal_comma: bool = True,
    zero_as_missing: bool = False,
    keep_debug_cols: bool = True,
) -> pd.DataFrame:
    """
    Transforma columnas X/Y proyectadas a lon/lat en EPSG:4326.
    """
    work = df.copy()

    x_vals, x_status = _normalize_projected_coord_series(
        work[x_col],
        decimal_comma=decimal_comma,
        zero_as_missing=zero_as_missing,
    )
    y_vals, y_status = _normalize_projected_coord_series(
        work[y_col],
        decimal_comma=decimal_comma,
        zero_as_missing=zero_as_missing,
    )

    valid_mask = x_vals.notna() & y_vals.notna()

    work[lon_col] = np.nan
    work[lat_col] = np.nan

    transformer = Transformer.from_crs(source_crs, "EPSG:4326", always_xy=True)

    if valid_mask.any():
        lon, lat = transformer.transform(
            x_vals[valid_mask].to_numpy(),
            y_vals[valid_mask].to_numpy(),
        )
        work.loc[valid_mask, lon_col] = lon
        work.loc[valid_mask, lat_col] = lat

    if keep_debug_cols:
        work[f"__{x_col}_parsed"] = x_vals
        work[f"__{y_col}_parsed"] = y_vals
        work[f"__{x_col}_status"] = x_status
        work[f"__{y_col}_status"] = y_status
        work[f"__{lon_col}_latlon_status"] = pd.Series(
            np.where(valid_mask, "transformed", "not_transformed"),
            index=work.index,
            dtype="string",
        )

    return work


def add_od_latlon_from_projected_xy(
    df: pd.DataFrame,
    *,
    origin_x_col: str,
    origin_y_col: str,
    destination_x_col: str,
    destination_y_col: str,
    source_crs: str,
    decimal_comma: bool = True,
    zero_as_missing: bool = False,
    keep_debug_cols: bool = True,
) -> pd.DataFrame:
    work = project_xy_to_latlon(
        df,
        x_col=origin_x_col,
        y_col=origin_y_col,
        source_crs=source_crs,
        lon_col="origin_longitude",
        lat_col="origin_latitude",
        decimal_comma=decimal_comma,
        zero_as_missing=zero_as_missing,
        keep_debug_cols=keep_debug_cols,
    )

    work = project_xy_to_latlon(
        work,
        x_col=destination_x_col,
        y_col=destination_y_col,
        source_crs=source_crs,
        lon_col="destination_longitude",
        lat_col="destination_latitude",
        decimal_comma=decimal_comma,
        zero_as_missing=zero_as_missing,
        keep_debug_cols=keep_debug_cols,
    )

    return work

In [98]:
import pandas as pd
import numpy as np

df_eod_like = pd.DataFrame({
    "Persona": [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
    "HoraIni": ["08:10", "09:05", "07:45", "18:20", "12:30", "06:50", "14:15", "21:00", "10:25", "16:40", "11:10", "19:30"],
    "HoraFin": ["08:40", "09:35", "08:10", "18:55", "13:00", "07:10", "14:45", "21:20", "10:55", "17:10", "11:40", "20:00"],

    # origen
    "OrigenCoordX": [
        "353816,4273",   # bueno
        "358826,2253",   # bueno
        "346642,6950",   # bueno
        "0",             # cero -> missing
        None,            # null
        " 352169,0237 ", # bueno con espacios
        "abc",           # inválido
        "342049,4163",   # bueno
        "356736,2080",   # bueno
        "353816.4273",   # bueno con punto
        "",              # vacío
        "350000,5000",   # bueno
    ],
    "OrigenCoordY": [
        "6298143,9671",
        "6304148,3259",
        "6297606,8326",
        "6298143,9671",
        "6294205,2865",
        "6301020,3200",
        "6298143,9671",
        "6294205,2865",
        "6306634,4332",
        "6298143.9671",
        "",
        "6300000,2500",
    ],

    # destino
    "DestinoCoordX": [
        "358826,2253",
        "346642,6950",
        "352169,0237",
        "356736,2080",
        "342049,4163",
        "0",
        "353816,4273",
        "bad",
        None,
        "342049,4163",
        "356736,2080",
        "358826,2253",
    ],
    "DestinoCoordY": [
        "6304148,3259",
        "6297606,8326",
        "6301020,3200",
        "6306634,4332",
        "6294205,2865",
        "6298143,9671",
        "6298143,9671",
        "6304148,3259",
        "6297606,8326",
        "",
        None,
        "6304148,3259",
    ],
})

print("DataFrame de ejemplo con coordenadas proyectadas en formato EOD-like (con casos sucios):")
display(df_eod_like)

DataFrame de ejemplo con coordenadas proyectadas en formato EOD-like (con casos sucios):


,Persona,HoraIni,HoraFin,OrigenCoordX,OrigenCoordY,DestinoCoordX,DestinoCoordY
0,101,08:10,08:40,"353816,4273","6298143,9671","358826,2253","6304148,3259"
1,102,09:05,09:35,"358826,2253","6304148,3259","346642,6950","6297606,8326"
2,103,07:45,08:10,"346642,6950","6297606,8326","352169,0237","6301020,3200"
3,104,18:20,18:55,0,"6298143,9671","356736,2080","6306634,4332"
4,105,12:30,13:00,None,"6294205,2865","342049,4163","6294205,2865"
5,106,06:50,07:10,"352169,0237","6301020,3200",0,"6298143,9671"
6,107,14:15,14:45,abc,"6298143,9671","353816,4273","6298143,9671"
7,108,21:00,21:20,"342049,4163","6294205,2865",bad,"6304148,3259"
8,109,10:25,10:55,"356736,2080","6306634,4332",None,"6297606,8326"
9,110,16:40,17:10,353816.4273,6298143.9671,"342049,4163",


In [100]:
df_eod_proj = add_od_latlon_from_projected_xy(
    df_eod_like,
    origin_x_col="OrigenCoordX",
    origin_y_col="OrigenCoordY",
    destination_x_col="DestinoCoordX",
    destination_y_col="DestinoCoordY",
    source_crs="EPSG:5361",
    decimal_comma=True,
    zero_as_missing=True,
    keep_debug_cols=True,
)

cols_to_show = [
    "Persona",
    "OrigenCoordX", "OrigenCoordY",
    "origin_longitude", "origin_latitude",
    "DestinoCoordX", "DestinoCoordY",
    "destination_longitude", "destination_latitude",
    "__OrigenCoordX_status", "__OrigenCoordY_status",
    "__DestinoCoordX_status", "__DestinoCoordY_status",
    "__origin_longitude_latlon_status",
    "__destination_longitude_latlon_status",
]

display(df_eod_proj[cols_to_show])

,Persona,OrigenCoordX,OrigenCoordY,origin_longitude,origin_latitude,DestinoCoordX,DestinoCoordY,destination_longitude,destination_latitude,__OrigenCoordX_status,__OrigenCoordY_status,__DestinoCoordX_status,__DestinoCoordY_status,__origin_longitude_latlon_status,__destination_longitude_latlon_status
0,101,"353816,4273","6298143,9671",-70.572755,-33.446160,"358826,2253","6304148,3259",-70.517930,-33.392693,ok_string,ok_string,ok_string,ok_string,transformed,transformed
1,102,"358826,2253","6304148,3259",-70.517930,-33.392693,"346642,6950","6297606,8326",-70.650000,-33.450000,ok_string,ok_string,ok_string,ok_string,transformed,transformed
2,103,"346642,6950","6297606,8326",-70.650000,-33.450000,"352169,0237","6301020,3200",-70.590000,-33.420000,ok_string,ok_string,ok_string,ok_string,transformed,transformed
3,104,0,"6298143,9671",NaN,NaN,"356736,2080","6306634,4332",-70.540000,-33.370000,zero_as_missing,ok_string,ok_string,ok_string,not_transformed,transformed
4,105,None,"6294205,2865",NaN,NaN,"342049,4163","6294205,2865",-70.700000,-33.480000,null,ok_string,ok_string,ok_string,not_transformed,transformed
5,106,"352169,0237","6301020,3200",-70.590000,-33.420000,0,"6298143,9671",NaN,NaN,ok_string,ok_string,zero_as_missing,ok_string,transformed,not_transformed
6,107,abc,"6298143,9671",NaN,NaN,"353816,4273","6298143,9671",-70.572755,-33.446160,non_numeric,ok_string,ok_string,ok_string,not_transformed,transformed
7,108,"342049,4163","6294205,2865",-70.700000,-33.480000,bad,"6304148,3259",NaN,NaN,ok_string,ok_string,non_numeric,ok_string,transformed,not_transformed
8,109,"356736,2080","6306634,4332",-70.540000,-33.370000,None,"6297606,8326",NaN,NaN,ok_string,ok_string,null,ok_string,transformed,not_transformed
9,110,353816.4273,6298143.9671,-70.572755,-33.446160,"342049,4163",,NaN,NaN,ok_string,ok_string,ok_string,empty,transformed,not_transformed


In [103]:
df_stations_like = pd.DataFrame({
    "parada/est.metro": [
        "PA001", "PA002", "PA003", "PA004", "PA005", "PA006", "PA007", "PA008"
    ],
    "x": [
        353816, 358826, 346643, 352169, 342049, 356736, 351200, 349900
    ],
    "y": [
        6298144, 6304148, 6297607, 6301020, 6294205, 6306634, 6299500, 6302500
    ],
})

df_adatrap_like = pd.DataFrame({
    "id": [2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010],

    "paraderosubida_1era":  ["PA001", "PA002", "PA003", "-",    "PA005", "PA001", "PA006", "PA003", "PA999", "PA004"],
    "paraderobajada_1era":  ["PA002", "PA003", "PA004", "PA001", "-",     "PA005", "PA001", "PA006", "PA002", "PA008"],
    "tiemposubida_1era":    ["07:10", "08:00", "09:15", "10:05", "11:20", "12:00", "13:40", "14:10", "15:00", "16:30"],
    "tiempobajada_1era":    ["07:40", "08:25", "09:45", "10:30", "11:50", "12:35", "14:05", "14:45", "15:30", "17:00"],

    "paraderosubida_2da":   ["-",     "PA003", "-",     "PA002", "PA001", "-",     "PA004", "-",     "-",     "PA005"],
    "paraderobajada_2da":   ["-",     "PA004", "-",     "PA005", "PA006", "-",     "PA003", "-",     "-",     "PA001"],
    "tiemposubida_2da":     [None,    "08:35", None,    "10:40", "12:40", None,    "14:15", None,    None,    "17:10"],
    "tiempobajada_2da":     [None,    "08:55", None,    "11:05", "13:00", None,    "14:35", None,    None,    "17:30"],
})

print("DataFrame de ejemplo con coordenadas proyectadas de estaciones (limpio):")
display(df_stations_like)

print("\nDataFrame de ejemplo con datos de ADATRAP-like (con casos sucios):")
display(df_adatrap_like)

DataFrame de ejemplo con coordenadas proyectadas de estaciones (limpio):


,parada/est.metro,x,y
0,PA001,353816,6298144
1,PA002,358826,6304148
2,PA003,346643,6297607
3,PA004,352169,6301020
4,PA005,342049,6294205
5,PA006,356736,6306634
6,PA007,351200,6299500
7,PA008,349900,6302500



DataFrame de ejemplo con datos de ADATRAP-like (con casos sucios):


,id,paraderosubida_1era,paraderobajada_1era,tiemposubida_1era,tiempobajada_1era,paraderosubida_2da,paraderobajada_2da,tiemposubida_2da,tiempobajada_2da
0,2001,PA001,PA002,07:10,07:40,-,-,None,None
1,2002,PA002,PA003,08:00,08:25,PA003,PA004,08:35,08:55
2,2003,PA003,PA004,09:15,09:45,-,-,None,None
3,2004,-,PA001,10:05,10:30,PA002,PA005,10:40,11:05
4,2005,PA005,-,11:20,11:50,PA001,PA006,12:40,13:00
5,2006,PA001,PA005,12:00,12:35,-,-,None,None
6,2007,PA006,PA001,13:40,14:05,PA004,PA003,14:15,14:35
7,2008,PA003,PA006,14:10,14:45,-,-,None,None
8,2009,PA999,PA002,15:00,15:30,-,-,None,None
9,2010,PA004,PA008,16:30,17:00,PA005,PA001,17:10,17:30


In [105]:
stations_table = df_stations_like.rename(columns={"parada/est.metro": "parada"})

df_adatrap_stage1 = df_adatrap_like[
    ["id", "paraderosubida_1era", "paraderobajada_1era", "tiemposubida_1era", "tiempobajada_1era"]
].rename(columns={
    "id": "user_id",
    "paraderosubida_1era": "subida",
    "paraderobajada_1era": "bajada",
    "tiemposubida_1era": "o_time",
    "tiempobajada_1era": "d_time",
})

df_adatrap_stage1 = df_adatrap_stage1[
    (df_adatrap_stage1["subida"] != "-") &
    (df_adatrap_stage1["bajada"] != "-")
].copy()

# merge origen
df_adatrap_m1 = df_adatrap_stage1.merge(
    stations_table,
    how="left",
    left_on="subida",
    right_on="parada",
).rename(columns={"x": "o_x", "y": "o_y"}).drop(columns=["parada"])

# merge destino
df_adatrap_m2 = df_adatrap_m1.merge(
    stations_table,
    how="left",
    left_on="bajada",
    right_on="parada",
).rename(columns={"x": "d_x", "y": "d_y"}).drop(columns=["parada"])


df_adatrap_proj = add_od_latlon_from_projected_xy(
    df_adatrap_m2,
    origin_x_col="o_x",
    origin_y_col="o_y",
    destination_x_col="d_x",
    destination_y_col="d_y",
    source_crs="EPSG:5361",
    decimal_comma=False,
    zero_as_missing=False,
    keep_debug_cols=True,
)

df_adatrap_proj[
    [
        "user_id", "subida", "bajada",
        "o_x", "o_y", "origin_longitude", "origin_latitude",
        "d_x", "d_y", "destination_longitude", "destination_latitude",
        "__origin_longitude_latlon_status",
        "__destination_longitude_latlon_status",
    ]
]

,user_id,subida,bajada,o_x,o_y,origin_longitude,origin_latitude,d_x,d_y,destination_longitude,destination_latitude,__origin_longitude_latlon_status,__destination_longitude_latlon_status
0,2001,PA001,PA002,353816.0,6298144.0,-70.572760,-33.446160,358826,6304148,-70.517932,-33.392696,transformed,transformed
1,2002,PA002,PA003,358826.0,6304148.0,-70.517932,-33.392696,346643,6297607,-70.649997,-33.449999,transformed,transformed
2,2003,PA003,PA004,346643.0,6297607.0,-70.649997,-33.449999,352169,6301020,-70.590000,-33.420003,transformed,transformed
3,2006,PA001,PA005,353816.0,6298144.0,-70.572760,-33.446160,342049,6294205,-70.700005,-33.480003,transformed,transformed
4,2007,PA006,PA001,356736.0,6306634.0,-70.540002,-33.370004,353816,6298144,-70.572760,-33.446160,transformed,transformed
5,2008,PA003,PA006,346643.0,6297607.0,-70.649997,-33.449999,356736,6306634,-70.540002,-33.370004,transformed,transformed
6,2009,PA999,PA002,NaN,NaN,NaN,NaN,358826,6304148,-70.517932,-33.392696,not_transformed,transformed
7,2010,PA004,PA008,352169.0,6301020.0,-70.590000,-33.420003,349900,6302500,-70.614149,-33.406344,transformed,transformed


### 23. la generacion de indices h3

In [107]:
import pandas as pd
import numpy as np
import h3

df_h3_test = pd.DataFrame({
    "trip_id": [
        "t00", "t01", "t02", "t03", "t04", "t05",
        "t06", "t07", "t08", "t09", "t10", "t11"
    ],

    "origin_latitude": [
        -33.4489,  # Santiago centro
        -33.4489,  # igual a t00
        -33.4565,
        np.nan,    # origen nulo
        -33.3927,
        -33.5000,
        -33.4200,
        -33.4489,  # igual a t00
        -33.4600,
        -33.4700,
        np.nan,    # origen nulo
        -33.3900
    ],
    "origin_longitude": [
        -70.6693,
        -70.6693,
        -70.6483,
        -70.6500,
        -70.5728,
        -70.6000,
        -70.6100,
        -70.6693,
        -70.7000,
        -70.5800,
        np.nan,
        -70.5200
    ],

    "destination_latitude": [
        -33.4565,
        -33.4565,  # igual a t00
        np.nan,    # destino nulo
        -33.5000,
        -33.4200,
        np.nan,    # destino nulo
        -33.3900,
        -33.4565,
        -33.4489,
        -33.3927,
        -33.5000,
        np.nan     # destino nulo
    ],
    "destination_longitude": [
        -70.6483,
        -70.6483,
        -70.6400,
        -70.6000,
        -70.6100,
        np.nan,
        -70.5200,
        -70.6483,
        -70.6693,
        -70.5728,
        -70.6000,
        np.nan
    ],
})

print("DataFrame de ejemplo para testeo de asignación de celdas H3 (con casos nulos y repetidos):")
display(df_h3_test)

DataFrame de ejemplo para testeo de asignación de celdas H3 (con casos nulos y repetidos):


,trip_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude
0,t00,-33.4489,-70.6693,-33.4565,-70.6483
1,t01,-33.4489,-70.6693,-33.4565,-70.6483
2,t02,-33.4565,-70.6483,NaN,-70.6400
3,t03,NaN,-70.6500,-33.5000,-70.6000
4,t04,-33.3927,-70.5728,-33.4200,-70.6100
5,t05,-33.5000,-70.6000,NaN,NaN
6,t06,-33.4200,-70.6100,-33.3900,-70.5200
7,t07,-33.4489,-70.6693,-33.4565,-70.6483
8,t08,-33.4600,-70.7000,-33.4489,-70.6693
9,t09,-33.4700,-70.5800,-33.3927,-70.5728


In [108]:
def is_valid_h3_resolution(resolution: int) -> bool:
    return isinstance(resolution, int) and 0 <= resolution <= 15

def derive_h3_column(
    df: pd.DataFrame,
    *,
    lat_col: str,
    lon_col: str,
    out_col: str,
    resolution: int,
    keep_status: bool = True,
) -> pd.DataFrame:
    if not is_valid_h3_resolution(resolution):
        raise ValueError(f"h3_resolution inválida: {resolution}. Debe estar entre 0 y 15.")

    if lat_col not in df.columns or lon_col not in df.columns:
        raise KeyError(f"Faltan columnas requeridas: {lat_col}, {lon_col}")

    work = df.copy()

    out_values = []
    out_status = []

    for lat, lon in zip(work[lat_col], work[lon_col]):
        if pd.isna(lat) or pd.isna(lon):
            out_values.append(pd.NA)
            out_status.append("missing_coord")
            continue

        try:
            cell = h3.latlng_to_cell(float(lat), float(lon), resolution)
            out_values.append(cell)
            out_status.append("ok")
        except Exception as e:
            out_values.append(pd.NA)
            out_status.append(f"error:{type(e).__name__}")

    work[out_col] = pd.Series(out_values, index=work.index, dtype="string")

    if keep_status:
        work[f"__{out_col}_status"] = pd.Series(out_status, index=work.index, dtype="string")

    return work

def derive_od_h3_indices(
    df: pd.DataFrame,
    *,
    resolution: int,
    keep_status: bool = True,
):
    if not is_valid_h3_resolution(resolution):
        raise ValueError(f"h3_resolution inválida: {resolution}. Debe estar entre 0 y 15.")

    work = df.copy()

    h3_meta = {
        "resolution": resolution,
        "source_fields": [],
        "derived_fields": [],
    }

    origin_pair = {"origin_latitude", "origin_longitude"}
    destination_pair = {"destination_latitude", "destination_longitude"}

    if origin_pair.issubset(work.columns):
        work = derive_h3_column(
            work,
            lat_col="origin_latitude",
            lon_col="origin_longitude",
            out_col="origin_h3_index",
            resolution=resolution,
            keep_status=keep_status,
        )
        h3_meta["source_fields"].append(["origin_latitude", "origin_longitude"])
        h3_meta["derived_fields"].append("origin_h3_index")

    if destination_pair.issubset(work.columns):
        work = derive_h3_column(
            work,
            lat_col="destination_latitude",
            lon_col="destination_longitude",
            out_col="destination_h3_index",
            resolution=resolution,
            keep_status=keep_status,
        )
        h3_meta["source_fields"].append(["destination_latitude", "destination_longitude"])
        h3_meta["derived_fields"].append("destination_h3_index")

    return work, h3_meta

df_h3_out, h3_meta = derive_od_h3_indices(
    df_h3_test,
    resolution=8,
    keep_status=True,
)

df_h3_out[
    [
        "trip_id",
        "origin_latitude", "origin_longitude", "origin_h3_index", "__origin_h3_index_status",
        "destination_latitude", "destination_longitude", "destination_h3_index", "__destination_h3_index_status",
    ]
]

,trip_id,origin_latitude,origin_longitude,origin_h3_index,__origin_h3_index_status,destination_latitude,destination_longitude,destination_h3_index,__destination_h3_index_status
0,t00,-33.4489,-70.6693,88b2c5543bfffff,ok,-33.4565,-70.6483,88b2c554ebfffff,ok
1,t01,-33.4489,-70.6693,88b2c5543bfffff,ok,-33.4565,-70.6483,88b2c554ebfffff,ok
2,t02,-33.4565,-70.6483,88b2c554ebfffff,ok,NaN,-70.6400,<NA>,missing_coord
3,t03,NaN,-70.6500,<NA>,missing_coord,-33.5000,-70.6000,88b2c5090dfffff,ok
4,t04,-33.3927,-70.5728,88b2c519edfffff,ok,-33.4200,-70.6100,88b2c55681fffff,ok
5,t05,-33.5000,-70.6000,88b2c5090dfffff,ok,NaN,NaN,<NA>,missing_coord
6,t06,-33.4200,-70.6100,88b2c55681fffff,ok,-33.3900,-70.5200,88b2c519d9fffff,ok
7,t07,-33.4489,-70.6693,88b2c5543bfffff,ok,-33.4565,-70.6483,88b2c554ebfffff,ok
8,t08,-33.4600,-70.7000,88b2c555c1fffff,ok,-33.4489,-70.6693,88b2c5543bfffff,ok
9,t09,-33.4700,-70.5800,88b2c50b3dfffff,ok,-33.3927,-70.5728,88b2c519edfffff,ok


In [109]:
h3_meta

{'resolution': 8,
 'source_fields': [['origin_latitude', 'origin_longitude'],
  ['destination_latitude', 'destination_longitude']],
 'derived_fields': ['origin_h3_index', 'destination_h3_index']}

In [116]:
try:
    derive_od_h3_indices(df_h3_test, resolution=162)
    assert False, "Debió fallar por resolución inválida"
except ValueError as e:
    print("fallo con error: ", e)
    pass

fallo con error:  h3_resolution inválida: 162. Debe estar entre 0 y 15.


### 24. Chequeo de duplicados cuando movement_id vino de la fuente

In [117]:
import pandas as pd
import numpy as np

df_movement_ok = pd.DataFrame({
    "movement_id": ["m001", "m002", "m003", "m004", "m005", "m006", "m007", "m008"],
    "trip_id":     ["t01",  "t02",  "t03",  "t04",  "t05",  "t06",  "t07",  "t08"],
    "origin_latitude":      [-33.45, -33.46, -33.47, -33.48, -33.49, -33.50, -33.51, -33.52],
    "origin_longitude":     [-70.60, -70.61, -70.62, -70.63, -70.64, -70.65, -70.66, -70.67],
    "destination_latitude": [-33.40, -33.41, -33.42, -33.43, -33.44, -33.45, -33.46, -33.47],
    "destination_longitude":[-70.50, -70.51, -70.52, -70.53, -70.54, -70.55, -70.56, -70.57],
})

df_movement_dup = pd.DataFrame({
    "movement_id": ["m001", "m002", "m003", "m002", "m005", "m006", "m003", "m008", "m008", "m010"],
    "trip_id":     ["t01",  "t02",  "t03",  "t04",  "t05",  "t06",  "t07",  "t08",  "t09",  "t10"],
    "origin_latitude":      [-33.45, -33.46, -33.47, -33.48, -33.49, -33.50, -33.51, -33.52, -33.53, -33.54],
    "origin_longitude":     [-70.60, -70.61, -70.62, -70.63, -70.64, -70.65, -70.66, -70.67, -70.68, -70.69],
    "destination_latitude": [-33.40, -33.41, -33.42, -33.43, -33.44, -33.45, -33.46, -33.47, -33.48, -33.49],
    "destination_longitude":[-70.50, -70.51, -70.52, -70.53, -70.54, -70.55, -70.56, -70.57, -70.58, -70.59],
})

print("DataFrame original con movimientos (sin duplicados):")
display(df_movement_ok)

print("\nDataFrame original con movimientos (con duplicados en movement_id):")
display(df_movement_dup)

DataFrame original con movimientos (sin duplicados):


,movement_id,trip_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude
0,m001,t01,-33.45,-70.60,-33.40,-70.50
1,m002,t02,-33.46,-70.61,-33.41,-70.51
2,m003,t03,-33.47,-70.62,-33.42,-70.52
3,m004,t04,-33.48,-70.63,-33.43,-70.53
4,m005,t05,-33.49,-70.64,-33.44,-70.54
5,m006,t06,-33.50,-70.65,-33.45,-70.55
6,m007,t07,-33.51,-70.66,-33.46,-70.56
7,m008,t08,-33.52,-70.67,-33.47,-70.57



DataFrame original con movimientos (con duplicados en movement_id):


,movement_id,trip_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude
0,m001,t01,-33.45,-70.60,-33.40,-70.50
1,m002,t02,-33.46,-70.61,-33.41,-70.51
2,m003,t03,-33.47,-70.62,-33.42,-70.52
3,m002,t04,-33.48,-70.63,-33.43,-70.53
4,m005,t05,-33.49,-70.64,-33.44,-70.54
5,m006,t06,-33.50,-70.65,-33.45,-70.55
6,m003,t07,-33.51,-70.66,-33.46,-70.56
7,m008,t08,-33.52,-70.67,-33.47,-70.57
8,m008,t09,-33.53,-70.68,-33.48,-70.58
9,m010,t10,-33.54,-70.69,-33.49,-70.59


In [126]:
dup_mask_ok = df_movement_ok["movement_id"].duplicated(keep=False)
dup_mask_dup = df_movement_dup["movement_id"].duplicated(keep=False)

print("OK - filas duplicadas:", dup_mask_ok.sum())
print("DUP - filas duplicadas:", dup_mask_dup.sum())

print("\nMovimientos duplicados en DataFrame OK:")
display(df_movement_dup[dup_mask_dup])

# obtener los IDs repetridos
duplicated_ids = (
    df_movement_dup.loc[dup_mask_dup, "movement_id"]
    .drop_duplicates()
    .tolist()
)

print("IDs duplicados:", duplicated_ids)

# obtener conteo por ID repetido
dup_counts = (
    df_movement_dup.loc[dup_mask_dup, "movement_id"]
    .value_counts()
)

print("\nConteo por ID duplicado:")
print(dup_counts)

OK - filas duplicadas: 0
DUP - filas duplicadas: 6

Movimientos duplicados en DataFrame OK:


,movement_id,trip_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude
1,m002,t02,-33.46,-70.61,-33.41,-70.51
2,m003,t03,-33.47,-70.62,-33.42,-70.52
3,m002,t04,-33.48,-70.63,-33.43,-70.53
6,m003,t07,-33.51,-70.66,-33.46,-70.56
7,m008,t08,-33.52,-70.67,-33.47,-70.57
8,m008,t09,-33.53,-70.68,-33.48,-70.58


IDs duplicados: ['m002', 'm003', 'm008']

Conteo por ID duplicado:
movement_id
m002    2
m003    2
m008    2
Name: count, dtype: int64


In [128]:
def detect_duplicate_movement_ids(df: pd.DataFrame, col: str = "movement_id") -> dict:
    if col not in df.columns:
        raise KeyError(f"No existe la columna {col!r}")

    dup_mask = df[col].duplicated(keep=False)

    duplicated_rows = df.loc[dup_mask].copy()
    duplicated_ids = duplicated_rows[col].drop_duplicates().tolist()
    duplicated_counts = duplicated_rows[col].value_counts().to_dict()

    return {
        "has_duplicates": bool(dup_mask.any()),
        "n_duplicated_rows": int(dup_mask.sum()),
        "duplicated_ids": duplicated_ids,
        "duplicated_counts": duplicated_counts,
        "duplicated_rows": duplicated_rows,
    }

res_ok = detect_duplicate_movement_ids(df_movement_ok)
res_dup = detect_duplicate_movement_ids(df_movement_dup)

print("DataFrame OK - tiene duplicados:", res_ok["has_duplicates"])
print("DataFrame DUP - tiene duplicados:", res_dup["has_duplicates"])
print("IDs duplicados en DataFrame DUP:", res_dup["duplicated_ids"])
print("Conteo por ID duplicado en DataFrame DUP:", res_dup["duplicated_counts"])

DataFrame OK - tiene duplicados: False
DataFrame DUP - tiene duplicados: True
IDs duplicados en DataFrame DUP: ['m002', 'm003', 'm008']
Conteo por ID duplicado en DataFrame DUP: {'m002': 2, 'm003': 2, 'm008': 2}


### 25. Generacion de movement_id cuando no viene en la fuente

In [129]:
import pandas as pd
import numpy as np

df_no_movement_id = pd.DataFrame({
    "trip_id": ["t01", "t02", "t03", "t04", "t05", "t06", "t07", "t08"],
    "origin_latitude":      [-33.45, -33.46, -33.47, -33.48, np.nan, -33.50, -33.51, -33.52],
    "origin_longitude":     [-70.60, -70.61, -70.62, -70.63, -70.64, np.nan, -70.66, -70.67],
    "destination_latitude": [-33.40, -33.41, -33.42, np.nan, -33.44, -33.45, -33.46, -33.47],
    "destination_longitude":[-70.50, -70.51, -70.52, -70.53, -70.54, -70.55, np.nan, -70.57],
    "purpose": ["work", "study", "shopping", "home", "other", "work", "leisure", "study"],
})

print("DataFrame de movimientos sin columna movement_id (con casos nulos):")
display(df_no_movement_id)

DataFrame de movimientos sin columna movement_id (con casos nulos):


,trip_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,purpose
0,t01,-33.45,-70.60,-33.40,-70.50,work
1,t02,-33.46,-70.61,-33.41,-70.51,study
2,t03,-33.47,-70.62,-33.42,-70.52,shopping
3,t04,-33.48,-70.63,NaN,-70.53,home
4,t05,NaN,-70.64,-33.44,-70.54,other
5,t06,-33.50,NaN,-33.45,-70.55,work
6,t07,-33.51,-70.66,-33.46,NaN,leisure
7,t08,-33.52,-70.67,-33.47,-70.57,study


In [136]:
def add_movement_id_if_missing(df: pd.DataFrame, col: str = "movement_id"):
    work = df.copy()
    columns_added = []

    if col not in work.columns:
        movement_ids = pd.Series(
            [f"m{i}" for i in range(len(work))],
            index=work.index,
            dtype="string",
        )
        work.insert(0, col, movement_ids)
        columns_added.append(col)

    return work, columns_added

df_with_movement_id, columns_added = add_movement_id_if_missing(df_no_movement_id)

print("DataFrame después de agregar movement_id si faltaba:")
display(df_with_movement_id)
print("Columnas agregadas:", columns_added)

DataFrame después de agregar movement_id si faltaba:


,movement_id,trip_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,purpose
0,m0,t01,-33.45,-70.60,-33.40,-70.50,work
1,m1,t02,-33.46,-70.61,-33.41,-70.51,study
2,m2,t03,-33.47,-70.62,-33.42,-70.52,shopping
3,m3,t04,-33.48,-70.63,NaN,-70.53,home
4,m4,t05,NaN,-70.64,-33.44,-70.54,other
5,m5,t06,-33.50,NaN,-33.45,-70.55,work
6,m6,t07,-33.51,-70.66,-33.46,NaN,leisure
7,m7,t08,-33.52,-70.67,-33.47,-70.57,study


Columnas agregadas: ['movement_id']


In [137]:
df_indexed = df_with_movement_id.set_index("movement_id", drop=False)
df_indexed.head()

,movement_id,trip_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,purpose
movement_id,,,,,,,
m0,m0,t01,-33.45,-70.60,-33.40,-70.50,work
m1,m1,t02,-33.46,-70.61,-33.41,-70.51,study
m2,m2,t03,-33.47,-70.62,-33.42,-70.52,shopping
m3,m3,t04,-33.48,-70.63,NaN,-70.53,home
m4,m4,t05,NaN,-70.64,-33.44,-70.54,other


### 26. Cuando single_stage=True probar la generación de campos  trip_id y movement_seq

In [139]:
import pandas as pd
import numpy as np

df_single_stage_base = pd.DataFrame({
    "movement_id": ["m0", "m1", "m2", "m3", "m4", "m5"],
    "origin_latitude":      [-33.45, -33.46, -33.47, -33.48, np.nan, -33.50],
    "origin_longitude":     [-70.60, -70.61, -70.62, -70.63, -70.64, np.nan],
    "destination_latitude": [-33.40, -33.41, -33.42, np.nan, -33.44, -33.45],
    "destination_longitude":[-70.50, -70.51, -70.52, -70.53, np.nan, -70.55],
    "purpose": ["work", "study", "shopping", "home", "other", "leisure"],
})

print("DataFrame base para testeo de función de detección de etapa única (con casos nulos):")
display(df_single_stage_base)

df_missing_trip_id = df_single_stage_base.copy()
df_missing_trip_id["movement_seq"] = pd.Series([0, 0, 0, 0, 0, 0], dtype="Int64")
print("DataFrame con columna de secuencia de movimiento (todas 0, indicando etapa única):")
display(df_missing_trip_id)

df_missing_movement_seq = df_single_stage_base.copy()
df_missing_movement_seq["trip_id"] = pd.Series(
    df_missing_movement_seq["movement_id"],
    dtype="string"
)
print("DataFrame con columna de ID de viaje (copiada del ID de movimiento):")
display(df_missing_movement_seq)


DataFrame base para testeo de función de detección de etapa única (con casos nulos):


,movement_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,purpose
0,m0,-33.45,-70.60,-33.40,-70.50,work
1,m1,-33.46,-70.61,-33.41,-70.51,study
2,m2,-33.47,-70.62,-33.42,-70.52,shopping
3,m3,-33.48,-70.63,NaN,-70.53,home
4,m4,NaN,-70.64,-33.44,NaN,other
5,m5,-33.50,NaN,-33.45,-70.55,leisure


DataFrame con columna de secuencia de movimiento (todas 0, indicando etapa única):


,movement_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,purpose,movement_seq
0,m0,-33.45,-70.60,-33.40,-70.50,work,0
1,m1,-33.46,-70.61,-33.41,-70.51,study,0
2,m2,-33.47,-70.62,-33.42,-70.52,shopping,0
3,m3,-33.48,-70.63,NaN,-70.53,home,0
4,m4,NaN,-70.64,-33.44,NaN,other,0
5,m5,-33.50,NaN,-33.45,-70.55,leisure,0


DataFrame con columna de ID de viaje (copiada del ID de movimiento):


,movement_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,purpose,trip_id
0,m0,-33.45,-70.60,-33.40,-70.50,work,m0
1,m1,-33.46,-70.61,-33.41,-70.51,study,m1
2,m2,-33.47,-70.62,-33.42,-70.52,shopping,m2
3,m3,-33.48,-70.63,NaN,-70.53,home,m3
4,m4,NaN,-70.64,-33.44,NaN,other,m4
5,m5,-33.50,NaN,-33.45,-70.55,leisure,m5


In [140]:
def add_single_stage_fields_if_missing(
    df: pd.DataFrame,
    *,
    single_stage: bool,
    movement_id_col: str = "movement_id",
):
    work = df.copy()
    columns_added = []

    if not single_stage:
        return work, columns_added

    if movement_id_col not in work.columns:
        raise KeyError(f"Falta la columna requerida {movement_id_col!r} para derivar trip_id en single_stage=True")

    if "trip_id" not in work.columns:
        trip_ids = pd.Series(work[movement_id_col], index=work.index, dtype="string")
        work.insert(1 if work.columns[0] == movement_id_col else 0, "trip_id", trip_ids)
        columns_added.append("trip_id")

    if "movement_seq" not in work.columns:
        movement_seq = pd.Series([0] * len(work), index=work.index, dtype="Int64")
        insert_pos = 2 if {"movement_id", "trip_id"}.issubset(work.columns[:2]) else len(work.columns)
        work.insert(insert_pos, "movement_seq", movement_seq)
        columns_added.append("movement_seq")

    return work, columns_added

df_s12_out, columns_added = add_single_stage_fields_if_missing(
    df_single_stage_base,
    single_stage=True,
)

print("DataFrame después de agregar campos de etapa única si faltaban:")
display(df_s12_out)
print("Columnas agregadas:", columns_added)

DataFrame después de agregar campos de etapa única si faltaban:


,movement_id,trip_id,movement_seq,origin_latitude,origin_longitude,destination_latitude,destination_longitude,purpose
0,m0,m0,0,-33.45,-70.60,-33.40,-70.50,work
1,m1,m1,0,-33.46,-70.61,-33.41,-70.51,study
2,m2,m2,0,-33.47,-70.62,-33.42,-70.52,shopping
3,m3,m3,0,-33.48,-70.63,NaN,-70.53,home
4,m4,m4,0,NaN,-70.64,-33.44,NaN,other
5,m5,m5,0,-33.50,NaN,-33.45,-70.55,leisure


Columnas agregadas: ['trip_id', 'movement_seq']


In [144]:
# Caso en que falta solo trip_id
df_out_a, added_a = add_single_stage_fields_if_missing(
    df_missing_trip_id,
    single_stage=True,
)
print("DataFrame después de agregar campos de etapa única (sin trip_id) si faltaban:")
display(df_out_a)
print("Columnas agregadas:", added_a)

DataFrame después de agregar campos de etapa única (sin trip_id) si faltaban:


,movement_id,trip_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,purpose,movement_seq
0,m0,m0,-33.45,-70.60,-33.40,-70.50,work,0
1,m1,m1,-33.46,-70.61,-33.41,-70.51,study,0
2,m2,m2,-33.47,-70.62,-33.42,-70.52,shopping,0
3,m3,m3,-33.48,-70.63,NaN,-70.53,home,0
4,m4,m4,NaN,-70.64,-33.44,NaN,other,0
5,m5,m5,-33.50,NaN,-33.45,-70.55,leisure,0


Columnas agregadas: ['trip_id']


In [145]:
# Caso en que falta solo movement_seq
df_out_b, added_b = add_single_stage_fields_if_missing(
    df_missing_movement_seq,
    single_stage=True,
)
print("DataFrame después de agregar campos de etapa única (sin movement_seq) si faltaban:")
display(df_out_b)
print("Columnas agregadas:", added_b)

DataFrame después de agregar campos de etapa única (sin movement_seq) si faltaban:


,movement_id,origin_latitude,origin_longitude,destination_latitude,destination_longitude,purpose,trip_id,movement_seq
0,m0,-33.45,-70.60,-33.40,-70.50,work,m0,0
1,m1,-33.46,-70.61,-33.41,-70.51,study,m1,0
2,m2,-33.47,-70.62,-33.42,-70.52,shopping,m2,0
3,m3,-33.48,-70.63,NaN,-70.53,home,m3,0
4,m4,NaN,-70.64,-33.44,NaN,other,m4,0
5,m5,-33.50,NaN,-33.45,-70.55,leisure,m5,0


Columnas agregadas: ['movement_seq']


### 27. Cuando  single_stage=False Probar la deteccion de los campos trip_id y movement_seq en los exigidos por el schema

In [146]:
import pandas as pd
import numpy as np
from pylondrina.schema import TripSchema, FieldSpec

schema_requires_trip_and_seq = TripSchema(
    version="0.1.0",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=True),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=True),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=False),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=False),
    },
    required=["movement_id", "trip_id", "movement_seq"],
    semantic_rules=None,
)

schema_does_not_require_trip_and_seq = TripSchema(
    version="0.1.0",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=False),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=False),
    },
    required=["movement_id"],
    semantic_rules=None,
)

df_with_trip_and_seq_mixed = pd.DataFrame({
    "movement_id": ["m0", "m1", "m2", "m3", "m4", "m5", "m6", "m7"],
    "trip_id": [
        "t0",      # válido
        101,       # válido
        "t2",      # válido
        None,      # nulo
        3.14,      # inválido
        ["x"],     # inválido
        "t6",      # válido
        True,      # mejor tratarlo como inválido
    ],
    "movement_seq": [
        0,         # válido
        1,         # válido
        2,         # válido
        None,      # nulo
        1.5,       # inválido
        "2",       # inválido para esta prueba
        -1,        # int pero negativo
        True,      # mejor tratarlo como inválido
    ],
    "origin_latitude": [-33.45, -33.46, -33.47, -33.48, -33.49, -33.50, -33.51, -33.52],
    "origin_longitude": [-70.60, -70.61, -70.62, -70.63, -70.64, -70.65, -70.66, -70.67],
})

df_without_trip_and_seq = pd.DataFrame({
    "movement_id": ["m0", "m1", "m2", "m3"],
    "origin_latitude": [-33.45, -33.46, -33.47, -33.48],
    "origin_longitude": [-70.60, -70.61, -70.62, -70.63],
})

print("DataFrame con trip_id y movement_seq con valores mixtos (válidos, nulos e inválidos):")
display(df_with_trip_and_seq_mixed)
print("\nDataFrame sin trip_id y movement_seq:")
display(df_without_trip_and_seq)

DataFrame con trip_id y movement_seq con valores mixtos (válidos, nulos e inválidos):


,movement_id,trip_id,movement_seq,origin_latitude,origin_longitude
0,m0,t0,0,-33.45,-70.60
1,m1,101,1,-33.46,-70.61
2,m2,t2,2,-33.47,-70.62
3,m3,None,None,-33.48,-70.63
4,m4,3.14,1.5,-33.49,-70.64
5,m5,[x],2,-33.50,-70.65
6,m6,t6,-1,-33.51,-70.66
7,m7,True,True,-33.52,-70.67



DataFrame sin trip_id y movement_seq:


,movement_id,origin_latitude,origin_longitude
0,m0,-33.45,-70.60
1,m1,-33.46,-70.61
2,m2,-33.47,-70.62
3,m3,-33.48,-70.63


In [ ]:
# Detectar si el schema exige esos campos

def schema_requires_fields(schema: TripSchema, fields: list[str]) -> dict[str, bool]:
    required_set = set(schema.required)
    return {f: f in required_set for f in fields}

print("Campos requeridos en schema_requires_trip_and_seq:")
print(schema_requires_fields(schema_requires_trip_and_seq, ["trip_id", "movement_seq"]))

print("\nCampos requeridos en schema_does_not_require_trip_and_seq:")
print(schema_requires_fields(schema_does_not_require_trip_and_seq, ["trip_id", "movement_seq"]))

Campos requeridos en schema_requires_trip_and_seq:
{'trip_id': True, 'movement_seq': True}

Campos requeridos en schema_does_not_require_trip_and_seq:
{'trip_id': False, 'movement_seq': False}


In [148]:
# Detectar si estan presentes en el DataFrame

def dataframe_has_fields(df: pd.DataFrame, fields: list[str]) -> dict[str, bool]:
    cols = set(df.columns)
    return {f: f in cols for f in fields}

print("Campos presentes en df_with_trip_and_seq_mixed:")
print(dataframe_has_fields(df_with_trip_and_seq_mixed, ["trip_id", "movement_seq"]))

print("\nCampos presentes en df_without_trip_and_seq:")
print(dataframe_has_fields(df_without_trip_and_seq, ["trip_id", "movement_seq"]))

Campos presentes en df_with_trip_and_seq_mixed:
{'trip_id': True, 'movement_seq': True}

Campos presentes en df_without_trip_and_seq:
{'trip_id': False, 'movement_seq': False}


Estas validaciones no deberian hacerse en import, se deben hacer en validate trips indicando los dtype y constraints correspondientes. Asi que esta parte es solo de prueba:

In [149]:
import numpy as np
import pandas as pd

def is_valid_trip_id_value(v) -> bool:
    if pd.isna(v):
        return True
    if isinstance(v, bool):
        return False
    return isinstance(v, (str, int, np.integer))

def is_valid_movement_seq_int(v) -> bool:
    if pd.isna(v):
        return True
    if isinstance(v, bool):
        return False
    return isinstance(v, (int, np.integer))

def is_valid_movement_seq_nonnegative(v) -> bool:
    if pd.isna(v):
        return True
    if isinstance(v, bool):
        return False
    return isinstance(v, (int, np.integer)) and v >= 0

def check_trip_and_seq_for_single_stage_false(
    df: pd.DataFrame,
    schema: TripSchema,
):
    required_flags = schema_requires_fields(schema, ["trip_id", "movement_seq"])
    present_flags = dataframe_has_fields(df, ["trip_id", "movement_seq"])

    missing_required = [
        f for f in ["trip_id", "movement_seq"]
        if required_flags[f] and not present_flags[f]
    ]

    trip_id_bad_rows = []
    movement_seq_bad_type_rows = []
    movement_seq_negative_rows = []

    if present_flags["trip_id"]:
        for idx, v in df["trip_id"].items():
            if not is_valid_trip_id_value(v):
                trip_id_bad_rows.append(idx)

    if present_flags["movement_seq"]:
        for idx, v in df["movement_seq"].items():
            if not is_valid_movement_seq_int(v):
                movement_seq_bad_type_rows.append(idx)
            elif not is_valid_movement_seq_nonnegative(v):
                movement_seq_negative_rows.append(idx)

    return {
        "schema_requires": required_flags,
        "dataframe_has": present_flags,
        "missing_required": missing_required,
        "trip_id_bad_rows": trip_id_bad_rows,
        "movement_seq_bad_type_rows": movement_seq_bad_type_rows,
        "movement_seq_negative_rows": movement_seq_negative_rows,
        "should_abort_for_missing_required": len(missing_required) > 0,
    }

In [150]:
# Caso 1: schema exige ambos, dataframe los tiene pero con valores malos
res1 = check_trip_and_seq_for_single_stage_false(
    df_with_trip_and_seq_mixed,
    schema_requires_trip_and_seq,
)

res1

{'schema_requires': {'trip_id': True, 'movement_seq': True},
 'dataframe_has': {'trip_id': True, 'movement_seq': True},
 'missing_required': [],
 'trip_id_bad_rows': [4, 5, 7],
 'movement_seq_bad_type_rows': [4, 5, 7],
 'movement_seq_negative_rows': [6],
 'should_abort_for_missing_required': False}

In [151]:
# Caso 2: schema exige ambos, dataframe no los tiene
res2 = check_trip_and_seq_for_single_stage_false(
    df_without_trip_and_seq,
    schema_requires_trip_and_seq,
)

res2

{'schema_requires': {'trip_id': True, 'movement_seq': True},
 'dataframe_has': {'trip_id': False, 'movement_seq': False},
 'missing_required': ['trip_id', 'movement_seq'],
 'trip_id_bad_rows': [],
 'movement_seq_bad_type_rows': [],
 'movement_seq_negative_rows': [],
 'should_abort_for_missing_required': True}

In [152]:
# Caso 3: schema no los exige, dataframe no los tiene
res3 = check_trip_and_seq_for_single_stage_false(
    df_without_trip_and_seq,
    schema_does_not_require_trip_and_seq,
)

res3

{'schema_requires': {'trip_id': False, 'movement_seq': False},
 'dataframe_has': {'trip_id': False, 'movement_seq': False},
 'missing_required': [],
 'trip_id_bad_rows': [],
 'movement_seq_bad_type_rows': [],
 'movement_seq_negative_rows': [],
 'should_abort_for_missing_required': False}

Regla operativa que deja mucho más simple:

In [ ]:
required_trip_fields = {"trip_id", "movement_seq"} & set(schema.required)
missing_required_trip_fields = required_trip_fields - set(work.columns)

#f not single_stage and missing_required_trip_fields:
    # abort

### 28. Seleccion final de columnas del schema

In [157]:
import pandas as pd
from pylondrina.schema import TripSchema, FieldSpec
from pylondrina.importing import ImportOptions

schema28 = TripSchema(
    version="0.1.0",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=True),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=True),
        "purpose": FieldSpec(name="purpose", dtype="categorical", required=False),
        "mode": FieldSpec(name="mode", dtype="categorical", required=False),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=False),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=False),
        "destination_latitude": FieldSpec(name="destination_latitude", dtype="float", required=False),
        "destination_longitude": FieldSpec(name="destination_longitude", dtype="float", required=False),
        "gender": FieldSpec(name="gender", dtype="categorical", required=False),
    },
    required=["movement_id", "trip_id", "movement_seq"],
)

options28_ok = ImportOptions(
    keep_extra_fields=False,
    selected_fields=["purpose", "mode", "origin_latitude", "origin_longitude"],
)

options28_bad = ImportOptions(
    keep_extra_fields=False,
    selected_fields=["purpose", "mode", "fake_field", "another_fake"],
)

work28 = pd.DataFrame({
    "movement_id": ["m0", "m1", "m2"],
    "trip_id": ["t0", "t1", "t2"],
    "movement_seq": [0, 0, 0],
    "purpose": ["work", "study", "shopping"],
    "mode": ["bus", "metro", "walk"],
    "origin_latitude": [-33.45, -33.46, -33.47],
    "origin_longitude": [-70.60, -70.61, -70.62],
    "destination_latitude": [-33.40, -33.41, -33.42],
    "destination_longitude": [-70.50, -70.51, -70.52],
    "gender": ["F", "M", "F"],
    "raw_source_col": ["x", "y", "z"],       # extra
    "debug_flag": [True, False, True],       # extra
})

print("DataFrame original con campos extra no seleccionados en opciones de importación:")
display(work28)

DataFrame original con campos extra no seleccionados en opciones de importación:


,movement_id,trip_id,movement_seq,purpose,mode,origin_latitude,origin_longitude,destination_latitude,destination_longitude,gender,raw_source_col,debug_flag
0,m0,t0,0,work,bus,-33.45,-70.60,-33.40,-70.50,F,x,True
1,m1,t1,0,study,metro,-33.46,-70.61,-33.41,-70.51,M,y,False
2,m2,t2,0,shopping,walk,-33.47,-70.62,-33.42,-70.52,F,z,True


In [160]:
def build_keep_schema_fields(schema: TripSchema, selected_fields):
    schema_fields = set(schema.fields.keys())
    required_fields = set(schema.required)

    if selected_fields:
        selected_set = set(selected_fields)
        invalid_selected = sorted(selected_set - schema_fields)
        if invalid_selected:
            raise ValueError(
                f"selected_fields contiene campos fuera del schema: {invalid_selected}"
            )
        keep_schema_fields = required_fields | selected_set
    else:
        keep_schema_fields = set(schema_fields)

    return {
        "schema_fields": schema_fields,
        "required_fields": required_fields,
        "keep_schema_fields": keep_schema_fields,
    }

# Caso valido
res_ok = build_keep_schema_fields(schema28, options28_ok.selected_fields)
print("schema_fields:", sorted(res_ok["schema_fields"]))
print("required_fields:", sorted(res_ok["required_fields"]))
print("keep_schema_fields:", sorted(res_ok["keep_schema_fields"]))

schema_fields: ['destination_latitude', 'destination_longitude', 'gender', 'mode', 'movement_id', 'movement_seq', 'origin_latitude', 'origin_longitude', 'purpose', 'trip_id']
required_fields: ['movement_id', 'movement_seq', 'trip_id']
keep_schema_fields: ['mode', 'movement_id', 'movement_seq', 'origin_latitude', 'origin_longitude', 'purpose', 'trip_id']


In [ ]:
# Simulacion de un drop real
def apply_final_column_selection(work: pd.DataFrame, schema: TripSchema, options: ImportOptions):
    cols_before = list(work.columns)

    res = build_keep_schema_fields(schema, options.selected_fields)
    keep_schema_fields = res["keep_schema_fields"]
    schema_fields = set(schema.fields.keys())

    if options.keep_extra_fields:
        final_cols = [c for c in work.columns if (c in keep_schema_fields) or (c not in schema_fields)]
    else:
        final_cols = [c for c in work.columns if c in keep_schema_fields]

    work_out = work.loc[:, final_cols].copy()

    columns_deleted = [c for c in cols_before if c not in work_out.columns]
    extra_fields_kept = [c for c in work_out.columns if c not in schema_fields]

    return work_out, columns_deleted, extra_fields_kept, sorted(keep_schema_fields)

work28_out, columns_deleted, extra_fields_kept, keep_schema_fields = apply_final_column_selection(
    work28,
    schema28,
    options28_ok,
)

print("keep_schema_fields:", keep_schema_fields)
print("columns_deleted:", columns_deleted)
print("extra_fields_kept:", extra_fields_kept)
display(work28_out)


keep_schema_fields: ['mode', 'movement_id', 'movement_seq', 'origin_latitude', 'origin_longitude', 'purpose', 'trip_id']
columns_deleted: ['destination_latitude', 'destination_longitude', 'gender', 'raw_source_col', 'debug_flag']
extra_fields_kept: []


,movement_id,trip_id,movement_seq,purpose,mode,origin_latitude,origin_longitude
0,m0,t0,0,work,bus,-33.45,-70.60
1,m1,t1,0,study,metro,-33.46,-70.61
2,m2,t2,0,shopping,walk,-33.47,-70.62


In [163]:
try:
    work28_out, columns_deleted, extra_fields_kept, keep_schema_fields = apply_final_column_selection(
        work28,
        schema28,
        options28_bad,
    )
    print("keep_schema_fields:", keep_schema_fields)
    print("columns_deleted:", columns_deleted)
    print("extra_fields_kept:", extra_fields_kept)
    display(work28_out)
except ValueError as e:
    print(f"Error: {e}")


Error: selected_fields contiene campos fuera del schema: ['another_fake', 'fake_field']


### 29. Para el caso keep_extra_fields=False probar eliminar campos que no estan en el schema

In [164]:
import pandas as pd
import numpy as np
from pylondrina.schema import TripSchema, FieldSpec
from pylondrina.importing import ImportOptions

schema29 = TripSchema(
    version="0.1.0",
    fields={
        # requeridos
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=True),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=True),

        # opcionales del schema
        "purpose": FieldSpec(name="purpose", dtype="categorical", required=False),
        "mode": FieldSpec(name="mode", dtype="categorical", required=False),
        "gender": FieldSpec(name="gender", dtype="categorical", required=False),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=False),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=False),
        "destination_latitude": FieldSpec(name="destination_latitude", dtype="float", required=False),
        "destination_longitude": FieldSpec(name="destination_longitude", dtype="float", required=False),
        "origin_h3_index": FieldSpec(name="origin_h3_index", dtype="string", required=False),
        "destination_h3_index": FieldSpec(name="destination_h3_index", dtype="string", required=False),
        "survey_date": FieldSpec(name="survey_date", dtype="string", required=False),
    },
    required=["movement_id", "trip_id", "movement_seq"],
)

work29 = pd.DataFrame({
    # requeridos
    "movement_id": ["m0", "m1", "m2", "m3", "m4", "m5"],
    "trip_id": ["t0", "t1", "t2", "t3", "t4", "t5"],
    "movement_seq": pd.Series([0, 0, 0, 0, 0, 0], dtype="Int64"),

    # campos del schema
    "purpose": ["work", "study", "shopping", "home", "other", "leisure"],
    "mode": ["bus", "metro", "walk", "bike", "car", "bus"],
    "gender": ["F", "M", "F", "M", pd.NA, "F"],
    "origin_latitude": [-33.45, -33.46, -33.47, -33.48, np.nan, -33.50],
    "origin_longitude": [-70.60, -70.61, -70.62, -70.63, -70.64, np.nan],
    "destination_latitude": [-33.40, -33.41, -33.42, np.nan, -33.44, -33.45],
    "destination_longitude": [-70.50, -70.51, -70.52, -70.53, np.nan, -70.55],
    "origin_h3_index": ["88e35a99a1fffff", "88e35a99a1fffff", pd.NA, "88e35a99a9fffff", pd.NA, "88e35a99abfffff"],
    "destination_h3_index": ["88e35a99b3fffff", pd.NA, "88e35a99b7fffff", "88e35a99bbfffff", pd.NA, pd.NA],
    "survey_date": ["2025-03-01", "2025-03-01", "2025-03-02", "2025-03-02", "2025-03-03", "2025-03-03"],

    # extras
    "raw_source_col": ["A", "B", "C", "D", "E", "F"],
    "debug_flag": [True, False, True, False, True, False],
    "import_batch": ["b1", "b1", "b1", "b2", "b2", "b2"],
    "source_quality_score": [0.95, 0.88, 0.91, 0.72, 0.67, 0.99],
})

print("DataFrame original con campos extras:")
display(work29)

DataFrame original con campos extras:


,movement_id,trip_id,movement_seq,purpose,mode,gender,origin_latitude,origin_longitude,destination_latitude,destination_longitude,origin_h3_index,destination_h3_index,survey_date,raw_source_col,debug_flag,import_batch,source_quality_score
0,m0,t0,0,work,bus,F,-33.45,-70.60,-33.40,-70.50,88e35a99a1fffff,88e35a99b3fffff,2025-03-01,A,True,b1,0.95
1,m1,t1,0,study,metro,M,-33.46,-70.61,-33.41,-70.51,88e35a99a1fffff,<NA>,2025-03-01,B,False,b1,0.88
2,m2,t2,0,shopping,walk,F,-33.47,-70.62,-33.42,-70.52,<NA>,88e35a99b7fffff,2025-03-02,C,True,b1,0.91
3,m3,t3,0,home,bike,M,-33.48,-70.63,NaN,-70.53,88e35a99a9fffff,88e35a99bbfffff,2025-03-02,D,False,b2,0.72
4,m4,t4,0,other,car,<NA>,NaN,-70.64,-33.44,NaN,<NA>,<NA>,2025-03-03,E,True,b2,0.67
5,m5,t5,0,leisure,bus,F,-33.50,NaN,-33.45,-70.55,88e35a99abfffff,<NA>,2025-03-03,F,False,b2,0.99


In [165]:
options29_1 = ImportOptions(
    selected_fields=[
        "purpose",
        "mode",
        "origin_latitude",
        "origin_longitude",
    ],
    keep_extra_fields=False,
)

options29_2 = ImportOptions(
    selected_fields=None,
    keep_extra_fields=False,
)

options29_3 = ImportOptions(
    selected_fields=[
        "purpose",
        "mode",
        "origin_latitude",
        "origin_longitude",
    ],
    keep_extra_fields=True,
)

options29_4 = ImportOptions(
    selected_fields=None,
    keep_extra_fields=True,
)

In [166]:
# Caso: 1. options.selected_fields es un subconjunto de schema.fields y options.keep_extra_fields=False
print("Caso 1: selected_fields es un subconjunto de schema.fields y keep_extra_fields=False")

res29_1 = build_keep_schema_fields(schema29, options29_1.selected_fields)
work29_1, columns_deleted29_1, extra_fields_kept29_1, keep_schema_fields29_1 = apply_final_column_selection(
    work29,
    schema29,
    options29_1,
)

print("keep_schema_fields29_1:", keep_schema_fields29_1)
print("columns_deleted29_1:", columns_deleted29_1)
print("extra_fields_kept29_1:", extra_fields_kept29_1)
display(work29_1)

Caso 1: selected_fields es un subconjunto de schema.fields y keep_extra_fields=False
keep_schema_fields29_1: ['mode', 'movement_id', 'movement_seq', 'origin_latitude', 'origin_longitude', 'purpose', 'trip_id']
columns_deleted29_1: ['gender', 'destination_latitude', 'destination_longitude', 'origin_h3_index', 'destination_h3_index', 'survey_date', 'raw_source_col', 'debug_flag', 'import_batch', 'source_quality_score']
extra_fields_kept29_1: []


,movement_id,trip_id,movement_seq,purpose,mode,origin_latitude,origin_longitude
0,m0,t0,0,work,bus,-33.45,-70.60
1,m1,t1,0,study,metro,-33.46,-70.61
2,m2,t2,0,shopping,walk,-33.47,-70.62
3,m3,t3,0,home,bike,-33.48,-70.63
4,m4,t4,0,other,car,NaN,-70.64
5,m5,t5,0,leisure,bus,-33.50,NaN


In [167]:
# Caso: 2. options.selected_fields es None y options.keep_extra_fields=False
print("Caso 2: selected_fields es None y keep_extra_fields=False")

res29_2 = build_keep_schema_fields(schema29, options29_2.selected_fields)
work29_2, columns_deleted29_2, extra_fields_kept29_2, keep_schema_fields29_2 = apply_final_column_selection(
    work29,
    schema29,
    options29_2,
)

print("keep_schema_fields29_2:", keep_schema_fields29_2)
print("columns_deleted29_2:", columns_deleted29_2)
print("extra_fields_kept29_2:", extra_fields_kept29_2)
display(work29_2)

Caso 2: selected_fields es None y keep_extra_fields=False
keep_schema_fields29_2: ['destination_h3_index', 'destination_latitude', 'destination_longitude', 'gender', 'mode', 'movement_id', 'movement_seq', 'origin_h3_index', 'origin_latitude', 'origin_longitude', 'purpose', 'survey_date', 'trip_id']
columns_deleted29_2: ['raw_source_col', 'debug_flag', 'import_batch', 'source_quality_score']
extra_fields_kept29_2: []


,movement_id,trip_id,movement_seq,purpose,mode,gender,origin_latitude,origin_longitude,destination_latitude,destination_longitude,origin_h3_index,destination_h3_index,survey_date
0,m0,t0,0,work,bus,F,-33.45,-70.60,-33.40,-70.50,88e35a99a1fffff,88e35a99b3fffff,2025-03-01
1,m1,t1,0,study,metro,M,-33.46,-70.61,-33.41,-70.51,88e35a99a1fffff,<NA>,2025-03-01
2,m2,t2,0,shopping,walk,F,-33.47,-70.62,-33.42,-70.52,<NA>,88e35a99b7fffff,2025-03-02
3,m3,t3,0,home,bike,M,-33.48,-70.63,NaN,-70.53,88e35a99a9fffff,88e35a99bbfffff,2025-03-02
4,m4,t4,0,other,car,<NA>,NaN,-70.64,-33.44,NaN,<NA>,<NA>,2025-03-03
5,m5,t5,0,leisure,bus,F,-33.50,NaN,-33.45,-70.55,88e35a99abfffff,<NA>,2025-03-03


In [168]:
# Caso: 3. options.selected_fields es un subconjunto de schema.fields y options.keep_extra_fields=True
print("Caso 3: selected_fields es un subconjunto de schema.fields y keep_extra_fields=True")

res29_3 = build_keep_schema_fields(schema29, options29_3.selected_fields)
work29_3, columns_deleted29_3, extra_fields_kept29_3, keep_schema_fields29_3 = apply_final_column_selection(
    work29,
    schema29,
    options29_3,
)

print("keep_schema_fields29_3:", keep_schema_fields29_3)
print("columns_deleted29_3:", columns_deleted29_3)
print("extra_fields_kept29_3:", extra_fields_kept29_3)
display(work29_3)

Caso 3: selected_fields es un subconjunto de schema.fields y keep_extra_fields=True
keep_schema_fields29_3: ['mode', 'movement_id', 'movement_seq', 'origin_latitude', 'origin_longitude', 'purpose', 'trip_id']
columns_deleted29_3: ['gender', 'destination_latitude', 'destination_longitude', 'origin_h3_index', 'destination_h3_index', 'survey_date']
extra_fields_kept29_3: ['raw_source_col', 'debug_flag', 'import_batch', 'source_quality_score']


,movement_id,trip_id,movement_seq,purpose,mode,origin_latitude,origin_longitude,raw_source_col,debug_flag,import_batch,source_quality_score
0,m0,t0,0,work,bus,-33.45,-70.60,A,True,b1,0.95
1,m1,t1,0,study,metro,-33.46,-70.61,B,False,b1,0.88
2,m2,t2,0,shopping,walk,-33.47,-70.62,C,True,b1,0.91
3,m3,t3,0,home,bike,-33.48,-70.63,D,False,b2,0.72
4,m4,t4,0,other,car,NaN,-70.64,E,True,b2,0.67
5,m5,t5,0,leisure,bus,-33.50,NaN,F,False,b2,0.99


In [169]:
# Caso: 4. options.selected_fields es None y options.keep_extra_fields=True
print("Caso 4: selected_fields es None y keep_extra_fields=True")
res29_4 = build_keep_schema_fields(schema29, options29_4.selected_fields)
work29_4, columns_deleted29_4, extra_fields_kept29_4, keep_schema_fields29_4 = apply_final_column_selection(
    work29,
    schema29,
    options29_4,
)

print("keep_schema_fields29_4:", keep_schema_fields29_4)
print("columns_deleted29_4:", columns_deleted29_4)
print("extra_fields_kept29_4:", extra_fields_kept29_4)
display(work29_4)

Caso 4: selected_fields es None y keep_extra_fields=True
keep_schema_fields29_4: ['destination_h3_index', 'destination_latitude', 'destination_longitude', 'gender', 'mode', 'movement_id', 'movement_seq', 'origin_h3_index', 'origin_latitude', 'origin_longitude', 'purpose', 'survey_date', 'trip_id']
columns_deleted29_4: []
extra_fields_kept29_4: ['raw_source_col', 'debug_flag', 'import_batch', 'source_quality_score']


,movement_id,trip_id,movement_seq,purpose,mode,gender,origin_latitude,origin_longitude,destination_latitude,destination_longitude,origin_h3_index,destination_h3_index,survey_date,raw_source_col,debug_flag,import_batch,source_quality_score
0,m0,t0,0,work,bus,F,-33.45,-70.60,-33.40,-70.50,88e35a99a1fffff,88e35a99b3fffff,2025-03-01,A,True,b1,0.95
1,m1,t1,0,study,metro,M,-33.46,-70.61,-33.41,-70.51,88e35a99a1fffff,<NA>,2025-03-01,B,False,b1,0.88
2,m2,t2,0,shopping,walk,F,-33.47,-70.62,-33.42,-70.52,<NA>,88e35a99b7fffff,2025-03-02,C,True,b1,0.91
3,m3,t3,0,home,bike,M,-33.48,-70.63,NaN,-70.53,88e35a99a9fffff,88e35a99bbfffff,2025-03-02,D,False,b2,0.72
4,m4,t4,0,other,car,<NA>,NaN,-70.64,-33.44,NaN,<NA>,<NA>,2025-03-03,E,True,b2,0.67
5,m5,t5,0,leisure,bus,F,-33.50,NaN,-33.45,-70.55,88e35a99abfffff,<NA>,2025-03-03,F,False,b2,0.99


### 31. El chequeo final de los campos obligatorios (post-derivaciones y port-seleccion)

In [170]:
import pandas as pd
import numpy as np
from pylondrina.schema import TripSchema, FieldSpec

schema31 = TripSchema(
    version="0.1.0",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=False),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=False),
        "purpose": FieldSpec(name="purpose", dtype="categorical", required=True),
        "mode": FieldSpec(name="mode", dtype="categorical", required=True),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=True),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=True),
        "destination_latitude": FieldSpec(name="destination_latitude", dtype="float", required=False),
        "destination_longitude": FieldSpec(name="destination_longitude", dtype="float", required=False),
    },
    required=[
        "movement_id",
        "purpose",
        "mode",
        "origin_latitude",
        "origin_longitude",
    ],
)


df31_complete = pd.DataFrame({
    "movement_id": ["m0", "m1", "m2", "m3", "m4"],
    "trip_id": ["t0", "t1", "t2", "t3", "t4"],
    "movement_seq": pd.Series([0, 0, 0, 0, 0], dtype="Int64"),
    "purpose": ["work", "study", "shopping", "home", "other"],
    "mode": ["bus", "metro", "walk", "bike", "car"],
    "origin_latitude": [-33.45, -33.46, -33.47, -33.48, -33.49],
    "origin_longitude": [-70.60, -70.61, -70.62, -70.63, -70.64],
    "destination_latitude": [-33.40, -33.41, -33.42, -33.43, -33.44],
    "destination_longitude": [-70.50, -70.51, -70.52, -70.53, -70.54],
})

df31_without_ids = pd.DataFrame({
    "purpose": ["work", "study", "shopping", "home", "other"],
    "mode": ["bus", "metro", "walk", "bike", "car"],
    "origin_latitude": [-33.45, -33.46, -33.47, -33.48, -33.49],
    "origin_longitude": [-70.60, -70.61, -70.62, -70.63, -70.64],
    "destination_latitude": [-33.40, -33.41, -33.42, -33.43, -33.44],
    "destination_longitude": [-70.50, -70.51, -70.52, -70.53, -70.54],
})

df31_missing_required = pd.DataFrame({
    "movement_id": ["m0", "m1", "m2", "m3", "m4"],
    "trip_id": ["t0", "t1", "t2", "t3", "t4"],
    "movement_seq": pd.Series([0, 0, 0, 0, 0], dtype="Int64"),
    "purpose": ["work", "study", "shopping", "home", "other"],
    # "mode" falta
    "origin_latitude": [-33.45, -33.46, -33.47, -33.48, -33.49],
    "origin_longitude": [-70.60, -70.61, -70.62, -70.63, -70.64],
})

print("DataFrame completo con todos los campos requeridos y opcionales:")
display(df31_complete)

print("DataFrame sin los campos de identificación:")
display(df31_without_ids)

print("DataFrame con campos requeridos faltantes:")
display(df31_missing_required)

DataFrame completo con todos los campos requeridos y opcionales:


,movement_id,trip_id,movement_seq,purpose,mode,origin_latitude,origin_longitude,destination_latitude,destination_longitude
0,m0,t0,0,work,bus,-33.45,-70.60,-33.40,-70.50
1,m1,t1,0,study,metro,-33.46,-70.61,-33.41,-70.51
2,m2,t2,0,shopping,walk,-33.47,-70.62,-33.42,-70.52
3,m3,t3,0,home,bike,-33.48,-70.63,-33.43,-70.53
4,m4,t4,0,other,car,-33.49,-70.64,-33.44,-70.54


DataFrame sin los campos de identificación:


,purpose,mode,origin_latitude,origin_longitude,destination_latitude,destination_longitude
0,work,bus,-33.45,-70.60,-33.40,-70.50
1,study,metro,-33.46,-70.61,-33.41,-70.51
2,shopping,walk,-33.47,-70.62,-33.42,-70.52
3,home,bike,-33.48,-70.63,-33.43,-70.53
4,other,car,-33.49,-70.64,-33.44,-70.54


DataFrame con campos requeridos faltantes:


,movement_id,trip_id,movement_seq,purpose,origin_latitude,origin_longitude
0,m0,t0,0,work,-33.45,-70.60
1,m1,t1,0,study,-33.46,-70.61
2,m2,t2,0,shopping,-33.47,-70.62
3,m3,t3,0,home,-33.48,-70.63
4,m4,t4,0,other,-33.49,-70.64


In [171]:
def check_final_required_fields(
    work: pd.DataFrame,
    schema: TripSchema,
    *,
    single_stage: bool,
):
    work_cols = set(work.columns)
    required_fields = set(schema.required)

    missing_required_final = sorted(required_fields - work_cols)

    movement_id_present = "movement_id" in work_cols

    missing_single_stage_fields = []
    if single_stage:
        for f in ["trip_id", "movement_seq"]:
            if f not in work_cols:
                missing_single_stage_fields.append(f)

    should_abort = (
        bool(missing_required_final)
        or not movement_id_present
        or bool(missing_single_stage_fields)
    )

    return {
        "required_fields": sorted(required_fields),
        "work_columns": sorted(work.columns),
        "missing_required_final": missing_required_final,
        "movement_id_present": movement_id_present,
        "missing_single_stage_fields": missing_single_stage_fields,
        "should_abort": should_abort,
    }

In [174]:
res31_complete = check_final_required_fields(
    df31_complete,
    schema31,
    single_stage=True,
)

for key, value in res31_complete.items():
    print(f"{key}: {value}")

required_fields: ['mode', 'movement_id', 'origin_latitude', 'origin_longitude', 'purpose']
work_columns: ['destination_latitude', 'destination_longitude', 'mode', 'movement_id', 'movement_seq', 'origin_latitude', 'origin_longitude', 'purpose', 'trip_id']
missing_required_final: []
movement_id_present: True
missing_single_stage_fields: []
should_abort: False


In [175]:
res31_without_ids = check_final_required_fields(
    df31_without_ids,
    schema31,
    single_stage=True,
)

for key, value in res31_without_ids.items():
    print(f"{key}: {value}")

required_fields: ['mode', 'movement_id', 'origin_latitude', 'origin_longitude', 'purpose']
work_columns: ['destination_latitude', 'destination_longitude', 'mode', 'origin_latitude', 'origin_longitude', 'purpose']
missing_required_final: ['movement_id']
movement_id_present: False
missing_single_stage_fields: ['trip_id', 'movement_seq']
should_abort: True


In [176]:
res31_missing_required = check_final_required_fields(
    df31_missing_required,
    schema31,
    single_stage=True,
)

for key, value in res31_missing_required.items():
    print(f"{key}: {value}")

required_fields: ['mode', 'movement_id', 'origin_latitude', 'origin_longitude', 'purpose']
work_columns: ['movement_id', 'movement_seq', 'origin_latitude', 'origin_longitude', 'purpose', 'trip_id']
missing_required_final: ['mode']
movement_id_present: True
missing_single_stage_fields: []
should_abort: True


In [177]:
res31_without_ids_not_single = check_final_required_fields(
    df31_without_ids,
    schema31,
    single_stage=False,
)

for key, value in res31_without_ids_not_single.items():
    print(f"{key}: {value}")

required_fields: ['mode', 'movement_id', 'origin_latitude', 'origin_longitude', 'purpose']
work_columns: ['destination_latitude', 'destination_longitude', 'mode', 'origin_latitude', 'origin_longitude', 'purpose']
missing_required_final: ['movement_id']
movement_id_present: False
missing_single_stage_fields: []
should_abort: True


### 32. Conversión TripSchema / TripSchemaEffective <-> dict sin perder estructura

#### A) Primero para TripSchema

In [1]:
from pylondrina.schema import TripSchema, FieldSpec, DomainSpec, TripSchemaEffective

schema32_min = TripSchema(
    version="0.1.0",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=False),
    },
    required=["movement_id"],
    semantic_rules=None,
)

schema32_cat = TripSchema(
    version="0.1.1",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "purpose": FieldSpec(
            name="purpose",
            dtype="categorical",
            required=False,
            domain=DomainSpec(
                values=["work", "study", "shopping"],
                extendable=True,
                aliases={"Trabajo": "work", "estudio": "study"},
            ),
        ),
        "mode": FieldSpec(
            name="mode",
            dtype="categorical",
            required=False,
            domain=DomainSpec(
                values=["bus", "metro", "walk"],
                extendable=False,
                aliases={"Bus": "bus", "WALK": "walk"},
            ),
        ),
    },
    required=["movement_id"],
    semantic_rules={"trip_unit": "movement"},
)

schema32_rich = TripSchema(
    version="0.2.0",
    fields={
        "movement_id": FieldSpec(
            name="movement_id",
            dtype="string",
            required=True,
            constraints={"nullable": False, "unique": True},
        ),
        "trip_id": FieldSpec(
            name="trip_id",
            dtype="string",
            required=False,
            constraints={"nullable": False},
        ),
        "movement_seq": FieldSpec(
            name="movement_seq",
            dtype="int",
            required=False,
            constraints={"nullable": False, "range": {"min": 0}},
        ),
        "origin_time_utc": FieldSpec(
            name="origin_time_utc",
            dtype="datetime",
            required=False,
            constraints={"nullable": True, "datetime": {"tz": "UTC"}},
        ),
        "origin_h3_index": FieldSpec(
            name="origin_h3_index",
            dtype="string",
            required=False,
            constraints={"h3": {"resolution": 8}},
        ),
    },
    required=["movement_id"],
    semantic_rules={"single_stage_default": True},
)

schema32_bad_dict_1 = {
    "version": "0.1.0",
    "fields": {
        "movement_id": {
            # falta "name"
            "dtype": "string",
            "required": True,
            "constraints": None,
            "domain": None,
        }
    },
    "required": ["movement_id"],
    "semantic_rules": None,
}

schema32_bad_dict_2 = {
    "version": "0.1.0",
    "fields": {
        "purpose": {
            "name": "purpose",
            "dtype": "categorical",
            "required": False,
            "constraints": None,
            "domain": {
                "values": ["work", "study"],
                "extendable": True,
                "aliases": ["bad", "aliases"],  # debería ser dict o None
            },
        }
    },
    "required": ["movement_id"],  # required no está en fields
    "semantic_rules": None,
}



In [2]:
from dataclasses import asdict

def trip_schema_to_dict(schema: TripSchema) -> dict:
    return asdict(schema)

def trip_schema_from_dict(d: dict) -> TripSchema:
    if not isinstance(d, dict):
        raise TypeError("TripSchema dict debe ser un dict")

    required_top = {"version", "fields", "required", "semantic_rules"}
    missing_top = required_top - set(d.keys())
    if missing_top:
        raise ValueError(f"Faltan llaves de nivel superior en TripSchema: {sorted(missing_top)}")

    fields_raw = d["fields"]
    if not isinstance(fields_raw, dict):
        raise TypeError("TripSchema['fields'] debe ser dict")

    fields = {}

    for field_name, fs_raw in fields_raw.items():
        if not isinstance(fs_raw, dict):
            raise TypeError(f"FieldSpec de {field_name!r} debe ser dict")

        required_fs = {"name", "dtype", "required", "constraints", "domain"}
        missing_fs = required_fs - set(fs_raw.keys())
        if missing_fs:
            raise ValueError(f"Faltan llaves en FieldSpec {field_name!r}: {sorted(missing_fs)}")

        domain_raw = fs_raw["domain"]
        domain_obj = None

        if domain_raw is not None:
            if not isinstance(domain_raw, dict):
                raise TypeError(f"domain de {field_name!r} debe ser dict o None")

            required_domain = {"values", "extendable", "aliases"}
            missing_domain = required_domain - set(domain_raw.keys())
            if missing_domain:
                raise ValueError(f"Faltan llaves en DomainSpec de {field_name!r}: {sorted(missing_domain)}")

            aliases = domain_raw["aliases"]
            if aliases is not None and not isinstance(aliases, dict):
                raise TypeError(f"aliases de {field_name!r} debe ser dict o None")

            domain_obj = DomainSpec(
                values=domain_raw["values"],
                extendable=domain_raw["extendable"],
                aliases=aliases,
            )

        fs_obj = FieldSpec(
            name=fs_raw["name"],
            dtype=fs_raw["dtype"],
            required=fs_raw["required"],
            constraints=fs_raw["constraints"],
            domain=domain_obj,
        )

        fields[field_name] = fs_obj

    required_fields = d["required"]
    if not isinstance(required_fields, list):
        raise TypeError("TripSchema['required'] debe ser list")

    unknown_required = sorted(set(required_fields) - set(fields.keys()))
    if unknown_required:
        raise ValueError(f"required contiene campos no definidos en fields: {unknown_required}")

    return TripSchema(
        version=d["version"],
        fields=fields,
        required=required_fields,
        semantic_rules=d["semantic_rules"],
    )

def print_trip_schema_dict(d: dict):
    for key, value in d.items():
        if key == "fields":
            print(f"{key}:")
            for fkey, fvalue in value.items():
                print(f"  {fkey}:")
                for subkey, subvalue in fvalue.items():
                    if subkey == "domain":
                        print(f"    {subkey}:")
                        if subvalue is not None:
                            for dkey, dvalue in subvalue.items():
                                print(f"      {dkey}: {dvalue}")
                        else:
                            print("      None")
                    else:
                        print(f"    {subkey}: {subvalue}")
        else:
            print(f"{key}: {value}")

#### Probando con Schema mínimo

In [3]:
import pprint
schema32_min_dict = trip_schema_to_dict(schema32_min)
print("schema32_min_dict:")
print_trip_schema_dict(schema32_min_dict)

schema32_min_dict:
version: 0.1.0
fields:
  movement_id:
    name: movement_id
    dtype: string
    required: True
    constraints: None
    domain:
      None
  origin_latitude:
    name: origin_latitude
    dtype: float
    required: False
    constraints: None
    domain:
      None
required: ['movement_id']
semantic_rules: None


In [4]:
print("\nComparación de schema32_min:")
pprint.pprint(schema32_min)


Comparación de schema32_min:
TripSchema(version='0.1.0',
           fields={'movement_id': FieldSpec(name='movement_id',
                                            dtype='string',
                                            required=True,
                                            constraints=None,
                                            domain=None),
                   'origin_latitude': FieldSpec(name='origin_latitude',
                                                dtype='float',
                                                required=False,
                                                constraints=None,
                                                domain=None)},
           required=['movement_id'],
           semantic_rules=None)


In [5]:
schema32_min_rt = trip_schema_from_dict(schema32_min_dict)
print("\nschema32_min_rt:")
pprint.pprint(schema32_min_rt)


schema32_min_rt:
TripSchema(version='0.1.0',
           fields={'movement_id': FieldSpec(name='movement_id',
                                            dtype='string',
                                            required=True,
                                            constraints=None,
                                            domain=None),
                   'origin_latitude': FieldSpec(name='origin_latitude',
                                                dtype='float',
                                                required=False,
                                                constraints=None,
                                                domain=None)},
           required=['movement_id'],
           semantic_rules=None)


In [6]:
schema32_min == schema32_min_rt

True

#### Probando con Schema con categóricos y dominios

In [8]:
schema32_cat_dict = trip_schema_to_dict(schema32_cat)
print("schema32_cat_dict:")
print_trip_schema_dict(schema32_cat_dict)

schema32_cat_dict:
version: 0.1.1
fields:
  movement_id:
    name: movement_id
    dtype: string
    required: True
    constraints: None
    domain:
      None
  purpose:
    name: purpose
    dtype: categorical
    required: False
    constraints: None
    domain:
      values: ['work', 'study', 'shopping']
      extendable: True
      aliases: {'Trabajo': 'work', 'estudio': 'study'}
  mode:
    name: mode
    dtype: categorical
    required: False
    constraints: None
    domain:
      values: ['bus', 'metro', 'walk']
      extendable: False
      aliases: {'Bus': 'bus', 'WALK': 'walk'}
required: ['movement_id']
semantic_rules: {'trip_unit': 'movement'}


In [14]:
print("\nComparación de schema32_cat:")
pprint.pprint(schema32_cat)


Comparación de schema32_cat:
TripSchema(version='0.1.1',
           fields={'mode': FieldSpec(name='mode',
                                     dtype='categorical',
                                     required=False,
                                     constraints=None,
                                     domain=DomainSpec(values=['bus',
                                                               'metro',
                                                               'walk'],
                                                       extendable=False,
                                                       aliases={'Bus': 'bus',
                                                                'WALK': 'walk'})),
                   'movement_id': FieldSpec(name='movement_id',
                                            dtype='string',
                                            required=True,
                                            constraints=None,
                                 

In [11]:
schema32_cat_rt = trip_schema_from_dict(schema32_cat_dict)
print("schema32_cat_rt:")
pprint.pprint(schema32_cat_rt)

schema32_cat_rt:
TripSchema(version='0.1.1',
           fields={'mode': FieldSpec(name='mode',
                                     dtype='categorical',
                                     required=False,
                                     constraints=None,
                                     domain=DomainSpec(values=['bus',
                                                               'metro',
                                                               'walk'],
                                                       extendable=False,
                                                       aliases={'Bus': 'bus',
                                                                'WALK': 'walk'})),
                   'movement_id': FieldSpec(name='movement_id',
                                            dtype='string',
                                            required=True,
                                            constraints=None,
                                            do

In [12]:
schema32_cat_rt == schema32_cat

True

#### Probando con Schema más rico, con constraints

In [13]:
schema32_rich_dict = trip_schema_to_dict(schema32_rich)
print("schema32_rich_dict:")
print_trip_schema_dict(schema32_rich_dict)

schema32_rich_dict:
version: 0.2.0
fields:
  movement_id:
    name: movement_id
    dtype: string
    required: True
    constraints: {'nullable': False, 'unique': True}
    domain:
      None
  trip_id:
    name: trip_id
    dtype: string
    required: False
    constraints: {'nullable': False}
    domain:
      None
  movement_seq:
    name: movement_seq
    dtype: int
    required: False
    constraints: {'nullable': False, 'range': {'min': 0}}
    domain:
      None
  origin_time_utc:
    name: origin_time_utc
    dtype: datetime
    required: False
    constraints: {'nullable': True, 'datetime': {'tz': 'UTC'}}
    domain:
      None
  origin_h3_index:
    name: origin_h3_index
    dtype: string
    required: False
    constraints: {'h3': {'resolution': 8}}
    domain:
      None
required: ['movement_id']
semantic_rules: {'single_stage_default': True}


In [15]:
print("\nComparación de schema32_rich:")
pprint.pprint(schema32_rich)


Comparación de schema32_rich:
TripSchema(version='0.2.0',
           fields={'movement_id': FieldSpec(name='movement_id',
                                            dtype='string',
                                            required=True,
                                            constraints={'nullable': False,
                                                         'unique': True},
                                            domain=None),
                   'movement_seq': FieldSpec(name='movement_seq',
                                             dtype='int',
                                             required=False,
                                             constraints={'nullable': False,
                                                          'range': {'min': 0}},
                                             domain=None),
                   'origin_h3_index': FieldSpec(name='origin_h3_index',
                                                dtype='string',
             

In [16]:
schema32_rich_rt = trip_schema_from_dict(schema32_rich_dict)
print("schema32_rich_rt:")
pprint.pprint(schema32_rich_rt)

schema32_rich_rt:
TripSchema(version='0.2.0',
           fields={'movement_id': FieldSpec(name='movement_id',
                                            dtype='string',
                                            required=True,
                                            constraints={'nullable': False,
                                                         'unique': True},
                                            domain=None),
                   'movement_seq': FieldSpec(name='movement_seq',
                                             dtype='int',
                                             required=False,
                                             constraints={'nullable': False,
                                                          'range': {'min': 0}},
                                             domain=None),
                   'origin_h3_index': FieldSpec(name='origin_h3_index',
                                                dtype='string',
                          

In [17]:
schema32_rich_rt == schema32_rich

True

#### Prueba de dicts fallidos/incorrectos

In [18]:
for bad in [schema32_bad_dict_1, schema32_bad_dict_2]:
    try:
        trip_schema_from_dict(bad)
        print("ERROR: debió fallar y no falló")
    except Exception as e:
        print("OK TripSchema falló:", type(e).__name__, "-", e)

OK TripSchema falló: ValueError - Faltan llaves en FieldSpec 'movement_id': ['name']
OK TripSchema falló: TypeError - aliases de 'purpose' debe ser dict o None


#### B) Ahora para TripSchemaEffective

In [19]:
from pylondrina.schema import TripSchemaEffective

eff32_min = TripSchemaEffective(
    dtype_effective={
        "movement_id": "string",
        "origin_latitude": "float",
    },
    overrides={},
    domains_effective={},
    temporal={},
    fields_effective=[
        "movement_id",
        "origin_latitude",
    ],
)

eff32_rich = TripSchemaEffective(
    dtype_effective={
        "movement_id": "string",
        "purpose": "categorical",
        "mode": "categorical",
        "origin_time_utc": "datetime",
    },
    overrides={
        "purpose": {
            "reasons": ["domain_extended"],
            "added_values": ["health", "leisure"],
        },
        "origin_time_utc": {
            "reasons": ["datetime_normalized"],
            "status": "tzaware_to_utc",
        },
    },
    domains_effective={
        "purpose": {
            "base_values": ["work", "study", "shopping"],
            "observed_values": ["work", "study", "shopping", "health", "leisure"],
            "extended_values": ["health", "leisure"],
            "unknown_values": [],
            "extendable": True,
            "unknown_value": "unknown",
            "n_unique_observed": 5,
            "n_added": 2,
            "value_correspondence_applied": {"Trabajo": "work"},
        },
        "mode": {
            "base_values": ["bus", "metro", "walk"],
            "observed_values": ["bus", "metro", "walk"],
            "extended_values": [],
            "unknown_values": [],
            "extendable": False,
            "unknown_value": "unknown",
            "n_unique_observed": 3,
            "n_added": 0,
            "value_correspondence_applied": {"Bus": "bus"},
        },
    },
    temporal={
        "tier": "tier_1",
        "fields_present": ["origin_time_utc", "destination_time_utc"],
        "normalization": {"origin_time_utc": "tzaware_to_utc"},
    },
    fields_effective=[
        "movement_id",
        "purpose",
        "mode",
        "origin_time_utc",
    ],
)

eff32_bad_dict_1 = {
    "dtype_effective": ["movement_id", "string"],  # debería ser dict
    "overrides": {},
    "domains_effective": {},
    "temporal": {},
    "fields_effective": ["movement_id"],
}

eff32_bad_dict_2 = {
    "dtype_effective": {"movement_id": "string"},
    "overrides": {},
    "domains_effective": {},
    "temporal": {},
    # falta "fields_effective"
}

eff32_bad_dict_3 = {
    "dtype_effective": {"movement_id": "string"},
    "overrides": {},
    "domains_effective": {},
    "temporal": {},
    "fields_effective": {"movement_id": True},  # debería ser list
}

eff32_bad_dict_4 = {
    "dtype_effective": {"movement_id": "string"},
    "overrides": {},
    "domains_effective": {},
    "temporal": {},
    "fields_effective": ["movement_id", 123],  # 123 no debería aceptarse
}

In [20]:
def trip_schema_effective_to_dict(eff: TripSchemaEffective) -> dict:
    return eff.to_dict()

def trip_schema_effective_from_dict(d: dict) -> TripSchemaEffective:
    if not isinstance(d, dict):
        raise TypeError("TripSchemaEffective dict debe ser un dict")

    required_top = {
        "dtype_effective",
        "overrides",
        "domains_effective",
        "temporal",
        "fields_effective",
    }
    missing_top = required_top - set(d.keys())
    if missing_top:
        raise ValueError(
            f"Faltan llaves de nivel superior en TripSchemaEffective: {sorted(missing_top)}"
        )

    for key in ["dtype_effective", "overrides", "domains_effective", "temporal"]:
        if not isinstance(d[key], dict):
            raise TypeError(f"TripSchemaEffective[{key!r}] debe ser dict")

    if not isinstance(d["fields_effective"], list):
        raise TypeError("TripSchemaEffective['fields_effective'] debe ser list")

    non_string_fields = [x for x in d["fields_effective"] if not isinstance(x, str)]
    if non_string_fields:
        raise TypeError(
            f"TripSchemaEffective['fields_effective'] debe contener solo strings. "
            f"Inválidos: {non_string_fields}"
        )

    return TripSchemaEffective(
        dtype_effective=d["dtype_effective"],
        overrides=d["overrides"],
        domains_effective=d["domains_effective"],
        temporal=d["temporal"],
        fields_effective=d["fields_effective"],
    )


#### Prueba de Effective minimo

In [23]:
eff32_min_dict = trip_schema_effective_to_dict(eff32_min)
print("eff32_min_dict:")
display(eff32_min_dict)

eff32_min_dict:


{'dtype_effective': {'movement_id': 'string', 'origin_latitude': 'float'},
 'overrides': {},
 'domains_effective': {},
 'temporal': {},
 'fields_effective': ['movement_id', 'origin_latitude']}

In [24]:
print("\nComparación de eff32_min:")
pprint.pprint(eff32_min)


Comparación de eff32_min:
TripSchemaEffective(dtype_effective={'movement_id': 'string',
                                     'origin_latitude': 'float'},
                    overrides={},
                    domains_effective={},
                    temporal={},
                    fields_effective=['movement_id', 'origin_latitude'])


In [25]:
eff32_min_rt = trip_schema_effective_from_dict(eff32_min_dict)
print("eff32_min_rt:")
pprint.pprint(eff32_min_rt)

eff32_min_rt:
TripSchemaEffective(dtype_effective={'movement_id': 'string',
                                     'origin_latitude': 'float'},
                    overrides={},
                    domains_effective={},
                    temporal={},
                    fields_effective=['movement_id', 'origin_latitude'])


In [26]:
eff32_min == eff32_min_rt

True

#### Prueba de Effective rico

In [27]:
eff32_rich_dict = trip_schema_effective_to_dict(eff32_rich)
print("eff32_rich_dict:")
display(eff32_rich_dict)

eff32_rich_dict:


{'dtype_effective': {'movement_id': 'string',
  'purpose': 'categorical',
  'mode': 'categorical',
  'origin_time_utc': 'datetime'},
 'overrides': {'purpose': {'reasons': ['domain_extended'],
   'added_values': ['health', 'leisure']},
  'origin_time_utc': {'reasons': ['datetime_normalized'],
   'status': 'tzaware_to_utc'}},
 'domains_effective': {'purpose': {'base_values': ['work',
    'study',
    'shopping'],
   'observed_values': ['work', 'study', 'shopping', 'health', 'leisure'],
   'extended_values': ['health', 'leisure'],
   'unknown_values': [],
   'extendable': True,
   'unknown_value': 'unknown',
   'n_unique_observed': 5,
   'n_added': 2,
   'value_correspondence_applied': {'Trabajo': 'work'}},
  'mode': {'base_values': ['bus', 'metro', 'walk'],
   'observed_values': ['bus', 'metro', 'walk'],
   'extended_values': [],
   'unknown_values': [],
   'extendable': False,
   'unknown_value': 'unknown',
   'n_unique_observed': 3,
   'n_added': 0,
   'value_correspondence_applied': {

In [28]:
print("\nComparación de eff32_rich:")
pprint.pprint(eff32_rich)


Comparación de eff32_rich:
TripSchemaEffective(dtype_effective={'mode': 'categorical',
                                     'movement_id': 'string',
                                     'origin_time_utc': 'datetime',
                                     'purpose': 'categorical'},
                    overrides={'origin_time_utc': {'reasons': ['datetime_normalized'],
                                                   'status': 'tzaware_to_utc'},
                               'purpose': {'added_values': ['health',
                                                            'leisure'],
                                           'reasons': ['domain_extended']}},
                    domains_effective={'mode': {'base_values': ['bus',
                                                                'metro',
                                                                'walk'],
                                                'extendable': False,
                                              

In [29]:
eff32_rich_rt = trip_schema_effective_from_dict(eff32_rich_dict)
print("eff32_rich_rt:")
pprint.pprint(eff32_rich_rt)

eff32_rich_rt:
TripSchemaEffective(dtype_effective={'mode': 'categorical',
                                     'movement_id': 'string',
                                     'origin_time_utc': 'datetime',
                                     'purpose': 'categorical'},
                    overrides={'origin_time_utc': {'reasons': ['datetime_normalized'],
                                                   'status': 'tzaware_to_utc'},
                               'purpose': {'added_values': ['health',
                                                            'leisure'],
                                           'reasons': ['domain_extended']}},
                    domains_effective={'mode': {'base_values': ['bus',
                                                                'metro',
                                                                'walk'],
                                                'extendable': False,
                                                'extended_v

In [30]:
eff32_rich == eff32_rich_rt

True

#### Fallos esperados

In [31]:
for bad in [eff32_bad_dict_1, eff32_bad_dict_2, eff32_bad_dict_3, eff32_bad_dict_4]:
    try:
        trip_schema_effective_from_dict(bad)
        print("ERROR: debió fallar y no falló")
    except Exception as e:
        print("OK TripSchemaEffective falló:", type(e).__name__, "-", e)

OK TripSchemaEffective falló: TypeError - TripSchemaEffective['dtype_effective'] debe ser dict
OK TripSchemaEffective falló: ValueError - Faltan llaves de nivel superior en TripSchemaEffective: ['fields_effective']
OK TripSchemaEffective falló: TypeError - TripSchemaEffective['fields_effective'] debe ser list
OK TripSchemaEffective falló: TypeError - TripSchemaEffective['fields_effective'] debe contener solo strings. Inválidos: [123]


### 33. Construccion de provenance, el “event” que va en el import, y metadata

In [32]:
import pandas as pd
import numpy as np
from pylondrina.schema import TripSchemaEffective

work33 = pd.DataFrame({
    "movement_id": ["m0", "m1", "m2", "m3"],
    "trip_id": ["t0", "t1", "t2", "t3"],
    "movement_seq": pd.Series([0, 0, 0, 0], dtype="Int64"),
    "purpose": ["work", "study", "shopping", "home"],
    "mode": ["bus", "metro", "walk", "bike"],
    "origin_latitude": [-33.45, -33.46, -33.47, -33.48],
    "origin_longitude": [-70.60, -70.61, -70.62, -70.63],
    "origin_h3_index": ["88e35a99a1fffff", "88e35a99a1fffff", pd.NA, "88e35a99a9fffff"],
})

rows_in = 6
rows_out = len(work33)

schema_version = "0.1.0"
source_name = "eod_santiago_2012"

field_correspondence_applied = {
    "purpose": "motivo",
    "mode": "modo",
    "origin_latitude": "o_lat",
    "origin_longitude": "o_lon",
}

value_correspondence_applied = {
    "purpose": {"Trabajo": "work", "Estudio": "study"},
    "mode": {"Bus": "bus", "WALK": "walk"},
}

domains_effective33 = {
    "purpose": {
        "base_values": ["work", "study", "shopping"],
        "observed_values": ["work", "study", "shopping", "home"],
        "extended_values": ["home"],
        "unknown_values": [],
        "extendable": True,
        "unknown_value": "unknown",
        "n_unique_observed": 4,
        "n_added": 1,
        "value_correspondence_applied": {"Trabajo": "work", "Estudio": "study"},
    },
    "mode": {
        "base_values": ["bus", "metro", "walk"],
        "observed_values": ["bus", "metro", "walk", "bike"],
        "extended_values": ["bike"],
        "unknown_values": [],
        "extendable": True,
        "unknown_value": "unknown",
        "n_unique_observed": 4,
        "n_added": 1,
        "value_correspondence_applied": {"Bus": "bus", "WALK": "walk"},
    },
}

domains_extended33 = ["purpose", "mode"]

extra_fields_kept33 = ["raw_source_col", "debug_flag"]

h3_meta33 = {
    "resolution": 8,
    "source_fields": [["origin_latitude", "origin_longitude"]],
    "derived_fields": ["origin_h3_index"],
}

parameters_effective33 = {
    "keep_extra_fields": True,
    "selected_fields": ["purpose", "mode", "origin_latitude", "origin_longitude"],
    "strict": False,
    "strict_domains": False,
    "single_stage": True,
    "h3_resolution": 8,
    "source_name": source_name,
}

columns_added33 = ["movement_id", "trip_id", "movement_seq", "origin_h3_index"]
columns_deleted33 = ["destination_latitude", "destination_longitude"]

n_fields_mapped33 = len(field_correspondence_applied)
n_domain_mappings_applied33 = sum(len(v) for v in value_correspondence_applied.values())

temporal_tier_detected33 = "tier_3"
temporal_fields_present33 = []
datetime_normalization_status_by_field33 = {}
source_timezone_used33 = None
temporal_notes33 = "Dataset sin campos temporales OD explícitos"

issues33 = [
    {"level": "warning", "code": "TEMPORAL_TIER_3", "message": "Dataset sin tiempo OD explícito"},
    {"level": "warning", "code": "DOMAIN_EXTENDED", "message": "purpose extendido"},
    {"level": "warning", "code": "DOMAIN_EXTENDED", "message": "mode extendido"},
]

schema_effective33 = TripSchemaEffective(
    dtype_effective={
        "movement_id": "string",
        "trip_id": "string",
        "movement_seq": "int",
        "purpose": "categorical",
        "mode": "categorical",
        "origin_latitude": "float",
        "origin_longitude": "float",
        "origin_h3_index": "string",
    },
    overrides={
        "purpose": {"reasons": ["domain_extended"], "added_values": ["home"]},
        "mode": {"reasons": ["domain_extended"], "added_values": ["bike"]},
    },
    domains_effective=domains_effective33,
    temporal={
        "tier": temporal_tier_detected33,
        "fields_present": temporal_fields_present33,
    },
    fields_effective=[
        "movement_id",
        "trip_id",
        "movement_seq",
        "purpose",
        "mode",
        "origin_latitude",
        "origin_longitude",
        "origin_h3_index",
    ],
)

#### Construcción de Provenance

In [33]:
from datetime import datetime, timezone

def build_min_provenance(source_name: str, provided_provenance=None):
    if provided_provenance is not None:
        return provided_provenance

    return {
        "source_name": source_name,
        "created_by_op": "import_trips",
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }

provenance33 = build_min_provenance(source_name)
provenance33

{'source_name': 'eod_santiago_2012',
 'created_by_op': 'import_trips',
 'created_at_utc': '2026-03-17T23:20:05.964010+00:00'}

#### Construccion del event import

In [36]:
from collections import Counter

def build_issues_summary(issues):
    level_counts = Counter()
    code_counts = Counter()

    for issue in issues:
        level = issue.get("level", "unknown")
        code = issue.get("code", "unknown")
        level_counts[level] += 1
        code_counts[code] += 1

    return {
        "counts": dict(level_counts),
        "by_code": dict(code_counts),
    }

def build_import_event(
    *,
    parameters_effective,
    rows_in,
    rows_out,
    n_fields_mapped,
    n_domain_mappings_applied,
    columns_added,
    columns_deleted,
    domains_extended,
    issues,
    temporal_tier,
    temporal_notes=None,
):
    issues_summary = build_issues_summary(issues)

    return {
        "op": "import_trips",
        "ts_utc": datetime.now(timezone.utc).isoformat(),
        "parameters": parameters_effective,
        "summary": {
            "input_rows": rows_in,
            "output_rows": rows_out,
            "rows_dropped": rows_in - rows_out,
            "n_fields_mapped": n_fields_mapped,
            "n_domain_mappings_applied": n_domain_mappings_applied,
            "columns_added": columns_added,
            "columns_deleted": columns_deleted,
            "domains_extended_count": len(domains_extended),
            "temporal_tier": temporal_tier,
            "temporal_notes": temporal_notes,
        },
        "issues_summary": issues_summary,
    }

event_import33 = build_import_event(
    parameters_effective=parameters_effective33,
    rows_in=rows_in,
    rows_out=rows_out,
    n_fields_mapped=n_fields_mapped33,
    n_domain_mappings_applied=n_domain_mappings_applied33,
    columns_added=columns_added33,
    columns_deleted=columns_deleted33,
    domains_extended=domains_extended33,
    issues=issues33,
    temporal_tier=temporal_tier_detected33,
    temporal_notes=temporal_notes33,
)

display(event_import33)

{'op': 'import_trips',
 'ts_utc': '2026-03-18T03:45:31.514924+00:00',
 'parameters': {'keep_extra_fields': True,
  'selected_fields': ['purpose',
   'mode',
   'origin_latitude',
   'origin_longitude'],
  'strict': False,
  'strict_domains': False,
  'single_stage': True,
  'h3_resolution': 8,
  'source_name': 'eod_santiago_2012'},
 'summary': {'input_rows': 6,
  'output_rows': 4,
  'rows_dropped': 2,
  'n_fields_mapped': 4,
  'n_domain_mappings_applied': 4,
  'columns_added': ['movement_id',
   'trip_id',
   'movement_seq',
   'origin_h3_index'],
  'columns_deleted': ['destination_latitude', 'destination_longitude'],
  'domains_extended_count': 2,
  'temporal_tier': 'tier_3',
  'temporal_notes': 'Dataset sin campos temporales OD explícitos'},
 'issues_summary': {'counts': {'warning': 3},
  'by_code': {'TEMPORAL_TIER_3': 1, 'DOMAIN_EXTENDED': 2}}}

#### Construccion de metadata

In [37]:
def build_import_metadata(
    *,
    dataset_id,
    schema_dict,
    schema_effective,
    field_correspondence_applied,
    value_correspondence_applied,
    domains_effective,
    domains_extended,
    extra_fields_kept,
    h3_meta,
    temporal_tier_detected,
    temporal_fields_present,
    datetime_normalization_status_by_field,
    source_timezone_used,
    event_import,
):
    metadata = {
        "dataset_id": dataset_id,
        "is_validated": False,
        "schema": schema_dict,
        "schema_effective": schema_effective.to_dict(),
        "mappings": {
            "field_correspondence": field_correspondence_applied,
            "value_correspondence": value_correspondence_applied,
        },
        "domains_effective": domains_effective,
        "domains_extended": domains_extended,
        "extra_fields_kept": extra_fields_kept,
        "events": [event_import],
        "temporal": {
            "tier": temporal_tier_detected,
            "fields_present": temporal_fields_present,
        },
    }

    if h3_meta is not None:
        metadata["h3"] = h3_meta

    if temporal_tier_detected == "tier_1":
        metadata["temporal"]["normalization"] = datetime_normalization_status_by_field

    if source_timezone_used is not None:
        metadata["temporal"]["source_timezone_used"] = source_timezone_used

    return metadata

dataset_id33 = "tripds_eod_santiago_2012_001"
schema_dict33 = {
    "version": "0.1.0",
    "fields": {
        "movement_id": {"name": "movement_id", "dtype": "string", "required": True, "constraints": None, "domain": None},
        "trip_id": {"name": "trip_id", "dtype": "string", "required": False, "constraints": None, "domain": None},
        "movement_seq": {"name": "movement_seq", "dtype": "int", "required": False, "constraints": None, "domain": None},
        "purpose": {"name": "purpose", "dtype": "categorical", "required": False, "constraints": None, "domain": None},
        "mode": {"name": "mode", "dtype": "categorical", "required": False, "constraints": None, "domain": None},
        "origin_latitude": {"name": "origin_latitude", "dtype": "float", "required": False, "constraints": None, "domain": None},
        "origin_longitude": {"name": "origin_longitude", "dtype": "float", "required": False, "constraints": None, "domain": None},
        "origin_h3_index": {"name": "origin_h3_index", "dtype": "string", "required": False, "constraints": None, "domain": None},
    },
    "required": ["movement_id"],
    "semantic_rules": None,
}
metadata33 = build_import_metadata(
    dataset_id=dataset_id33,
    schema_dict=schema_dict33,
    schema_effective=schema_effective33,
    field_correspondence_applied=field_correspondence_applied,
    value_correspondence_applied=value_correspondence_applied,
    domains_effective=domains_effective33,
    domains_extended=domains_extended33,
    extra_fields_kept=extra_fields_kept33,
    h3_meta=h3_meta33,
    temporal_tier_detected=temporal_tier_detected33,
    temporal_fields_present=temporal_fields_present33,
    datetime_normalization_status_by_field=datetime_normalization_status_by_field33,
    source_timezone_used=source_timezone_used33,
    event_import=event_import33,
)

metadata33

{'dataset_id': 'tripds_eod_santiago_2012_001',
 'is_validated': False,
 'schema': {'version': '0.1.0',
  'fields': {'movement_id': {'name': 'movement_id',
    'dtype': 'string',
    'required': True,
    'constraints': None,
    'domain': None},
   'trip_id': {'name': 'trip_id',
    'dtype': 'string',
    'required': False,
    'constraints': None,
    'domain': None},
   'movement_seq': {'name': 'movement_seq',
    'dtype': 'int',
    'required': False,
    'constraints': None,
    'domain': None},
   'purpose': {'name': 'purpose',
    'dtype': 'categorical',
    'required': False,
    'constraints': None,
    'domain': None},
   'mode': {'name': 'mode',
    'dtype': 'categorical',
    'required': False,
    'constraints': None,
    'domain': None},
   'origin_latitude': {'name': 'origin_latitude',
    'dtype': 'float',
    'required': False,
    'constraints': None,
    'domain': None},
   'origin_longitude': {'name': 'origin_longitude',
    'dtype': 'float',
    'required': False,
 

### 34. Que probar: la creacion de un tripDataset

In [41]:
import pandas as pd
import numpy as np
from pylondrina.schema import TripSchema, FieldSpec, TripSchemaEffective

work34 = pd.DataFrame({
    "movement_id": ["m0", "m1", "m2", "m3"],
    "trip_id": ["t0", "t1", "t2", "t3"],
    "movement_seq": pd.Series([0, 0, 0, 0], dtype="Int64"),
    "purpose": ["work", "study", "shopping", "home"],
    "mode": ["bus", "metro", "walk", "bike"],
    "origin_latitude": [-33.45, -33.46, -33.47, -33.48],
    "origin_longitude": [-70.60, -70.61, -70.62, -70.63],
    "origin_h3_index": ["88e35a99a1fffff", "88e35a99a1fffff", pd.NA, "88e35a99a9fffff"],
})

schema34 = TripSchema(
    version="0.1.0",
    fields={
        "movement_id": FieldSpec(name="movement_id", dtype="string", required=True),
        "trip_id": FieldSpec(name="trip_id", dtype="string", required=False),
        "movement_seq": FieldSpec(name="movement_seq", dtype="int", required=False),
        "purpose": FieldSpec(name="purpose", dtype="categorical", required=False),
        "mode": FieldSpec(name="mode", dtype="categorical", required=False),
        "origin_latitude": FieldSpec(name="origin_latitude", dtype="float", required=False),
        "origin_longitude": FieldSpec(name="origin_longitude", dtype="float", required=False),
        "origin_h3_index": FieldSpec(name="origin_h3_index", dtype="string", required=False),
    },
    required=["movement_id"],
    semantic_rules=None,
)

schema_version34 = schema34.version

from datetime import datetime, timezone

provenance34 = {
    "source_name": "eod_santiago_2012",
    "created_by_op": "import_trips",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}

field_correspondence34 = {
    "purpose": "motivo",
    "mode": "modo",
    "origin_latitude": "o_lat",
    "origin_longitude": "o_lon",
}

value_correspondence34 = {
    "purpose": {"Trabajo": "work", "Estudio": "study"},
    "mode": {"Bus": "bus", "WALK": "walk"},
}

schema_effective34 = TripSchemaEffective(
    dtype_effective={
        "movement_id": "string",
        "trip_id": "string",
        "movement_seq": "int",
        "purpose": "categorical",
        "mode": "categorical",
        "origin_latitude": "float",
        "origin_longitude": "float",
        "origin_h3_index": "string",
    },
    overrides={
        "purpose": {"reasons": ["domain_extended"], "added_values": ["home"]},
        "mode": {"reasons": ["domain_extended"], "added_values": ["bike"]},
    },
    domains_effective={
        "purpose": {
            "base_values": ["work", "study", "shopping"],
            "observed_values": ["work", "study", "shopping", "home"],
            "extended_values": ["home"],
            "unknown_values": [],
            "extendable": True,
            "unknown_value": "unknown",
            "n_unique_observed": 4,
            "n_added": 1,
            "value_correspondence_applied": {"Trabajo": "work", "Estudio": "study"},
        },
        "mode": {
            "base_values": ["bus", "metro", "walk"],
            "observed_values": ["bus", "metro", "walk", "bike"],
            "extended_values": ["bike"],
            "unknown_values": [],
            "extendable": True,
            "unknown_value": "unknown",
            "n_unique_observed": 4,
            "n_added": 1,
            "value_correspondence_applied": {"Bus": "bus", "WALK": "walk"},
        },
    },
    temporal={
        "tier": "tier_3",
        "fields_present": [],
    },
    fields_effective=[
        "movement_id",
        "trip_id",
        "movement_seq",
        "purpose",
        "mode",
        "origin_latitude",
        "origin_longitude",
        "origin_h3_index",
    ],
)

metadata34 = {
    "dataset_id": "tripds_eod_santiago_2012_001",
    "is_validated": False,
    "schema": {
        "version": "0.1.0",
        "fields": {
            "movement_id": {"name": "movement_id", "dtype": "string", "required": True, "constraints": None, "domain": None},
            "trip_id": {"name": "trip_id", "dtype": "string", "required": False, "constraints": None, "domain": None},
            "movement_seq": {"name": "movement_seq", "dtype": "int", "required": False, "constraints": None, "domain": None},
            "purpose": {"name": "purpose", "dtype": "categorical", "required": False, "constraints": None, "domain": None},
            "mode": {"name": "mode", "dtype": "categorical", "required": False, "constraints": None, "domain": None},
            "origin_latitude": {"name": "origin_latitude", "dtype": "float", "required": False, "constraints": None, "domain": None},
            "origin_longitude": {"name": "origin_longitude", "dtype": "float", "required": False, "constraints": None, "domain": None},
            "origin_h3_index": {"name": "origin_h3_index", "dtype": "string", "required": False, "constraints": None, "domain": None},
        },
        "required": ["movement_id"],
        "semantic_rules": None,
    },
    "schema_effective": schema_effective34.to_dict(),
    "mappings": {
        "field_correspondence": field_correspondence34,
        "value_correspondence": value_correspondence34,
    },
    "domains_effective": schema_effective34.domains_effective,
    "domains_extended": ["purpose", "mode"],
    "extra_fields_kept": [],
    "h3": {
        "resolution": 8,
        "source_fields": [["origin_latitude", "origin_longitude"]],
        "derived_fields": ["origin_h3_index"],
    },
    "events": [
        {
            "op": "import_trips",
            "ts_utc": datetime.now(timezone.utc).isoformat(),
            "parameters": {
                "keep_extra_fields": False,
                "selected_fields": None,
                "strict": False,
                "strict_domains": False,
                "single_stage": True,
                "h3_resolution": 8,
                "source_name": "eod_santiago_2012",
            },
            "summary": {
                "input_rows": 4,
                "output_rows": 4,
                "rows_dropped": 0,
                "n_fields_mapped": 4,
                "n_domain_mappings_applied": 4,
                "columns_added": ["movement_id", "trip_id", "movement_seq", "origin_h3_index"],
                "columns_deleted": [],
                "domains_extended_count": 2,
                "temporal_tier": "tier_3",
                "temporal_notes": "Dataset sin tiempo OD explícito",
            },
            "issues_summary": {
                "counts": {"warning": 2},
                "by_code": {"DOMAIN_EXTENDED": 2},
            },
        }
    ],
    "temporal": {
        "tier": "tier_3",
        "fields_present": [],
    },
}

from pylondrina.datasets import TripDataset
trip_dataset34 = TripDataset(
    data=work34,
    schema=schema34,
    schema_version=schema_version34,
    provenance=provenance34,
    field_correspondence=field_correspondence34,
    value_correspondence=value_correspondence34,
    metadata=metadata34,
    schema_effective=schema_effective34,
)

pprint.pprint(trip_dataset34)

TripDataset(data=  movement_id trip_id  movement_seq   purpose   mode  origin_latitude  \
0          m0      t0             0      work    bus           -33.45   
1          m1      t1             0     study  metro           -33.46   
2          m2      t2             0  shopping   walk           -33.47   
3          m3      t3             0      home   bike           -33.48   

   origin_longitude  origin_h3_index  
0            -70.60  88e35a99a1fffff  
1            -70.61  88e35a99a1fffff  
2            -70.62             <NA>  
3            -70.63  88e35a99a9fffff  ,
            schema=TripSchema(version='0.1.0',
                              fields={'mode': FieldSpec(name='mode',
                                                        dtype='categorical',
                                                        required=False,
                                                        constraints=None,
                                                        domain=None),
            

In [42]:
print(type(trip_dataset34.data))
print(type(trip_dataset34.schema))
print(type(trip_dataset34.schema_effective))
print(trip_dataset34.schema_version)
print(trip_dataset34.provenance)
print(trip_dataset34.field_correspondence)
print(trip_dataset34.value_correspondence)
print(trip_dataset34.metadata["dataset_id"])

<class 'pandas.core.frame.DataFrame'>
<class 'pylondrina.schema.TripSchema'>
<class 'pylondrina.schema.TripSchemaEffective'>
0.1.0
{'source_name': 'eod_santiago_2012', 'created_by_op': 'import_trips', 'created_at_utc': '2026-03-18T04:12:46.756211+00:00'}
{'purpose': 'motivo', 'mode': 'modo', 'origin_latitude': 'o_lat', 'origin_longitude': 'o_lon'}
{'purpose': {'Trabajo': 'work', 'Estudio': 'study'}, 'mode': {'Bus': 'bus', 'WALK': 'walk'}}
tripds_eod_santiago_2012_001


### 35. Probar la creacion de un ImportReport

In [44]:
from pylondrina.reports import ImportReport, Issue

issues35 = [
    Issue(
        level="warning",
        code="DOMAIN_EXTENDED",
        message="El dominio de purpose fue extendido con valores observados.",
        field="purpose",
        row_count=2,
        details={"added_values": ["home"]},
    ),
    Issue(
        level="warning",
        code="TEMPORAL_TIER_3",
        message="Dataset sin tiempo OD explícito.",
        details={"tier": "tier_3"},
    ),
]

field_correspondence35 = {
    "purpose": "motivo",
    "mode": "modo",
    "origin_latitude": "o_lat",
    "origin_longitude": "o_lon",
}

value_correspondence35 = {
    "purpose": {"Trabajo": "work", "Estudio": "study"},
    "mode": {"Bus": "bus", "WALK": "walk"},
}

schema_version35 = "0.1.0"

rows_in35 = 6
rows_out35 = 4
n_fields_mapped35 = len(field_correspondence35)
n_domain_mappings_applied35 = sum(len(v) for v in value_correspondence35.values())

parameters_effective35 = {
    "keep_extra_fields": False,
    "selected_fields": ["purpose", "mode", "origin_latitude", "origin_longitude"],
    "strict": False,
    "strict_domains": False,
    "single_stage": True,
    "h3_resolution": 8,
    "source_name": "eod_santiago_2012",
}

metadata35 = {
    "schema_version": schema_version35,
    "dataset_id": "tripds_eod_santiago_2012_001",
    "source_name": "eod_santiago_2012",
    "parameters_effective": parameters_effective35,
    "summary": {
        "rows_in": rows_in35,
        "rows_out": rows_out35,
        "n_fields_mapped": n_fields_mapped35,
        "n_domain_mappings_applied": n_domain_mappings_applied35,
    },
}

In [46]:
report35 = ImportReport(
    ok=True,
    issues=issues35,
    summary={
        "rows_in": rows_in35,
        "rows_out": rows_out35,
        "n_fields_mapped": n_fields_mapped35,
        "n_domain_mappings_applied": n_domain_mappings_applied35,
    },
    parameters=parameters_effective35,
    field_correspondence=field_correspondence35,
    value_correspondence=value_correspondence35,
    schema_version=schema_version35,
    metadata=metadata35,
)

pprint.pprint(report35)

ImportReport(ok=True,
             issues=[Issue(level='warning',
                           code='DOMAIN_EXTENDED',
                           message='El dominio de purpose fue extendido con '
                                   'valores observados.',
                           field='purpose',
                           source_field=None,
                           row_count=2,
                           details={'added_values': ['home']}),
                     Issue(level='warning',
                           code='TEMPORAL_TIER_3',
                           message='Dataset sin tiempo OD explícito.',
                           field=None,
                           source_field=None,
                           row_count=None,
                           details={'tier': 'tier_3'})],
             summary={'n_domain_mappings_applied': 4,
                      'n_fields_mapped': 4,
                      'rows_in': 6,
                      'rows_out': 4},
             parameters={'h3_r